# Information Transmission in ESG Ratings — Full Replication Pipeline (Google Colab)

**Paper:** *Information Transmission in ESG Ratings: Evidence from MSCI Controversies* (Zuhri & Michelon).

This notebook reconstructs the **entire** pipeline end to end, in the order the data actually flows:

1. **Section 0 — Setup:** install packages, mount Google Drive, set paths and credentials.
2. **Section 1 — Data acquisition:** pull the raw data from its four licensed sources — **MSCI ESG**, **GDELT 2.0**, **WRDS** (CRSP, Compustat, CRSP Mutual-Fund Holdings), and **Refinitiv/LSEG** monthly total returns.
3. **Section 2 — Cleaning & matching:** the point-in-time (valid-time) rule, the ISIN↔ISSUERID and ultimate-parent keys, winsorisation.
4. **Section 3 — Variable construction:** builds the firm-year, monthly, issue-level, pricing and event-study panels and every constructed variable.
5. **Section 4 — Tests, tables & figures:** every manuscript exhibit, each cell mapped to its table/figure.

**How to read it.** Every code cell opens with a comment header saying what it does, what it reads, and what it writes. Markdown cells above each block explain the economics/mechanics in plain language.

> **Important:** the four raw sources are *licensed*. The acquisition cells in Section 1 are the faithful, runnable code for each pull, but they require **your own credentials** (WRDS login, Refinitiv app key) and the **MSCI/GDELT files in your Drive**. They cannot run without those. Everything downstream (Sections 2–4) runs purely on the files these cells produce.


## Section 0 — Colab setup

Installs the libraries, mounts Drive, and defines one `PATHS` dictionary that every later cell reads. Point `BASE` at the folder in your Drive that holds the raw files.

In [ ]:
# --- 0.1 Install the libraries the pipeline uses -----------------------------------
# duckdb      : fast out-of-core SQL over the large MSCI parquet files (ASOF joins)
# statsmodels : clustered-SE regressions for the tables
# wrds        : official WRDS Python client (CRSP / Compustat / MF-holdings pulls)
# refinitiv-data: LSEG/Refinitiv Data Library (monthly total returns)
!pip -q install duckdb==1.1.3 pandas numpy statsmodels pyarrow wrds refinitiv-data gdelt 2>/dev/null
import pandas as pd, numpy as np, duckdb, glob, os, json
print('libraries ready')


In [ ]:
# --- 0.2 Mount Google Drive -------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# --- 0.3 One place for every path -------------------------------------------------
# Put the licensed raw files in BASE (see Section 1 for the exact filenames expected).
BASE = '/content/drive/MyDrive/ESG_rating_transmission'      # <-- EDIT to your folder
PATHS = {
    # raw MSCI (you place these; Section 1A)
    'msci_scores' : f'{BASE}/raw/HIST_SUMMARY_SCORES_20260731.parquet',
    'msci_contr'  : f'{BASE}/raw/HIST_CONTROVERSIES_20260731.parquet',
    'msci_instr'  : f'{BASE}/raw/HIST_MASTER_INSTRUMENT_REFERENCE_none_20260831.parquet',
    'msci_issuer' : f'{BASE}/raw/HIST_MASTER_ISSUER_REFERENCE_20260831.parquet',
    # built by Section 1 acquisition cells
    'gdelt_clock' : f'{BASE}/built/keyword_dated_clock.csv',
    'lseg_returns': f'{BASE}/built/monthly_total_returns.csv',
    'crsp_daily'  : f'{BASE}/built/crsp_daily.parquet',
    'compustat'   : f'{BASE}/built/compustat_annual.parquet',
    'mf_holdings' : f'{BASE}/built/mf_holdings.parquet',
    'ff_factors'  : f'{BASE}/built/ff_monthly.parquet',
    # constructed panels (written by Section 3)
    'panel_fy'    : f'{BASE}/built/panel_fy_validtime.parquet',
    'keyword_dict': f'{BASE}/raw/K0_keyword_dictionary.csv',
}
os.makedirs(f'{BASE}/built', exist_ok=True)
print('BASE =', BASE)


## Section 1 — Data acquisition

Four licensed sources. Each sub-section pulls one source into a clean file under `BASE/built/` (or reads the MSCI files you place in `BASE/raw/`). Run each once; later runs just read the cached files.

### 1A — Raw MSCI ESG (July-2026 vintage)

MSCI is delivered as four parquet files. They are **long** (one row per issuer × field × effective date), which is why we use duckdb rather than loading them whole. This cell only *verifies* they are present and prints their schemas — the actual as-of extraction happens in Section 2.

| File | Key columns | Used for |
|---|---|---|
| `HIST_SUMMARY_SCORES` | `ISSUERID, FIELD_ID, VALUE, START_DATE, VERSION_START_DATE` | IAS (`INDUSTRY_ADJUSTED_SCORE`), WAKIS (`WEIGHTED_AVERAGE_SCORE`), letter rating (`IVA_COMPANY_RATING`) |
| `HIST_CONTROVERSIES` | per-issue `*_MODERATE/SEVERE/VERY_SEVERE_CONTROVERSIES`, `TOTAL_NUMBER_CONTROVERSIES` | severity counts → onsets |
| `HIST_MASTER_INSTRUMENT_REFERENCE` | `ISSUERID, ISIN, ASSET_TYPE` | join MSCI ↔ returns |
| `HIST_MASTER_ISSUER_REFERENCE` | `ISSUERID, GICS_SECTOR, ISSUER_CNTRY_DOMICILE, PARENT_ULTIMATE_ISSUERID` | industry FE, country, ultimate-parent clustering |


In [ ]:
# --- 1A  Verify MSCI parquet files and show schemas -------------------------------
import pyarrow.parquet as pq
for key in ['msci_scores','msci_contr','msci_instr','msci_issuer']:
    p = PATHS[key]
    assert os.path.exists(p), f'MISSING: {p}  (place the licensed MSCI file here)'
    print(f'\n{key}: {os.path.basename(p)}')
    print('   columns:', pq.read_schema(p).names[:12], '...')


### 1B — GDELT 2.0 external clock (when each controversy became public)

MSCI records *when it logged* a controversy, not *when it happened*. The external clock dates the **first public report** of each controversy independently, from news, so we can measure MSCI's logging lag. Method (frozen in `run_meta.json`):

* **Universe:** the 14,207 US+GB controversy onsets (English-language precision; excludes the 1-Jul-2019 backfill and 31-Aug-2017 floor dates).
* **Query:** for each onset, GDELT DOC API over `[record_date − 3 years, record_date]`, countries `US,GB`, company name(s) **AND** key-issue terms from the 718-term dictionary (`K0_keyword_dictionary.csv`), matched in article **body**, `max_fetch = 15`.
* **Output:** `keyword_dated_clock.csv` — earliest matched article date per onset (the automated clock). This is deliberately noisy; it is then screened (Claude AI to drop paywalls/false hits) and **hand-verified** down to the 223-case manuscript clock (Section 4, `audit/lag_v3.py`).


In [ ]:
# --- 1B  GDELT retrieval of the per-controversy public-report clock ----------------
# Reconstructs keyword_dated_clock.csv. Runtime is hours for the full 14,207 onsets, so
# GDELT is polite-rate-limited here; run in batches and cache. Requires no key (public API).
import requests, time, datetime as dt

RUN_META = dict(countries='US,GB', match_on='body', max_fetch=15,
                span_start='2015-09-01', span_end='2026-10-20', seed=7)

def gdelt_doc_query(company, issue_terms, d0, d1, maxrec=15):
    # Build a GDELT DOC 2.0 query: company name AND (any issue term), English, US/GB.
    terms = ' OR '.join(f'"{t}"' for t in issue_terms)
    q = f'"{company}" ({terms}) sourcelang:english'
    url = ('https://api.gdeltproject.org/api/v2/doc/doc'
           f'?query={requests.utils.quote(q)}&mode=artlist&format=json&maxrecords={maxrec}'
           f'&startdatetime={d0:%Y%m%d}000000&enddatetime={d1:%Y%m%d}000000&sort=dateasc')
    try:
        r = requests.get(url, timeout=30)
        arts = r.json().get('articles', [])
    except Exception:
        arts = []
    return arts

def build_gdelt_clock(onsets, kdict, out_csv):
    # onsets: DataFrame[ISSUERID, company, key_issue, record_date]; kdict: K0 dictionary.
    rows = []
    for _, o in onsets.iterrows():
        terms = kdict.loc[kdict.key_issue == o.key_issue, 'term'].tolist()
        d1 = pd.to_datetime(o.record_date); d0 = d1 - pd.DateOffset(years=3)
        arts = gdelt_doc_query(o.company, terms[:40], d0, d1, RUN_META['max_fetch'])
        first = min((a['seendate'] for a in arts), default=None)
        rows.append(dict(ISSUERID=o.ISSUERID, key_issue=o.key_issue,
                         record_date=o.record_date, first_report=first, n_articles=len(arts)))
        time.sleep(1.0)                      # GDELT rate-limit courtesy
    out = pd.DataFrame(rows)
    out.to_csv(out_csv, index=False)
    return out

# kdict = pd.read_csv(PATHS['keyword_dict'])
# onsets = ...   # built in Section 3 (issue-level onset panel) with company names joined
# build_gdelt_clock(onsets, kdict, PATHS['gdelt_clock'])
print('GDELT clock builder defined (uncomment to run on your onset list).')


### 1C — WRDS: CRSP daily, Compustat annual, CRSP Mutual-Fund Holdings

WRDS supplies the US price and fundamentals data. We pull three things:

* **CRSP daily stock file** (`crsp.dsf`) — daily returns for the short-window price event studies (the 116 US-listed verified controversies).
* **Compustat annual** (`comp.funda`) — shares outstanding and price (market cap) for firm size / the ownership dollar figure.
* **CRSP Mutual-Fund Holdings** (`crsp.s12` / thomson) — ESG-fund and index-fund ownership for the reallocation tests.

You need a WRDS account; `wrds.Connection()` prompts for it the first time and caches a `.pgpass`.

In [ ]:
# --- 1C  WRDS pulls ---------------------------------------------------------------
import wrds
db = wrds.Connection()          # prompts for WRDS username/password on first use

# (i) CRSP daily returns for the event studies (keep it to the firms you need via a permno list)
crsp_daily = db.raw_sql('''
    SELECT permno, date, ret, prc, shrout, vol
    FROM crsp.dsf
    WHERE date BETWEEN '2015-01-01' AND '2026-07-31'
''', date_cols=['date'])
crsp_daily.to_parquet(PATHS['crsp_daily'])

# (ii) Compustat annual — market cap = csho * prcc_f
comp = db.raw_sql('''
    SELECT gvkey, datadate, cusip, csho, prcc_f, at, ceq
    FROM comp.funda
    WHERE indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C'
      AND datadate BETWEEN '2015-01-01' AND '2026-07-31'
''', date_cols=['datadate'])
comp.to_parquet(PATHS['compustat'])

# (iii) CRSP Mutual-Fund Holdings (ESG / index funds — ownership reallocation tests)
mf = db.raw_sql('''
    SELECT h.crsp_portno, h.permno, h.report_dt, h.nbr_shares, f.crsp_obj_cd, f.fund_name
    FROM crsp.holdings h LEFT JOIN crsp.fund_hdr f USING (crsp_fundno)
    WHERE h.report_dt BETWEEN '2012-01-01' AND '2026-07-31'
''', date_cols=['report_dt'])
mf.to_parquet(PATHS['mf_holdings'])
print('WRDS pulls cached to Drive.')


### 1D — Refinitiv / LSEG: monthly total returns (global)

The main monthly return panel (`monthly_total_returns.csv`) is the LSEG/Refinitiv **total-return index** by ISIN, which lets us compute country-adjusted monthly returns for the global sample (not just US). Pulled with the Refinitiv Data Library and an app key (Workspace/Eikon running, or an RDP account).

In [ ]:
# --- 1D  Refinitiv/LSEG monthly total returns -------------------------------------
import refinitiv.data as rd
rd.open_session(app_key='YOUR_APP_KEY')     # <-- your Refinitiv app key / Workspace session

def lseg_monthly_total_returns(isins, start='2013-01-01', end='2026-07-31'):
    # Pulls the monthly total-return index (TR.TotalReturn) + country of domicile per ISIN.
    out = []
    for chunk in [isins[i:i+200] for i in range(0, len(isins), 200)]:   # API batch limit
        df = rd.get_history(universe=chunk,
                            fields=['TR.TotalReturn','TR.TotalReturn.date','TR.HQCountryCode'],
                            interval='monthly', start=start, end=end)
        out.append(df)
    res = pd.concat(out).reset_index()
    res.columns = ['isin','date','tr_return_pct','ISSUER_CNTRY_DOMICILE']
    return res

# isins = pd.read_parquet(PATHS['msci_instr']).ISIN.dropna().unique().tolist()
# lseg_monthly_total_returns(isins).to_csv(PATHS['lseg_returns'], index=False)
print('LSEG return puller defined (uncomment to run on your ISIN list).')


## Section 2 — Cleaning & matching

Three operations turn the raw sources into analysis-ready keys:

1. **Point-in-time (valid-time) rule.** MSCI restates history, so for each `(ISSUERID, field, period)` we take the **latest `VERSION_START_DATE`** per `START_DATE`, then the most recent `START_DATE ≤ period-end` (a duckdb `ASOF` join). For the per-severity controversy counts we additionally require `START_DATE ≤ year-end ≤ END_DATE` (valid-time), which fixes the Aug-2020 reissue. This removes back-filled restatements that would otherwise leak future information.
2. **Keys.** `ISIN ↔ ISSUERID` (instrument reference, `ASSET_TYPE='CMN'`) joins MSCI to returns; `PARENT_ULTIMATE_ISSUERID` is the clustering unit (subsidiaries inherit a parent's rating, so they are not independent).
3. **Winsorisation.** Forward returns are winsorised 1/99 (`fwd12w`); monthly event-study returns 0.5/99.5.

The code that performs all of this lives inside the Section 3 construction script (it is cleanest to build the panels and apply the as-of rule in one pass), so the canonical implementation is the `asof_total` / `validtime_severity` functions shown there.

In [ ]:
# --- 2.1  The as-of (valid-time) helper, shown standalone for reference ------------
# This is the exact rule the construction script applies; it is reproduced here so the
# cleaning step is legible on its own. It returns one point-in-time value per issuer-year.
con = duckdb.connect(); con.sql("PRAGMA threads=4")
def asof_latest(field, src, years=range(2016,2027)):
    yl = '['+','.join(map(str,years))+']'
    return con.sql(f'''
      WITH l AS (SELECT ISSUERID, START_DATE, TRY_CAST(VALUE AS DOUBLE) v,
                   ROW_NUMBER() OVER(PARTITION BY ISSUERID,START_DATE ORDER BY VERSION_START_DATE DESC) rn
                 FROM '{src}' WHERE FIELD_ID='{field}' AND START_DATE>=20150101),
           s AS (SELECT ISSUERID, START_DATE, v FROM l WHERE rn=1 AND v IS NOT NULL),
           g AS (SELECT DISTINCT ISSUERID, y, y*10000+1231 thr FROM s CROSS JOIN (SELECT UNNEST({yl}) y))
      SELECT g.ISSUERID, g.y yr, x.v FROM g
      ASOF LEFT JOIN s x ON g.ISSUERID=x.ISSUERID AND g.thr>=x.START_DATE
      WHERE x.v IS NOT NULL''').df()
# Example: IAS as of each year-end  ->  asof_latest('INDUSTRY_ADJUSTED_SCORE', PATHS['msci_scores'])
print('as-of helper defined')


## Section 3 — Variable construction (build the panels)

This is the heart of the pipeline. `02_table2_table3_severity.py` reads the cleaned MSCI data and builds the **canonical firm-year panel** `panel_fy_validtime.parquet` plus the per-severity decomposition, applying the valid-time rule from Section 2. Every downstream table reads this panel, so **run this cell first**.

Panels it defines (see `docs/CONSTRUCTION_DOCUMENTATION.md`):
* **P1 firm-year** (`panel_fy`): IAS, WAKIS, controversy counts by severity (minor/moderate/severe/very-severe), their adjacent-year first differences (`d_ias`, `d_was`, `d_sev`…), and the one-year lead `dias_t1`.
* **Binding measure**: per key issue the single highest-severity controversy (MSCI's single-highest-deduction rule).
* **Onset**: a positive month-to-month increment of a severity's count, dated to the observation month.


In [ ]:
# ====================================================================================
# PIPELINE FILE: pipeline/02_table2_table3_severity.py
# CONSTRUCTS the canonical firm-year panel (panel_fy_validtime.parquet) + Tables 2, 3, A2.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

# =====================================================================================
# 02_table2_table3_severity.py  (v25 — BITEMPORALLY CORRECTED)
# Table 2 (severity levels: count + binding), Table 3 (early vs late), Appendix Table A2.
#
# CORRECTIONS baked in (vs the earlier version):
#  1. VALID-TIME controversy counts: a controversy field counts in year t only if
#     START_DATE <= Dec31(t) <= END_DATE. The earlier "latest START_DATE" as-of (a)
#     ignored END_DATE and (b) summed co-dated fields before the as-of, which jointly
#     mis-measured the per-severity counts (notably around the Aug-2020 reissue).
#  2. ULTIMATE-PARENT clustering: SE clustered by PARENT_ULTIMATE_ISSUERID (subsidiaries
#     inherit a parent's rating and are not independent).  Parents-only = robustness.
#  3. BINDING treatment: per Key Issue, the single highest-severity controversy (MSCI's
#     single-highest-deduction rule); binding count at level s = # issues whose max = s.
#  The IAS outcome and the TOTAL_NUMBER_CONTROVERSIES field chain continuously and are
#  unaffected; only the per-severity decomposition changes.
# =====================================================================================
import duckdb, pandas as pd, numpy as np, glob, warnings
warnings.filterwarnings('ignore')
con=duckdb.connect(); con.sql("PRAGMA memory_limit='3GB'"); con.sql("PRAGMA temp_directory='tmp'"); con.sql('PRAGMA threads=4'); con.sql('PRAGMA disable_progress_bar')
SS='jul/HIST_SUMMARY_SCORES_20260731.parquet'; CF='jul/HIST_CONTROVERSIES_20260731.parquet'
REF=glob.glob('mref/HIST_MASTER_ISSUER_REFERENCE*')[0]
yl='['+','.join(str(y) for y in range(2016,2027))+']'

def asof_total(field, src):
    # continuous-chaining fields (IAS, TOTAL): latest START_DATE <= year-end
    return con.sql(f"""WITH l AS (SELECT ISSUERID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,
        ROW_NUMBER() OVER(PARTITION BY ISSUERID,START_DATE ORDER BY VERSION_START_DATE DESC) rn
        FROM '{src}' WHERE FIELD_ID='{field}' AND START_DATE>=20150101),
      s AS (SELECT ISSUERID,START_DATE,v FROM l WHERE rn=1 AND v IS NOT NULL),
      g AS (SELECT DISTINCT ISSUERID,y,y*10000+1231 thr FROM s CROSS JOIN (SELECT UNNEST({yl}) y))
      SELECT g.ISSUERID,g.y yr,x.v FROM g ASOF LEFT JOIN s x ON g.ISSUERID=x.ISSUERID AND g.thr>=x.START_DATE WHERE x.v IS NOT NULL""").df()

def validtime_severity():
    # per (issuer, issue, year): valid-time value at year-end for mod/sev/vs (START<=thr<=END)
    return con.sql(f"""
    WITH raw AS (SELECT ISSUERID, REGEXP_REPLACE(FIELD_ID,'_(VERY_SEVERE|SEVERE|MODERATE)_CONTROVERSIES$','') issue,
        CASE WHEN FIELD_ID LIKE '%VERY\\_SEVERE\\_CONTROVERSIES' ESCAPE '\\' THEN 'vs'
             WHEN FIELD_ID LIKE '%\\_MODERATE\\_CONTROVERSIES' ESCAPE '\\' THEN 'mod' ELSE 'sev' END lvl,
        START_DATE, END_DATE, TRY_CAST(VALUE AS DOUBLE) v,
        ROW_NUMBER() OVER(PARTITION BY ISSUERID,FIELD_ID,START_DATE ORDER BY VERSION_START_DATE DESC) rn
        FROM '{CF}' WHERE (FIELD_ID LIKE '%\\_MODERATE\\_CONTROVERSIES' ESCAPE '\\'
                        OR FIELD_ID LIKE '%\\_SEVERE\\_CONTROVERSIES' ESCAPE '\\'
                        OR FIELD_ID LIKE '%VERY\\_SEVERE\\_CONTROVERSIES' ESCAPE '\\')),
     r AS (SELECT ISSUERID,issue,lvl,START_DATE,END_DATE,v FROM raw WHERE rn=1 AND v IS NOT NULL),
     grid AS (SELECT DISTINCT ISSUERID,issue,lvl,y,y*10000+1231 thr FROM r CROSS JOIN (SELECT UNNEST({yl}) y)),
     valid AS (SELECT g.ISSUERID,g.issue,g.lvl,g.y,x.v,
                 ROW_NUMBER() OVER(PARTITION BY g.ISSUERID,g.issue,g.lvl,g.y ORDER BY x.START_DATE DESC) rn2
               FROM grid g JOIN r x ON x.ISSUERID=g.ISSUERID AND x.issue=g.issue AND x.lvl=g.lvl
                                    AND x.START_DATE<=g.thr AND x.END_DATE>=g.thr),
     iv AS (SELECT ISSUERID,issue,y, MAX(CASE WHEN lvl='mod' THEN v END) m,
                   MAX(CASE WHEN lvl='sev' THEN v END) s, MAX(CASE WHEN lvl='vs' THEN v END) vv
            FROM valid WHERE rn2=1 GROUP BY 1,2,3)
     SELECT ISSUERID, y yr, SUM(COALESCE(m,0)) c_mod, SUM(COALESCE(s,0)) c_sev, SUM(COALESCE(vv,0)) c_vs,
       SUM(CASE WHEN COALESCE(vv,0)>0 THEN 1 ELSE 0 END) b_vs,
       SUM(CASE WHEN COALESCE(vv,0)=0 AND COALESCE(s,0)>0 THEN 1 ELSE 0 END) b_sev,
       SUM(CASE WHEN COALESCE(vv,0)=0 AND COALESCE(s,0)=0 AND COALESCE(m,0)>0 THEN 1 ELSE 0 END) b_mod
     FROM iv GROUP BY 1,2""").df()

ias=asof_total('INDUSTRY_ADJUSTED_SCORE',SS).rename(columns={'v':'ias'})
tot=asof_total('TOTAL_NUMBER_CONTROVERSIES',CF).rename(columns={'v':'tot'})
was=asof_total('WEIGHTED_AVERAGE_SCORE',SS).rename(columns={'v':'was'})
sev=validtime_severity()
ref=pd.read_parquet(REF,columns=['ISSUERID','PARENT_ULTIMATE_ISSUERID']).drop_duplicates('ISSUERID')
P=ias.merge(tot,on=['ISSUERID','yr'],how='outer').merge(was,on=['ISSUERID','yr'],how='outer').merge(sev,on=['ISSUERID','yr'],how='left').merge(ref,on='ISSUERID',how='left')
P['group']=P['PARENT_ULTIMATE_ISSUERID'].fillna(P['ISSUERID'])
for c in ['tot','c_mod','c_sev','c_vs','b_mod','b_sev','b_vs']: P[c]=P[c].fillna(0)
P['c_minor']=(P['tot']-(P.c_mod+P.c_sev+P.c_vs)).clip(lower=0)          # proper residual
P=P.sort_values(['ISSUERID','yr']); g=P.groupby('ISSUERID'); P['py']=g.yr.shift(1); P['ny1']=g.yr.shift(-1)
for c in ['c_minor','c_mod','c_sev','c_vs','b_mod','b_sev','b_vs','tot']:
    P['d_'+c]=np.where(P.yr-P.py==1, P[c]-g[c].shift(1), np.nan)
P['dias']=np.where(P.yr-P.py==1, P.ias-g.ias.shift(1), np.nan)
P['dwas']=np.where(P.yr-P.py==1, P.was-g.was.shift(1), np.nan)
P['dias_t1']=np.where(P.ny1-P.yr==1, g.ias.shift(-1)-P.ias, np.nan)

def clustered(dat,y,x,cluster):
    d=dat.dropna(subset=[y,x,cluster]).copy()
    if len(d)<30 or d[x].std()==0: return (np.nan,np.nan,len(d))
    d['yr']=d.yr.astype(int); yrs=sorted(d.yr.unique())
    X=np.column_stack([np.ones(len(d))]+[(d.yr==yy).astype(float) for yy in yrs[1:]]+[d[x].values]); Y=d[y].values
    XtXi=np.linalg.inv(X.T@X); b=XtXi@(X.T@Y); e=Y-X@b
    S=pd.DataFrame(X*e[:,None]).groupby(d[cluster].values).sum().values; meat=S.T@S
    N,k=X.shape; G=S.shape[0]; c=(G/(G-1))*((N-1)/(N-k)); se=np.sqrt(np.diag(c*(XtXi@meat@XtXi)))
    return round(b[-1],4), round(b[-1]/se[-1],1), int(N)

F=P[P.yr.isin([2018,2019,2021,2022,2023,2024,2025])]; FP=F[F.PARENT_ULTIMATE_ISSUERID.isna()]
print("=== TABLE 2: severity levels (valid-time; cluster=ultimate parent | parents-only) ===")
for lab,cc,bb in [('Minor/residual','c_minor',None),('Moderate','c_mod','b_mod'),('Severe','c_sev','b_sev'),('Very severe','c_vs','b_vs')]:
    t0=clustered(F,'dias','d_'+cc,'group'); t1=clustered(F,'dias_t1','d_'+cc,'group')
    po=clustered(FP,'dias','d_'+cc,'group')
    bind=clustered(F,'dias','d_'+bb,'group') if bb else (None,)
    print(f"  {lab:14} mean {F[cc].mean():6.3f} | t0 {t0[0]:+.4f}({t0[1]:+.1f})  t+1 {t1[0]:+.4f}({t1[1]:+.1f})  PO_t0 {po[0]:+.4f}({po[1]:+.1f})"
          + (f"  BINDING {bind[0]:+.4f}({bind[1]:+.1f})" if bb else ""))
print("\n=== TABLE 3: early vs late (cluster=ultimate parent) ===")
for nm,ys in [('early',[2018,2019]),('late',[2021,2022,2023,2024,2025]),('full',[2018,2019,2021,2022,2023,2024,2025])]:
    d=P[P.yr.isin(ys)]
    print(f"  [{nm:5}] agg_t0 {clustered(d,'dias','d_tot','group')[:2]}  agg_t1 {clustered(d,'dias_t1','d_tot','group')[:2]}  sev_t0 {clustered(d,'dias','d_c_sev','group')[:2]}  sev_t1 {clustered(d,'dias_t1','d_c_sev','group')[:2]}")
print("\n=== APPENDIX A2: year-by-year severe (cluster=ultimate parent) ===")
for Y in range(2018,2027):
    print(f"  {Y}: sev {clustered(P[P.yr==Y],'dias','d_c_sev','group')[:2]}  agg {clustered(P[P.yr==Y],'dias','d_tot','group')[:2]}")
# save corrected panel + exhibit CSVs for downstream scripts
P.to_parquet('panel_fy_validtime.parquet')
print("\nsaved panel_fy_validtime.parquet ; N(analysis)=",F.dropna(subset=['dias','d_c_sev']).shape[0])


The monthly, issue-level, pricing and event-study panels (P2–P5) are built inside the test scripts that need them (each reads the MSCI files and the panel above). Their construction logic is documented in `CONSTRUCTION_DOCUMENTATION.md` §1 and appears in the cells below where it is first used (e.g. the monthly band panel in `04`, the event-study `ra`/`car6` panel in the downgrade-drift script, the returns `ra`/`mom`/`cmadj`/`ff4` panel in `downgrade_followup/H3_build_and_run.py`).

## Section 4 — Tests, tables & figures

Each cell below is one script from the pipeline, embedded verbatim with a header note saying which manuscript exhibit it produces. Grouped by theme. Run after Section 3.

### 4.1 Summary statistics & the own-score channel

**`pipeline/01_table1_table4_revfreq.py`** — Table 1 (summary stats) + Table 4 (own-score share of rating-change variance) + the §6.1 revision-frequency statistic.

In [ ]:
# ====================================================================================
# PIPELINE FILE: pipeline/01_table1_table4_revfreq.py
# Table 1 (summary stats) + Table 4 (own-score share of rating-change variance) + the §6.1 revision-frequency statistic.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

# 01_table1_table4_revfreq.py (v26) — Table 1 (summary stats) + Table 4 (own-score share).
# Reads the canonical valid-time panel built by 02 (panel_fy_validtime.parquet); clusters by
# ultimate parent.  Run 02 first.
import pandas as pd, numpy as np, glob
P=pd.read_parquet("panel_fy_validtime.parquet")
W=[2018,2019,2021,2022,2023,2024,2025]
A=P[P.dias.notna() & P.yr.isin(W)].copy()
# --- Table 1 ---
def st(x): x=x.dropna().astype(float); return (int(x.size),round(x.mean(),3),round(x.std(),3),round(x.quantile(.25),2),round(x.quantile(.5),2),round(x.quantile(.75),2))
print("=== TABLE 1 (canonical, ΔIAS analysis sample) ===  N mean sd p25 med p75")
for lab,c in [("IAS","ias"),("WAKIS","was"),("Total controversies","tot"),("Moderate","c_mod"),
              ("Severe","c_sev"),("Very severe","c_vs"),("Minor/residual","c_minor"),("dIAS","dias"),("dWAKIS","dwas")]:
    print(f"  {lab:20}",st(A[c]))
# --- Table 4: own-score share R^2 (dias ~ dwas) across FE specs ---
def r2(df,fe):
    d=df[["dias","dwas","yr"]].dropna()
    if fe=="pooled": X=np.column_stack([np.ones(len(d)),d.dwas.values])
    else:
        yrs=sorted(d.yr.unique()); X=np.column_stack([np.ones(len(d))]+[(d.yr==y).astype(float) for y in yrs[1:]]+[d.dwas.values])
    y=d.dias.values; b=np.linalg.lstsq(X,y,rcond=None)[0]; e=y-X@b; return 1-(e@e)/(((y-y.mean())**2).sum())
d=A[["dias","dwas","ISSUERID"]].dropna(); yw=(d.dias-d.groupby("ISSUERID").dias.transform("mean")).values
xw=(d.dwas-d.groupby("ISSUERID").dwas.transform("mean")).values; bw=(xw@yw)/(xw@xw); ew=yw-bw*xw
print("\n=== TABLE 4 own-score share R^2 ===")
print(f"  pooled {r2(A,'pooled')*100:.1f}%  +yearFE {r2(A,'year')*100:.1f}%  within-firm {(1-(ew@ew)/((yw**2).sum()))*100:.1f}%")
# --- revision frequency (Table 4 panel b / 6.1): share of firm-years with no change in severe count ---
A2=A.dropna(subset=['d_c_sev']); print(f"\n6.1 no-change in severe count: {(A2.d_c_sev==0).mean()*100:.1f}%")


**`pipeline/03_channel_passthrough.py`** — Table 5 channel: the WAKIS→IAS pass-through (how much of the controversy signal survives benchmarking).

In [ ]:
# ====================================================================================
# PIPELINE FILE: pipeline/03_channel_passthrough.py
# Table 5 channel: the WAKIS→IAS pass-through (how much of the controversy signal survives benchmarking).
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

# 03_channel_passthrough.py (v26) — asymmetric pass-through of WAKIS changes into IAS (6.4).
# Reads the canonical valid-time panel (02); clusters by ultimate parent.
import pandas as pd, numpy as np
P=pd.read_parquet("panel_fy_validtime.parquet")
W=[2018,2019,2021,2022,2023,2024,2025]; A=P[P.dias.notna() & P.yr.isin(W)].copy()
A["dwp"]=A.dwas.clip(lower=0); A["dwn"]=A.dwas.clip(upper=0)
def cl(df,xvs):
    d=df[["dias","yr","group"]+xvs].dropna(); yrs=sorted(d.yr.unique())
    X=np.column_stack([np.ones(len(d))]+[(d.yr==y).astype(float) for y in yrs[1:]]+[d[x].values for x in xvs]); y=d.dias.values
    XtXi=np.linalg.inv(X.T@X); b=XtXi@(X.T@y); e=y-X@b
    S=pd.DataFrame(X*e[:,None]).groupby(d["group"].values).sum().values; meat=S.T@S
    N,k=X.shape; G=S.shape[0]; c=(G/(G-1))*((N-1)/(N-k)); V=c*(XtXi@meat@XtXi); se=np.sqrt(np.diag(V))
    # Wald test pos=neg
    R=np.zeros(X.shape[1]); R[-2]=1; R[-1]=-1; diff=R@b; sed=np.sqrt(R@V@R); 
    return {xvs[i]:(round(b[-len(xvs)+i],3),round((b/se)[-len(xvs)+i],1)) for i in range(len(xvs))}, round(diff,3), round(diff/sed,1)
res,diff,zt=cl(A,["dwp","dwn"])
print("=== 6.4 ASYMMETRIC PASS-THROUGH (parent-clustered) ===")
print(f"  positive {res['dwp']}  negative {res['dwn']}  diff {diff} (z={zt})")


**`robustness/a5_implied.py`** — Methodology-implied ΔWAKIS check (does the observed IAS move match MSCI's stated deduction rule?).

In [ ]:
# ====================================================================================
# PIPELINE FILE: robustness/a5_implied.py
# Methodology-implied ΔWAKIS check (does the observed IAS move match MSCI's stated deduction rule?).
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""Analysis 5 (robustness): methodology-implied change in WAKIS from MSCI's controversy-deduction schedule vs observed ΔWAKIS."""
import duckdb, numpy as np, pandas as pd
W = "/home/claude/w2"; SS = f"{W}/jul/HIST_SUMMARY_SCORES_20260731.parquet"; CF = f"{W}/jul/HIST_CONTROVERSIES_20260731.parquet"
# midpoints of the non-structural/structural deduction range, by highest severity in the key issue
DED = {0: 0.0, 1: (0.8 + 1.3) / 2, 2: (1.7 + 2.5) / 2, 3: (3.0 + 5.0) / 2}
WMAP = {"TOXIC_EMISS": "TOXIC_EMISS_WSTE_WEIGHT"}
GOV = {"BUS_ETHICS_ETHICSFRAUD", "BUS_ETHICS_ANTICOMP", "BUS_ETHICS_CORRUPTION"}      # governance: deduction works through Key Metrics
A = pd.read_parquet("/home/claude/evt/panelA_vt_july.parquet").rename(columns={"mod": "MOD"})
A = A[~A.issue.isin(GOV)].copy()
A["h"] = np.select([A.vsev > 0, A.sev > 0, A.MOD > 0], [3, 2, 1], 0)
A = A.sort_values(["ISSUERID", "issue", "yr"]); g = A.groupby(["ISSUERID", "issue"])
A["h_prev"] = np.where(A.yr - g.yr.shift(1) == 1, g.h.shift(1), 0)          # issue not covered in t-1: treated as 0
A["dded"] = A.h.map(DED) - pd.Series(A.h_prev).map(DED).values
A["wfield"] = A.issue.map(lambda s: WMAP.get(s, f"{s}_WEIGHT"))
con = duckdb.connect(); con.sql("PRAGMA threads=6")
fl = pd.DataFrame({"FIELD_ID": A.wfield.unique()}); con.register("fl", fl)
w = con.sql(f"""WITH l AS (SELECT ISSUERID,FIELD_ID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,ROW_NUMBER() OVER(PARTITION BY ISSUERID,FIELD_ID,START_DATE ORDER BY VERSION_START_DATE DESC) rn
   FROM '{SS}' JOIN fl USING(FIELD_ID) WHERE START_DATE>=20150101), r AS (SELECT * FROM l WHERE rn=1 AND v IS NOT NULL), y AS (SELECT UNNEST(range(2016,2027)) yr)
   SELECT r.ISSUERID, r.FIELD_ID wfield, y.yr, arg_max(v,START_DATE) wt FROM r JOIN y ON START_DATE<=yr*10000+1231 GROUP BY 1,2,3""").df()
print("weight scale (median, p99):", w.wt.median(), w.wt.quantile(.99))
A = A.merge(w, on=["ISSUERID", "wfield", "yr"], how="left")
A["covered_w"] = A.wt.notna()
A["contrib"] = -(A.wt.fillna(0) / 100.0) * A.dded                   # weights are in percent
imp = A.groupby(["ISSUERID", "yr"]).agg(implied=("contrib", "sum"), n_chg=("dded", lambda s: int((s != 0).sum())),
                                         n_chg_w=("contrib", lambda s: int((s != 0).sum()))).reset_index()
P = pd.read_parquet(f"{W}/panel_fy_validtime.parquet")
fs = con.sql(f"SELECT ISSUERID, MIN(START_DATE)//10000 fs FROM '{CF}' WHERE regexp_matches(FIELD_ID,'_(MODERATE|SEVERE|VERY_SEVERE)_CONTROVERSIES$') GROUP BY 1").df()
ft = con.sql(f"SELECT ISSUERID, MIN(START_DATE)//10000 ft FROM '{CF}' WHERE FIELD_ID='TOTAL_NUMBER_CONTROVERSIES' GROUP BY 1").df()
bf = pd.read_csv(f"{W}/out/A2_bf_flags.csv") if False else None
P = P.merge(fs, on="ISSUERID", how="left").merge(ft, on="ISSUERID", how="left").merge(imp, on=["ISSUERID", "yr"], how="left")
P["implied"] = P.implied.fillna(0.0); P["n_chg"] = P.n_chg.fillna(0); P["n_chg_w"] = P.n_chg_w.fillna(0)
F = P[P.yr.isin([2018, 2019, 2021, 2022, 2023, 2024, 2025])].dropna(subset=["dias", "d_c_sev", "dwas"]).copy()
F = F[~((F.fs == F.yr) | (F.ft == F.yr)) & (F.yr != 2019) | ((F.yr == 2019) & ~((F.fs == F.yr) | (F.ft == F.yr)))]


def cl(d, y, x, c="group"):
    d = d.dropna(subset=[y, x]).copy(); d["yr"] = d.yr.astype(int); yrs = sorted(d.yr.unique())
    X = np.column_stack([np.ones(len(d))] + [(d.yr == yy).astype(float) for yy in yrs[1:]] + [d[x].values]); Y = d[y].values
    XtXi = np.linalg.inv(X.T @ X); b = XtXi @ (X.T @ Y); e = Y - X @ b
    S = pd.DataFrame(X * e[:, None]).groupby(d[c].values).sum().values; N, k = X.shape; G = S.shape[0]
    se = np.sqrt(np.diag((G / (G - 1)) * ((N - 1) / (N - k)) * (XtXi @ (S.T @ S) @ XtXi)))
    return b[-1], b[-1] / se[-1], N, G


rows = []
for lab, D in [("All firm-years (implied = 0 where no change)", F),
               ("Firm-years with a weighted key-issue severity change", F[F.n_chg_w > 0]),
               ("  of which: severity rises only (onsets)", F[(F.n_chg_w > 0) & (F.implied < 0)]),
               ("  of which: a single changed key issue", F[F.n_chg_w == 1])]:
    b, t, n, gg = cl(D, "dwas", "implied")
    b2, t2, _, _ = cl(D, "dias", "implied")
    rows.append(dict(sample=lab, slope_dWAKIS=b, t_dWAKIS=t, slope_dIAS=b2, t_dIAS=t2, N=n, clusters=gg,
                     mean_implied=D.implied.mean(), mean_dWAKIS=D.dwas.mean()))
T = pd.DataFrame(rows); T.to_csv(f"{W}/out/A5_implied.csv", index=False)
pd.set_option("display.width", 220); print(T.round(4).to_string())
print("share of changed issue-years with no weight in force:", round(1 - A[A.dded != 0].covered_w.mean(), 3))


### 4.2 Severity-graded incorporation & timing

**`robustness/a2_table2_backfill.py`** — Table 3 robustness: re-estimates the timing excluding the 2019 backfill / re-rating dates.

In [ ]:
# ====================================================================================
# PIPELINE FILE: robustness/a2_table2_backfill.py
# Table 3 robustness: re-estimates the timing excluding the 2019 backfill / re-rating dates.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""Analysis 2: Table 2 (contemporaneous and t+1) without firm-years whose severity change comes from onsets dated
2019-07-01 (backfill) or 2017-08-31 (floor); overlap with the first-covered-year flag. Paper sample, parent clusters."""
import duckdb, numpy as np, pandas as pd
W = "/home/claude/w2"; CF = f"{W}/jul/HIST_CONTROVERSIES_20260731.parquet"
P = pd.read_parquet(f"{W}/panel_fy_validtime.parquet")
con = duckdb.connect()
sevre = "regexp_matches(FIELD_ID,'_(MODERATE|SEVERE|VERY_SEVERE)_CONTROVERSIES$')"
fs = con.sql(f"SELECT ISSUERID, MIN(START_DATE)//10000 fs FROM '{CF}' WHERE {sevre} GROUP BY 1").df()
ft = con.sql(f"SELECT ISSUERID, MIN(START_DATE)//10000 ft FROM '{CF}' WHERE FIELD_ID='TOTAL_NUMBER_CONTROVERSIES' GROUP BY 1").df()
# issuer-years with a positive severity record starting on a backfill/floor date (latest version per record)
bf = con.sql(f"""WITH l AS (SELECT ISSUERID,FIELD_ID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,
    ROW_NUMBER() OVER(PARTITION BY ISSUERID,FIELD_ID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{CF}' WHERE {sevre})
    SELECT ISSUERID, START_DATE//10000 yr, START_DATE d, COUNT(*) n FROM l WHERE rn=1 AND v>0 AND START_DATE IN (20190701,20170831) GROUP BY 1,2,3""").df()
print("issuers with a positive severity record on 2019-07-01:", bf[bf.d == 20190701].ISSUERID.nunique(),
      "| on 2017-08-31:", bf[bf.d == 20170831].ISSUERID.nunique())
P = P.merge(fs, on="ISSUERID", how="left").merge(ft, on="ISSUERID", how="left")
P = P.merge(bf.groupby(["ISSUERID", "yr"]).size().rename("bfn").reset_index(), on=["ISSUERID", "yr"], how="left")
P["first"] = (P.fs == P.yr) | (P.ft == P.yr)
P["backfill"] = P.bfn.notna()
Wy = [2018, 2019, 2021, 2022, 2023, 2024, 2025]
F = P[P.yr.isin(Wy)].dropna(subset=["dias", "d_c_sev"]).copy()
# a flagged firm-year counts only if its severity counts actually change in that year
F["sevchg"] = (F.d_c_mod != 0) | (F.d_c_sev != 0) | (F.d_c_vs != 0)
F["bf_any"] = F.backfill
F["bf_chg"] = F.backfill & F.sevchg
ov = dict(canonical=len(F), backfill_any=int(F.bf_any.sum()), backfill_with_change=int(F.bf_chg.sum()),
          first_covered=int(F["first"].sum()), both=int((F.bf_any & F["first"]).sum()),
          backfill_not_first=int((F.bf_any & ~F["first"]).sum()), first_not_backfill=int((F["first"] & ~F.bf_any).sum()),
          union=int((F.bf_any | F["first"]).sum()))
ov["backfill_by_year"] = F[F.bf_any].groupby("yr").size().to_dict()
print(ov)


def cl(d, y, x, c="group"):
    d = d.dropna(subset=[y, x]).copy(); d["yr"] = d.yr.astype(int); yrs = sorted(d.yr.unique())
    X = np.column_stack([np.ones(len(d))] + [(d.yr == yy).astype(float) for yy in yrs[1:]] + [d[x].values]); Y = d[y].values
    XtXi = np.linalg.inv(X.T @ X); b = XtXi @ (X.T @ Y); e = Y - X @ b
    S = pd.DataFrame(X * e[:, None]).groupby(d[c].values).sum().values; N, k = X.shape; G = S.shape[0]
    se = np.sqrt(np.diag((G / (G - 1)) * ((N - 1) / (N - k)) * (XtXi @ (S.T @ S) @ XtXi)))
    return b[-1], b[-1] / se[-1], N, G


rows = []
samples = {"Table 2 (all)": F, "Excl. backfill/floor onsets": F[~F.bf_any], "Excl. first covered year": F[~F["first"]],
           "Excl. both": F[~(F.bf_any | F["first"])]}
for lab, D in samples.items():
    for c, nm in [("c_minor", "Other (residual)"), ("c_mod", "Moderate"), ("c_sev", "Severe"), ("c_vs", "Very severe")]:
        b0, t0, n0, g0 = cl(D, "dias", "d_" + c); b1, t1, n1, g1 = cl(D, "dias_t1", "d_" + c)
        rows.append(dict(sample=lab, level=nm, b_t=b0, t_t=t0, N_t=n0, G_t=g0, b_t1=b1, t_t1=t1, N_t1=n1, G_t1=g1))
T = pd.DataFrame(rows); T.to_csv(f"{W}/out/A2_table2_backfill.csv", index=False)
pd.Series(ov).to_csv(f"{W}/out/A2_overlap.csv")
pd.set_option("display.width", 250); print(T.round(4).to_string())


**`pipeline/10_robustness_firstcoverage.py`** — Table 3 robustness: restricts the late window to firms already rated in 2018 (acceleration is not a universe-expansion artefact).

In [ ]:
# ====================================================================================
# PIPELINE FILE: pipeline/10_robustness_firstcoverage.py
# Table 3 robustness: restricts the late window to firms already rated in 2018 (acceleration is not a universe-expansion artefact).
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

# =====================================================================================
# 10_robustness_firstcoverage.py  (v33)
# Robustness checks added after reconciling the supervisor memo with the draft.
# Requires panel_fy_validtime.parquet from 02 and the July-2026 extract (jul/), mref/.
#  A. First covered year: MSCI's key-issue severity fields begin in 2019 for most issuers and
#     TOTAL_NUMBER_CONTROVERSIES in 2017-2019. In a firm's first covered year the change in a
#     zero-filled count equals its whole stock. Flag: year == first START_DATE year of either family.
#  B. Table 2 / Table 3 without first covered years (same estimator as 02, parent-clustered).
#  C. Early-window lag on the original design (TOTAL recorded in both t and t-1).
#  D. Severity x Late regression (memo Section 4) on the canonical sample, parent- and firm-clustered.
#  E. Section 6.5 monthly hazard without events in a firm's first 12 months of case-type coverage.
# Canonical sample note: 02 reads IAS records with START_DATE >= 20150101; firm-years whose
# year-end IAS was last updated before 2015 drop out (414,175 -> 401,885). Clustering drops nothing.
# =====================================================================================
import duckdb, pandas as pd, numpy as np, glob, statsmodels.api as sm, warnings
warnings.filterwarnings('ignore')
SS='jul/HIST_SUMMARY_SCORES_20260731.parquet'; CF='jul/HIST_CONTROVERSIES_20260731.parquet'
REF=glob.glob('mref/HIST_MASTER_ISSUER_REFERENCE*')[0]
con=duckdb.connect(); con.sql('PRAGMA threads=4')
P=pd.read_parquet('panel_fy_validtime.parquet')
fs=con.sql(f"SELECT ISSUERID, MIN(START_DATE)//10000 fs FROM '{CF}' WHERE regexp_matches(FIELD_ID,'_(MODERATE|SEVERE|VERY_SEVERE)_CONTROVERSIES$') GROUP BY 1").df()
ft=con.sql(f"SELECT ISSUERID, MIN(START_DATE)//10000 ft FROM '{CF}' WHERE FIELD_ID='TOTAL_NUMBER_CONTROVERSIES' GROUP BY 1").df()
P=P.merge(fs,on='ISSUERID',how='left').merge(ft,on='ISSUERID',how='left'); P['first']=(P.fs==P.yr)|(P.ft==P.yr)

def cl(d,y,xs,c='group'):
    d=d.dropna(subset=[y]+xs).copy(); d['yr']=d.yr.astype(int); yrs=sorted(d.yr.unique())
    X=np.column_stack([np.ones(len(d))]+[(d.yr==yy).astype(float) for yy in yrs[1:]]+[d[x].values for x in xs]); Y=d[y].values
    XtXi=np.linalg.inv(X.T@X); b=XtXi@(X.T@Y); e=Y-X@b
    S=pd.DataFrame(X*e[:,None]).groupby(d[c].values).sum().values; N,k=X.shape; G=S.shape[0]
    se=np.sqrt(np.diag((G/(G-1))*((N-1)/(N-k))*(XtXi@(S.T@S)@XtXi))); k0=len(yrs)
    return {x:(round(float(b[k0+i]),4),round(float(b[k0+i]/se[k0+i]),1)) for i,x in enumerate(xs)}

W=[2018,2019,2021,2022,2023,2024,2025]; F=P[P.yr.isin(W)].dropna(subset=['dias','d_c_sev']).copy(); G=F[~F['first']]
print(f"[A] canonical N={len(F):,}; first-covered firm-years={int(F['first'].sum()):,}", F[F['first']].groupby('yr').size().to_dict())
print("[B] Table 2 t0 (one severity per regression):")
for c in ['c_minor','c_mod','c_sev','c_vs']: print(f"    {c:8} all {cl(F,'dias',['d_'+c])} | excl first {cl(G,'dias',['d_'+c])}")
print("[B] Table 3:")
for nm,ys in [('early',[2018,2019]),('late',[2021,2022,2023,2024,2025])]:
    for lab,D in [('all',F),('excl',G)]:
        d=D[D.yr.isin(ys)]; print(f"    {nm} {lab} N={len(d):,} agg_t0 {cl(d,'dias',['d_tot'])} agg_t1 {cl(d,'dias_t1',['d_tot'])} sev_t0 {cl(d,'dias',['d_c_sev'])} sev_t1 {cl(d,'dias_t1',['d_c_sev'])}")
E=F[F.yr.isin([2018,2019])]
print("[C] early-window lag:")
for lab,D in [('all',E),('TOTAL in both years',E[E.ft<E.yr]),('+ excl severity-first',E[(E.ft<E.yr)&(E.fs!=E.yr)])]:
    print(f"    {lab:24} N={len(D):,} agg t0 {cl(D,'dias',['d_tot'])} t+1 {cl(D,'dias_t1',['d_tot'])} | resid t+1 {cl(D,'dias_t1',['d_c_minor'])}")
print("[D] severity x Late (joint):")
for lab,D in [('all',F),('excl first',G)]:
    D=D.copy(); D['Late']=(D.yr>=2021).astype(float); xs=[]
    for s in ['c_mod','c_sev','c_vs','c_minor']: D['x_'+s]=D['d_'+s]; D['xl_'+s]=D['d_'+s]*D.Late; xs+=['x_'+s,'xl_'+s]
    print(f"    {lab} parent-cl {cl(D,'dias',xs)}"); print(f"    {lab} firm-cl   {cl(D,'dias',xs,'ISSUERID')}")
# ---- E. Section 6.5 ----
mi=lambda d:(d//10000)*12+((d//100)%100); DE=mi(20260731); H=24
ias=con.sql(f"""WITH l AS (SELECT ISSUERID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,ROW_NUMBER() OVER(PARTITION BY ISSUERID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{SS}' WHERE FIELD_ID='INDUSTRY_ADJUSTED_SCORE' AND START_DATE>=20160101) SELECT ISSUERID,START_DATE,v,LAG(v) OVER(PARTITION BY ISSUERID ORDER BY START_DATE) pv FROM l WHERE rn=1 AND v IS NOT NULL""").df()
ias['dias']=ias.v-ias.pv
sev=con.sql(f"""WITH l AS (SELECT ISSUERID,FIELD_ID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,ROW_NUMBER() OVER(PARTITION BY ISSUERID,FIELD_ID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{CF}' WHERE FIELD_ID LIKE 'CONTR\\_%\\_N\\_SEVERE' ESCAPE '\\' AND START_DATE>=20160101), pd AS (SELECT ISSUERID,START_DATE,SUM(v) s FROM l WHERE rn=1 AND v IS NOT NULL GROUP BY 1,2) SELECT ISSUERID,START_DATE,s,LAG(s) OVER(PARTITION BY ISSUERID ORDER BY START_DATE) ps, MIN(START_DATE) OVER(PARTITION BY ISSUERID) AS first_d FROM pd""").df()
ev=sev[(sev.s-sev.ps)>1e-9][['ISSUERID','START_DATE','first_d']].copy(); ev['ev']=mi(ev.START_DATE); yr=ev.START_DATE//10000
ev['era']=np.where((yr>=2018)&(yr<=2019),'early',np.where((yr>=2021)&(yr<=2025),'late','other')); ev=ev[ev.era!='other']; ev['since_first']=ev.ev-mi(ev.first_d)
dg=ias[ias.dias<-1e-9][['ISSUERID','START_DATE']].copy(); dg['dg']=mi(dg.START_DATE)
ev=pd.merge_asof(ev.sort_values('ev'),dg[['ISSUERID','dg']].sort_values('dg'),by='ISSUERID',left_on='ev',right_on='dg',direction='forward')
ev['lag']=ev.dg-ev.ev; ev['cens']=DE-ev.ev
ref=pd.read_parquet(REF,columns=['ISSUERID','PARENT_ULTIMATE_ISSUERID']).drop_duplicates('ISSUERID')
ev=ev.merge(ref,on='ISSUERID',how='left'); ev['grp']=ev.PARENT_ULTIMATE_ISSUERID.fillna(ev.ISSUERID); ev['late']=(ev.era=='late').astype(int)
def cum(k):
    c=1
    for m in range(1,H+1):
        ar=((k.lag.isna()&(k.cens>=m))|(k.lag>=m)).sum(); f=(k.lag==m).sum(); c*=1-(f/ar if ar else 0)
    return round(1-c,3)
def haz(k):
    rows=[]
    for g,late,lag,cens in zip(k.grp,k.late,k.lag.values,k.cens.values):
        h=int(min(H, lag if not np.isnan(lag) else cens))
        for m in range(1,h+1):
            f=1 if (not np.isnan(lag) and lag==m) else 0; rows.append((g,late,m,f))
            if f: break
    pm=pd.DataFrame(rows,columns=['g','late','month','fail'])
    X=sm.add_constant(pd.get_dummies(pm.month,prefix='m',drop_first=True).astype(float)); X['late']=pm.late.astype(float)
    r=sm.GLM(pm.fail,X,family=sm.families.Binomial(sm.families.links.CLogLog())).fit(cov_type='cluster',cov_kwds={'groups':pd.factorize(pm.g)[0]})
    return round(r.params['late'],3),round(r.tvalues['late'],1),round(float(np.exp(r.params['late'])),2)
print("[E] Section 6.5:")
for lab,k in [('as in paper',ev),('excl first 12m of coverage',ev[ev.since_first>=12])]:
    print(f"    {lab:28} early N={int((k.era=='early').sum()):,} cum24={cum(k[k.era=='early'])} | late N={int((k.era=='late').sum()):,} cum24={cum(k[k.era=='late'])} | cloglog late {haz(k)}")


**`pipeline/05_reclass_lead_hazard.py`** — Reclassification lead and the hazard of a letter change after a severe onset.

In [ ]:
# ====================================================================================
# PIPELINE FILE: pipeline/05_reclass_lead_hazard.py
# Reclassification lead and the hazard of a letter change after a severe onset.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

import duckdb, pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
con=duckdb.connect('a.db'); con.sql("PRAGMA memory_limit='2GB'"); con.sql("PRAGMA temp_directory='tmp'"); con.sql('PRAGMA threads=4'); con.sql('PRAGMA disable_progress_bar')
CF='jul/HIST_CONTROVERSIES_20260731.parquet'; SS='jul/HIST_SUMMARY_SCORES_20260731.parquet'
mi=lambda d:(d//10000)*12+((d//100)%100)
# ---- RECLASSIFICATION: per issue, moderate & severe(+vsev) counts over stored dates; at severe increase, what does moderate do? ----
con.sql(f"""CREATE OR REPLACE TABLE mc AS
 WITH raw AS (SELECT ISSUERID, REGEXP_REPLACE(FIELD_ID,'_(VERY_SEVERE|SEVERE|MODERATE)_CONTROVERSIES$','') issue,
    CASE WHEN FIELD_ID LIKE '%MODERATE_CONTROVERSIES' THEN 'mod' ELSE 'sev' END lvl, START_DATE, TRY_CAST(VALUE AS DOUBLE) v,
    ROW_NUMBER() OVER(PARTITION BY ISSUERID,FIELD_ID,START_DATE ORDER BY VERSION_START_DATE DESC) rn
    FROM '{CF}' WHERE (FIELD_ID LIKE '%MODERATE_CONTROVERSIES' OR FIELD_ID LIKE '%SEVERE_CONTROVERSIES') AND START_DATE>=20160101),
  agg AS (SELECT ISSUERID,issue,lvl,START_DATE,SUM(v) c FROM raw WHERE rn=1 AND v IS NOT NULL GROUP BY 1,2,3,4),
  w AS (SELECT ISSUERID,issue,START_DATE, MAX(CASE WHEN lvl='mod' THEN c END) modc, MAX(CASE WHEN lvl='sev' THEN c END) sevc FROM agg GROUP BY 1,2,3)
 SELECT ISSUERID,issue,START_DATE, COALESCE(modc,0)-LAG(COALESCE(modc,0)) OVER(PARTITION BY ISSUERID,issue ORDER BY START_DATE) dmod,
    COALESCE(sevc,0)-LAG(COALESCE(sevc,0)) OVER(PARTITION BY ISSUERID,issue ORDER BY START_DATE) dsev FROM w""")
ev=con.sql("SELECT dmod FROM mc WHERE dsev>0").df()
n=len(ev)
print("[RECLASSIFICATION] severe-onset issue-dates: %d | Δmod<0 (reclass) %.0f%% | Δmod=0 (new) %.0f%% | Δmod>0 %.0f%%"%(
  n,100*(ev.dmod<0).mean(),100*(ev.dmod==0).mean(),100*(ev.dmod>0).mean()))
# ---- LEAD: same-issue moderate onset -> next severe onset (months) ----
lead=con.sql(f"""WITH o AS (SELECT ISSUERID, REGEXP_REPLACE(FIELD_ID,'_(VERY_SEVERE|SEVERE|MODERATE)_CONTROVERSIES$','') issue,
    CASE WHEN FIELD_ID LIKE '%MODERATE_CONTROVERSIES' THEN 'mod' ELSE 'sev' END lvl, START_DATE, TRY_CAST(VALUE AS DOUBLE) v,
    ROW_NUMBER() OVER(PARTITION BY ISSUERID,FIELD_ID,START_DATE ORDER BY VERSION_START_DATE DESC) rn
    FROM '{CF}' WHERE (FIELD_ID LIKE '%MODERATE_CONTROVERSIES' OR FIELD_ID LIKE '%SEVERE_CONTROVERSIES') AND START_DATE>=20160101),
  a AS (SELECT ISSUERID,issue,lvl,START_DATE,SUM(v) c FROM o WHERE rn=1 AND v IS NOT NULL GROUP BY 1,2,3,4),
  d AS (SELECT ISSUERID,issue,lvl,START_DATE, c-LAG(c) OVER(PARTITION BY ISSUERID,issue,lvl ORDER BY START_DATE) dc FROM a),
  mo AS (SELECT ISSUERID,issue,(START_DATE//10000)*12+((START_DATE//100)%100) m FROM d WHERE lvl='mod' AND dc>0),
  so AS (SELECT ISSUERID,issue,(START_DATE//10000)*12+((START_DATE//100)%100) m FROM d WHERE lvl='sev' AND dc>0),
  pair AS (SELECT mo.ISSUERID,mo.issue,mo.m lm,MIN(so.m) sm FROM mo JOIN so ON so.ISSUERID=mo.ISSUERID AND so.issue=mo.issue AND so.m>mo.m GROUP BY 1,2,3)
  SELECT sm-lm gap FROM pair WHERE sm-lm BETWEEN 1 AND 36""").df()
print("[LEAD] same-issue moderate->severe: median %.0f mean %.1f months | %%>=6mo %.0f | n=%d"%(lead.gap.median(),lead.gap.mean(),100*(lead.gap>=6).mean(),len(lead)))
# ---- HAZARD (discrete-time cloglog): era effect on monthly downgrade hazard after a severe controversy ----
ias=con.sql(f"""WITH l AS (SELECT ISSUERID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,ROW_NUMBER() OVER(PARTITION BY ISSUERID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{SS}' WHERE FIELD_ID='INDUSTRY_ADJUSTED_SCORE' AND START_DATE>=20160101) SELECT ISSUERID,START_DATE,v,LAG(v) OVER(PARTITION BY ISSUERID ORDER BY START_DATE) pv FROM l WHERE rn=1 AND v IS NOT NULL""").df()
ias['dg']=np.where(ias.v-ias.pv<-1e-9,mi(ias.START_DATE),np.nan); dg=ias.dropna(subset=['dg'])[['ISSUERID','dg']].astype({'dg':int})
sev=con.sql(f"""WITH l AS (SELECT ISSUERID,FIELD_ID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,ROW_NUMBER() OVER(PARTITION BY ISSUERID,FIELD_ID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{CF}' WHERE FIELD_ID LIKE 'CONTR\\_%\\_N\\_SEVERE' ESCAPE '\\' AND START_DATE>=20160101), pd AS (SELECT ISSUERID,START_DATE,SUM(v) s FROM l WHERE rn=1 AND v IS NOT NULL GROUP BY 1,2) SELECT ISSUERID,START_DATE,s-LAG(s) OVER(PARTITION BY ISSUERID ORDER BY START_DATE) ds FROM pd""").df()
ev2=sev[sev.ds>1e-9].copy(); ev2['ev']=mi(ev2.START_DATE); yr=ev2.START_DATE//10000
ev2['late']=np.where((yr>=2021)&(yr<=2025),1,np.where((yr>=2018)&(yr<=2019),0,-1)); ev2=ev2[ev2.late>=0]
ev2['ev']=ev2['ev'].astype('int64'); dgs=dg.copy(); dgs['dg']=dgs['dg'].astype('int64'); dgs['ev']=dgs['dg']; ev2=ev2.sort_values('ev'); dgs=dgs.sort_values('dg')
ev2=pd.merge_asof(ev2,dgs[['ISSUERID','dg']],by='ISSUERID',left_on='ev',right_on='dg',direction='forward')
DE=mi(20260731); ev2['lag']=ev2.dg-ev2.ev; ev2['cens']=DE-ev2.ev
# person-month expansion to 24 months, discrete-time
rows=[]
for late,lag,cens in zip(ev2.late,ev2.lag.values,ev2.cens.values):
    horizon=int(min(24, lag if not np.isnan(lag) else cens))
    for m in range(1,horizon+1):
        fail=1 if (not np.isnan(lag) and lag==m) else 0
        rows.append((late,m,fail))
        if fail: break
pm=pd.DataFrame(rows,columns=['late','month','fail'])
import statsmodels.api as sm
X=sm.add_constant(pd.get_dummies(pm['month'],prefix='m',drop_first=True).astype(float)); X['late']=pm['late'].astype(float)
mdl=sm.GLM(pm['fail'],X,family=sm.families.Binomial(sm.families.links.CLogLog())).fit()
print("[HAZARD cloglog] late-era coef=%.3f (z=%.1f) -> hazard ratio=%.2f | events early=%d late=%d"%(
  mdl.params['late'],mdl.tvalues['late'],np.exp(mdl.params['late']),(ev2.late==0).sum(),(ev2.late==1).sum()))


**`pipeline/06_crossissue_downgradehorizon.py`** — Cross-issue spillover and the downgrade horizon.

In [ ]:
# ====================================================================================
# PIPELINE FILE: pipeline/06_crossissue_downgradehorizon.py
# Cross-issue spillover and the downgrade horizon.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

import duckdb, pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
con=duckdb.connect('a.db'); con.sql("PRAGMA memory_limit='2GB'"); con.sql("PRAGMA temp_directory='tmp'"); con.sql('PRAGMA threads=4'); con.sql('PRAGMA disable_progress_bar')
CF='jul/HIST_CONTROVERSIES_20260731.parquet'; SS='jul/HIST_SUMMARY_SCORES_20260731.parquet'
ISS=['ACCESS_TO_FIN','ACCESS_TO_HLTHCRE','BIODIV_LAND_USE','CARBON_EMISSIONS','CHEM_SAFETY','COMM_REL','CONTROV_SRC','E_WASTE','FINANCING_ENV_IMP','FIN_PROD_SAFETY','HLTH_SAFETY','HUMAN_CAPITAL_DEV','INS_CLIMATE_CHG_RISK','LABOR_MGMT','OPPS_CLN_TECH','OPPS_GREEN_BUILDING','OPPS_NUTRI_HLTH','OPPS_RENEW_ENERGY','PACK_MAT_WASTE','PRIVACY_DATA_SEC','PROD_CARB_FTPRNT','PROD_SFTY_QUALITY','RAW_MAT_SRC','RESPONSIBLE_INVEST','SUPPLY_CHAIN_LAB','TOXIC_EMISS_WSTE','WATER_STRESS']
import glob as _g
_ref=pd.read_parquet(_g.glob('mref/HIST_MASTER_ISSUER_REFERENCE*')[0],columns=['ISSUERID','PARENT_ULTIMATE_ISSUERID']).drop_duplicates('ISSUERID')
_PMAP=dict(zip(_ref.ISSUERID,_ref.PARENT_ULTIMATE_ISSUERID))
def _grp(s):
    s=pd.Series(list(s)); return s.map(_PMAP).fillna(s).values
def fe_lpm(d,y,xs):  # firm FE (fid) ; SE clustered by ULTIMATE PARENT (gid)
    dd=d.dropna(subset=[y]+xs); fid=pd.factorize(dd['ISSUERID'])[0].astype(np.int64); nf=fid.max()+1; cnt=np.bincount(fid,minlength=nf).astype(float)
    gid=pd.factorize(_grp(dd['ISSUERID']))[0].astype(np.int64); ng=gid.max()+1
    dm=lambda a:(a.astype(float)-(np.bincount(fid,weights=a.astype(float),minlength=nf)/cnt)[fid])
    Y=dm(dd[y].values); X=np.column_stack([dm(dd[x].values) for x in xs]); inv=np.linalg.pinv(X.T@X); b=inv@(X.T@Y); r=Y-X@b; sc=X*r[:,None]
    Sm=np.column_stack([np.bincount(gid,weights=sc[:,j],minlength=ng) for j in range(len(xs))]); V=inv@(Sm.T@Sm)@inv; se=np.sqrt(np.diag(V))
    return {xs[i]:(round(float(b[i]),5),round(float(b[i]/se[i]),2)) for i in range(len(xs))}, len(dd)
# onset tables
def onset(name,like):
    con.sql(f"""CREATE OR REPLACE TABLE {name} AS WITH l AS (SELECT ISSUERID,REGEXP_REPLACE(FIELD_ID,'_(VERY_SEVERE|SEVERE|MODERATE)_CONTROVERSIES$','') issue,FIELD_ID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,ROW_NUMBER() OVER(PARTITION BY ISSUERID,FIELD_ID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{CF}' WHERE {like} AND START_DATE>=20160101), pd AS (SELECT ISSUERID,issue,START_DATE,SUM(v) s FROM l WHERE rn=1 AND v IS NOT NULL GROUP BY 1,2,3), d AS (SELECT ISSUERID,issue,START_DATE, s-LAG(s) OVER(PARTITION BY ISSUERID,issue ORDER BY START_DATE) ds FROM pd) SELECT ISSUERID,issue,((START_DATE//10000)*12+((START_DATE//100)%100)) mi,1 ons FROM d WHERE ds>0 GROUP BY 1,2,3""")
onset('low_o',"FIELD_ID LIKE '%\\_MODERATE\\_CONTROVERSIES' ESCAPE '\\'"); onset('sev_o',"FIELD_ID LIKE '%\\_SEVERE\\_CONTROVERSIES' ESCAPE '\\'")
# monthly IAS as-of + band + firm-month low onset (for downgrade-horizon)
months=[y*100+m for y in range(2017,2027) for m in range(1,13) if not (y==2026 and m>7)]; ml='['+','.join(map(str,months))+']'
con.sql(f"""CREATE OR REPLACE TABLE fw AS
 WITH iasm AS (WITH l AS (SELECT ISSUERID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,ROW_NUMBER() OVER(PARTITION BY ISSUERID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{SS}' WHERE FIELD_ID='INDUSTRY_ADJUSTED_SCORE' AND START_DATE>=20160101), s AS (SELECT ISSUERID,START_DATE,v FROM l WHERE rn=1 AND v IS NOT NULL), grid AS (SELECT DISTINCT ISSUERID,ym,((ym//100)*10000+(ym%100)*100+31) thr,((ym//100)*12+(ym%100)) midx FROM s CROSS JOIN (SELECT UNNEST({ml}) ym)) SELECT g.ISSUERID,g.midx,x.v ias FROM grid g ASOF LEFT JOIN s x ON g.ISSUERID=x.ISSUERID AND g.thr>=x.START_DATE WHERE x.v IS NOT NULL),
   lom AS (SELECT ISSUERID, mi midx, SUM(ons) lo FROM low_o GROUP BY 1,2),
   j AS (SELECT i.ISSUERID,i.midx,i.ias,COALESCE(l.lo,0) lo, CAST(floor(i.ias/(10.0/7)) AS INT) band FROM iasm i LEFT JOIN lom l ON l.ISSUERID=i.ISSUERID AND l.midx=i.midx)
 SELECT *, LAG(band) OVER w bprev,
   SUM(CASE WHEN lo>0 THEN 1 ELSE 0 END) OVER (PARTITION BY ISSUERID ORDER BY midx RANGE BETWEEN 6 PRECEDING AND 1 PRECEDING) lowmo6,
   MIN(band) OVER (PARTITION BY ISSUERID ORDER BY midx RANGE BETWEEN 1 FOLLOWING AND 12 FOLLOWING) fmin12,
   MIN(band) OVER (PARTITION BY ISSUERID ORDER BY midx RANGE BETWEEN 1 FOLLOWING AND 24 FOLLOWING) fmin24,
   COUNT(*) OVER (PARTITION BY ISSUERID ORDER BY midx RANGE BETWEEN 6 PRECEDING AND 1 PRECEDING) nprev,
   COUNT(*) OVER (PARTITION BY ISSUERID ORDER BY midx RANGE BETWEEN 1 FOLLOWING AND 12 FOLLOWING) nf12
 FROM j WINDOW w AS (PARTITION BY ISSUERID ORDER BY midx)""")
d=con.sql("SELECT ISSUERID,ias,band,lowmo6,fmin12,fmin24,nprev,nf12 FROM fw WHERE nprev>=6").df()
d['sl']=(d.lowmo6>=2).astype(float)
for H,fc in [(12,'fmin12'),(24,'fmin24')]:
    dd=d[d.nf12>=1].copy() if H==12 else d.copy(); dd=dd[dd[fc].notna()]; dd['down']=(dd[fc]<dd['band']).astype(float)
    b=fe_lpm(dd,'down',['sl','ias'])[0]['sl']; print(f"[DOWNGRADE-HORIZON {H}m] sustained-low -> downgrade (firm FE+IAS): {b[0]:+.4f} (t={b[1]:+.1f})")
del d
# cross-issue: severe onset ~ cross-issue low(prior6m) + same-issue low, firm FE
con.sql("CREATE OR REPLACE TABLE fmths AS SELECT DISTINCT ISSUERID,midx FROM fw WHERE nprev>=6 AND ISSUERID IN (SELECT DISTINCT ISSUERID FROM sev_o)")
con.sql("CREATE OR REPLACE TABLE grid AS SELECT m.ISSUERID,m.midx,i.issue FROM fmths m CROSS JOIN (SELECT UNNEST("+str(ISS)+") issue) i")
con.sql("CREATE OR REPLACE TABLE lowfm AS SELECT ISSUERID,mi midx,SUM(ons) tot FROM low_o GROUP BY 1,2")
con.sql("""CREATE OR REPLACE TABLE reg AS WITH g AS (SELECT gr.ISSUERID,gr.midx,gr.issue,COALESCE(s.ons,0) sev_on,COALESCE(li.ons,0) lsame,COALESCE(lf.tot,0) ltot FROM grid gr LEFT JOIN sev_o s ON s.ISSUERID=gr.ISSUERID AND s.mi=gr.midx AND s.issue=gr.issue LEFT JOIN low_o li ON li.ISSUERID=gr.ISSUERID AND li.mi=gr.midx AND li.issue=gr.issue LEFT JOIN lowfm lf ON lf.ISSUERID=gr.ISSUERID AND lf.midx=gr.midx), w AS (SELECT *, SUM(lsame) OVER (PARTITION BY ISSUERID,issue ORDER BY midx RANGE BETWEEN 6 PRECEDING AND 1 PRECEDING) s6, SUM(ltot) OVER (PARTITION BY ISSUERID,issue ORDER BY midx RANGE BETWEEN 6 PRECEDING AND 1 PRECEDING) t6 FROM g) SELECT ISSUERID,sev_on,CASE WHEN s6>0 THEN 1 ELSE 0 END same6,CASE WHEN (t6-s6)>0 THEN 1 ELSE 0 END cross6 FROM w""")
ci=con.sql("SELECT ISSUERID,sev_on,same6,cross6 FROM reg").df().astype({'sev_on':'int8','same6':'int8','cross6':'int8'})
print("[CROSS-ISSUE] "+str(fe_lpm(ci,'sev_on',['cross6','same6'])[0]))


### 4.3 Letter-stage attenuation (distance to threshold) & figures

**`robustness/a3_threshold.py`** — Table 6 / Figure 2: distance-to-threshold LPM and letter-category bunching.

In [ ]:
# ====================================================================================
# PIPELINE FILE: robustness/a3_threshold.py
# Table 6 / Figure 2: distance-to-threshold LPM and letter-category bunching.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""Analysis 3: 12-month letter downgrade on distance-to-threshold bins x controversy severity (LPM, year FE,
ultimate-parent clusters); IAS distribution around the letter cut-offs (bunching check)."""
import duckdb, numpy as np, pandas as pd, statsmodels.api as sm, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
W = "/home/claude/w2"; SS = f"{W}/jul/HIST_SUMMARY_SCORES_20260731.parquet"; BW = 10 / 7
con = duckdb.connect(); con.sql("PRAGMA threads=6")


def asof(field):
    return con.sql(f"""WITH l AS (SELECT ISSUERID,START_DATE,VALUE s,TRY_CAST(VALUE AS DOUBLE) v,
        ROW_NUMBER() OVER(PARTITION BY ISSUERID,START_DATE ORDER BY VERSION_START_DATE DESC) rn
        FROM '{SS}' WHERE FIELD_ID='{field}' AND START_DATE>=20150101), r AS (SELECT * FROM l WHERE rn=1 AND s IS NOT NULL),
        y AS (SELECT UNNEST(range(2016,2027)) yr)
        SELECT r.ISSUERID, y.yr, arg_max(v,START_DATE) v, arg_max(s,START_DATE) s FROM r JOIN y ON START_DATE<=yr*10000+1231 GROUP BY 1,2""").df()


ias = asof("INDUSTRY_ADJUSTED_SCORE").rename(columns={"v": "ias"})[["ISSUERID", "yr", "ias"]]
let = asof("IVA_COMPANY_RATING").rename(columns={"s": "letter"})[["ISSUERID", "yr", "letter"]]
B = ias.merge(let, on=["ISSUERID", "yr"], how="inner").dropna()
rk = {"CCC": 1, "B": 2, "BB": 3, "BBB": 4, "A": 5, "AA": 6, "AAA": 7}
B["lr"] = B.letter.map(rk); B = B.dropna(subset=["lr"]).sort_values(["ISSUERID", "yr"])
g = B.groupby("ISSUERID"); B["lrn"] = g.lr.shift(-1); B["yn"] = g.yr.shift(-1)
B["down12"] = np.where(B.yn - B.yr == 1, (B.lrn < B.lr).astype(float), np.nan)
B["dist"] = B.ias - np.floor(B.ias / BW) * BW
# severity of new controversy onsets at t (issue-level onset from zero, July valid-time panel, as in the memo)
A = pd.read_parquet("/home/claude/evt/panelA_vt_july.parquet").rename(columns={"mod": "MOD"}).sort_values(["ISSUERID", "issue", "yr"])
gg = A.groupby(["ISSUERID", "issue"]); cons = A.yr - gg.yr.shift(1) == 1
A["o_v"] = (A.vsev >= 1) & ((gg.vsev.shift(1) < 1) | gg.vsev.shift(1).isna()) & cons
A["o_s"] = (A.sevvsev >= 1) & ((gg.sevvsev.shift(1) < 1) | gg.sevvsev.shift(1).isna()) & cons
A["o_m"] = (A.MOD >= 1) & ((gg.MOD.shift(1) < 1) | gg.MOD.shift(1).isna()) & cons
fo = A.groupby(["ISSUERID", "yr"]).agg(v=("o_v", "max"), s=("o_s", "max"), m=("o_m", "max")).reset_index()
fo["sevcat"] = np.where(fo.v, "very severe", np.where(fo.s, "severe", np.where(fo.m, "moderate", "none")))
D = B.merge(fo[["ISSUERID", "yr", "sevcat"]], on=["ISSUERID", "yr"], how="left"); D["sevcat"] = D.sevcat.fillna("none")
D = D[D.yr.isin([2018, 2019, 2021, 2022, 2023, 2024, 2025])].dropna(subset=["down12"])
nCCC = int((D.letter == "CCC").sum()); D = D[D.letter != "CCC"].copy()          # CCC cannot be downgraded
ref = pd.read_parquet(f"{W}/mref/HIST_MASTER_ISSUER_REFERENCE_20260831.parquet", columns=["ISSUERID", "PARENT_ULTIMATE_ISSUERID"]).drop_duplicates("ISSUERID")
D = D.merge(ref, on="ISSUERID", how="left"); D["grp"] = D.PARENT_ULTIMATE_ISSUERID.fillna(D.ISSUERID)
D["bin"] = pd.cut(D.dist, [-1e-9, 0.3, 0.6, 1.0, BW + 1e-9], labels=["<0.3", "0.3-0.6", "0.6-1.0", "1.0-1.43"])
print("N", len(D), "parents", D.grp.nunique(), "| dropped CCC firm-years:", nCCC)
cells = [(s, b) for s in ["none", "moderate", "severe", "very severe"] for b in ["<0.3", "0.3-0.6", "0.6-1.0", "1.0-1.43"]]
X = pd.DataFrame({f"{s}|{b}": ((D.sevcat == s) & (D["bin"] == b)).astype(float) for s, b in cells if (s, b) != ("none", "1.0-1.43")})
X = pd.concat([X, pd.get_dummies(D.yr.astype(int), prefix="y", drop_first=True).astype(float)], axis=1)
r = sm.OLS(D.down12.values, sm.add_constant(X).values).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(D.grp)[0]})
names = ["const"] + list(X.columns); b = pd.Series(r.params, index=names); t = pd.Series(r.tvalues, index=names)
rows = []
for s, bn in cells:
    k = f"{s}|{bn}"; sub = D[(D.sevcat == s) & (D["bin"] == bn)]
    rows.append(dict(severity=s, bin=bn, N=len(sub), raw_pct=100 * sub.down12.mean() if len(sub) else np.nan,
                     coef_pp=100 * b.get(k, 0.0), t=t.get(k, np.nan)))
T = pd.DataFrame(rows)
# severe minus none, within bin; and near-minus-far difference in that gap
V = r.cov_params(); ix = {n: i for i, n in enumerate(names)}


def lin(w):
    v = np.zeros(len(names))
    for k, c in w.items():
        if k in ix: v[ix[k]] += c
    est = v @ r.params; se = np.sqrt(v @ V @ v); return 100 * est, est / se


tests = {}
for s in ["moderate", "severe", "very severe"]:
    for bn in ["<0.3", "1.0-1.43"]:
        w = {f"{s}|{bn}": 1}; w.update({f"none|{bn}": -1} if bn != "1.0-1.43" else {})
        tests[f"{s} minus none, {bn}"] = lin(w)
    tests[f"{s}: (near gap) minus (far gap)"] = lin({f"{s}|<0.3": 1, "none|<0.3": -1, f"{s}|1.0-1.43": -1})
T.to_csv(f"{W}/out/A3_threshold_lpm.csv", index=False)
pd.DataFrame([(k, v[0], v[1]) for k, v in tests.items()], columns=["test", "pp", "t"]).to_csv(f"{W}/out/A3_tests.csv", index=False)
print(T.round(3).to_string()); print({k: (round(v[0], 2), round(v[1], 2)) for k, v in tests.items()})
print("year FE; N =", int(r.nobs), "; clusters =", D.grp.nunique())

# -------------------------------------------------------------- bunching around cut-offs
cut = [k * BW for k in range(1, 7)]
Y = B[B.yr.isin([2018, 2019, 2021, 2022, 2023, 2024, 2025])]
dec = (np.round(Y.ias * 10) - Y.ias * 10).abs() < 1e-6
print("share of year-end IAS values on the one-decimal grid:", round(dec.mean(), 3))
rec = con.sql(f"""WITH l AS (SELECT ISSUERID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,ROW_NUMBER() OVER(PARTITION BY ISSUERID,START_DATE ORDER BY VERSION_START_DATE DESC) rn
     FROM '{SS}' WHERE FIELD_ID='INDUSTRY_ADJUSTED_SCORE' AND START_DATE>=20180101) SELECT v FROM l WHERE rn=1 AND v IS NOT NULL""").df().v
brow = []
fig, ax = plt.subplots(2, 3, figsize=(11, 6), dpi=150, sharey=False)
for i, c in enumerate(cut):
    v = np.round(Y.ias.values, 1)
    grid = np.round(np.arange(np.floor(c * 10) / 10 - 0.2, np.floor(c * 10) / 10 + 0.31, 0.1), 1)
    cnt = [(v == gpt).sum() for gpt in grid]
    for gpt, n in zip(grid, cnt): brow.append(dict(cutoff=round(c, 3), ias=gpt, side="above" if gpt > c else "below", n=n))
    a = ax.flat[i]
    a.bar(grid, cnt, width=0.08, color=["#9aa5b1" if gpt < c else "#2f5d8a" for gpt in grid])
    a.axvline(c, color="black", lw=1, ls="--"); a.set_title(f"Cut-off {c:.3f} ({['CCC/B','B/BB','BB/BBB','BBB/A','A/AA','AA/AAA'][i]})", fontsize=9)
    a.set_xticks(grid); a.tick_params(labelsize=7)
fig.suptitle("Year-end IAS within about ±0.3 of each letter cut-off (rounded to 0.1), firm-years 2018–2025 excl. 2020", fontsize=10)
fig.tight_layout(); fig.savefig(f"{W}/out/A3_bunching.png"); pd.DataFrame(brow).to_csv(f"{W}/out/A3_bunching_counts.csv", index=False)
print(pd.DataFrame(brow).pivot(index="ias", columns="cutoff", values="n").fillna("").to_string())


**`pipeline/08_table6_twoevent_and_industryFE.py`** — Table 6: two-event specification and industry fixed effects.

In [ ]:
# ====================================================================================
# PIPELINE FILE: pipeline/08_table6_twoevent_and_industryFE.py
# Table 6: two-event specification and industry fixed effects.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

import duckdb, pandas as pd, numpy as np, statsmodels.api as sm, warnings
warnings.filterwarnings('ignore')
con=duckdb.connect(); con.sql('PRAGMA disable_progress_bar'); con.sql("PRAGMA memory_limit='3GB'"); con.sql("PRAGMA temp_directory='tmp'")
R='/mnt/user-data/uploads/monthly_total_returns__1_.csv'
IR='mref/HIST_MASTER_INSTRUMENT_REFERENCE_none_20260831.parquet'
ISR='mref/HIST_MASTER_ISSUER_REFERENCE_20260831.parquet'
SS='jul/HIST_SUMMARY_SCORES_20260731.parquet'; CF='jul/HIST_CONTROVERSIES_20260731.parquet'
# ---------- (A) INDUSTRY FE for §6.7 ----------
sec=con.sql(f"SELECT DISTINCT ISSUERID, ANY_VALUE(GICS_SECTOR) sector FROM '{ISR}' WHERE GICS_SECTOR IS NOT NULL GROUP BY ISSUERID").df()
D=pd.read_parquet('pricing_panel.parquet').merge(sec,on='ISSUERID',how='left')
def reg(dat,y,xs,fes):
    d=dat.dropna(subset=[y]+xs+fes).copy()
    parts=[d[xs]]
    for fe in fes: parts.append(pd.get_dummies(d[fe].astype(str),prefix=fe[:3],drop_first=True).astype(float))
    X=pd.concat(parts,axis=1)
    r=sm.OLS(d[y],sm.add_constant(X)).fit(cov_type='cluster',cov_kwds={'groups':pd.factorize(_grp(d['ISSUERID']))[0]})
    return {x:(round(r.params[x],4),round(r.tvalues[x],2)) for x in xs}, int(r.nobs)
print("=== §6.7 pricing WITH industry (GICS sector) FE added ===")
print(" fwd12 ~ own + nonown  [year+country+sector FE]:", reg(D,'fwd12w',['own','nonown'],['yr','ctry','sector']))
d=D.dropna(subset=['fwd12w','own','nonown','sector']).copy()
X=pd.concat([d[['own','nonown']],pd.get_dummies(d.yr.astype(str),prefix='y',drop_first=True).astype(float),pd.get_dummies(d.ctry,prefix='c',drop_first=True).astype(float),pd.get_dummies(d.sector,prefix='s',drop_first=True).astype(float)],axis=1)
r=sm.OLS(d.fwd12w,sm.add_constant(X)).fit(cov_type='cluster',cov_kwds={'groups':pd.factorize(_grp(d.ISSUERID))[0]})
w=r.f_test("own=0, nonown=0"); print("   joint Wald own=nonown=0: F=%.2f p=%.3f | N=%d"%(float(w.fvalue),float(w.pvalue),int(r.nobs)))

# ---------- (B) TWO-EVENT STUDY: returns around controversy onsets vs downgrades ----------
mi=lambda d:(d//10000)*12+((d//100)%100)
# monthly returns -> market(country x month)-adjusted
con.sql(f"""CREATE OR REPLACE TABLE ret AS SELECT isin, (EXTRACT(year FROM CAST(date AS DATE))*12+EXTRACT(month FROM CAST(date AS DATE))) midx,
  tr_return_pct/100.0 r, ISSUER_CNTRY_DOMICILE ctry FROM read_csv_auto('{R}') WHERE tr_return_pct IS NOT NULL""")
con.sql("CREATE OR REPLACE TABLE cm AS SELECT ctry, midx, AVG(r) mr FROM ret GROUP BY ctry,midx")
con.sql("CREATE OR REPLACE TABLE reta AS SELECT t.isin, t.midx, (t.r - c.mr) ra FROM ret t JOIN cm c ON c.ctry=t.ctry AND c.midx=t.midx")
# map isin->ISSUERID (primary = most return obs)
con.sql("CREATE OR REPLACE TABLE obs AS SELECT isin, COUNT(*) n FROM ret GROUP BY isin")
con.sql(f"""CREATE OR REPLACE TABLE map AS WITH p AS (SELECT DISTINCT ISSUERID,ISIN FROM '{IR}' WHERE ISIN IS NOT NULL AND ISSUERID IS NOT NULL),
  j AS (SELECT p.ISSUERID,p.ISIN,o.n,ROW_NUMBER() OVER(PARTITION BY p.ISSUERID ORDER BY o.n DESC) rk FROM p JOIN obs o ON o.isin=p.ISIN)
  SELECT ISSUERID,ISIN FROM j WHERE rk=1""")
con.sql("CREATE OR REPLACE TABLE fmret AS SELECT m.ISSUERID, r.midx, r.ra FROM reta r JOIN map m ON m.ISIN=r.isin")
# forward 6-month cumulative abnormal return CAR[+1,+6]
con.sql("""CREATE OR REPLACE TABLE car AS SELECT ISSUERID, midx,
   SUM(ra) OVER (PARTITION BY ISSUERID ORDER BY midx RANGE BETWEEN 1 FOLLOWING AND 6 FOLLOWING) car6,
   COUNT(*) OVER (PARTITION BY ISSUERID ORDER BY midx RANGE BETWEEN 1 FOLLOWING AND 6 FOLLOWING) nf FROM fmret""")
# events: downgrade (letter band drop) and severe onset
ias=con.sql(f"""WITH l AS (SELECT ISSUERID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,ROW_NUMBER() OVER(PARTITION BY ISSUERID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{SS}' WHERE FIELD_ID='INDUSTRY_ADJUSTED_SCORE' AND START_DATE>=20160101) SELECT ISSUERID, (START_DATE//10000*12+(START_DATE//100%100)) midx, CAST(floor(v/(10.0/7)) AS INT) band, CAST(floor(LAG(v) OVER(PARTITION BY ISSUERID ORDER BY START_DATE)/(10.0/7)) AS INT) pband FROM l WHERE rn=1 AND v IS NOT NULL""").df()
dg=ias[ias.band<ias.pband][['ISSUERID','midx']].drop_duplicates(); dg['downgrade']=1
sev=con.sql(f"""WITH l AS (SELECT ISSUERID,FIELD_ID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,ROW_NUMBER() OVER(PARTITION BY ISSUERID,FIELD_ID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{CF}' WHERE FIELD_ID LIKE 'CONTR\\_%\\_N\\_SEVERE' ESCAPE '\\' AND START_DATE>=20160101), p AS (SELECT ISSUERID,START_DATE,SUM(v) s FROM l WHERE rn=1 AND v IS NOT NULL GROUP BY 1,2), d AS (SELECT ISSUERID,(START_DATE//10000*12+(START_DATE//100%100)) midx, s-LAG(s) OVER(PARTITION BY ISSUERID ORDER BY START_DATE) ds FROM p) SELECT DISTINCT ISSUERID,midx FROM d WHERE ds>0""").df(); sev['sevonset']=1
C=con.sql("SELECT ISSUERID,midx,car6,nf FROM car WHERE nf>=6").df()
C=C.merge(dg,on=['ISSUERID','midx'],how='left').merge(sev,on=['ISSUERID','midx'],how='left').fillna({'downgrade':0,'sevonset':0})
# prior-12m severe controversy flag (for downgrades: was it controversy-linked?)
sv=sev[['ISSUERID','midx']].copy()
C=C.sort_values(['ISSUERID','midx'])
svset=set(map(tuple, sv.values))
def prior12(r):
    return int(any((r.ISSUERID,r.midx-k) in svset for k in range(0,13)))
# vectorized prior12 via merge_asof-like: mark months within 12 after a sev onset
sv2=sv.rename(columns={'midx':'sm'}); mrg=C[['ISSUERID','midx']].merge(sv2,on='ISSUERID'); mrg=mrg[(mrg.midx-mrg.sm).between(0,12)]
linked=mrg[['ISSUERID','midx']].drop_duplicates(); linked['contr12']=1
C=C.merge(linked,on=['ISSUERID','midx'],how='left').fillna({'contr12':0})
C=C[(C.midx>=2016*12) & (C.midx<=2025*12)]   # window with returns coverage
print("\n=== TWO-EVENT STUDY: 6-month forward CAR (market-adjusted) around events ===")
print(" mean CAR6  no event            : %+.4f (N=%d)"%(C[(C.downgrade==0)&(C.sevonset==0)].car6.mean(), ((C.downgrade==0)&(C.sevonset==0)).sum()))
print(" mean CAR6  severe controversy   : %+.4f (N=%d)"%(C[C.sevonset==1].car6.mean(), (C.sevonset==1).sum()))
print(" mean CAR6  downgrade (all)      : %+.4f (N=%d)"%(C[C.downgrade==1].car6.mean(), (C.downgrade==1).sum()))
print(" mean CAR6  downgrade, controversy-linked (prior12) : %+.4f (N=%d)"%(C[(C.downgrade==1)&(C.contr12==1)].car6.mean(),((C.downgrade==1)&(C.contr12==1)).sum()))
print(" mean CAR6  downgrade, NOT controversy-linked        : %+.4f (N=%d)"%(C[(C.downgrade==1)&(C.contr12==0)].car6.mean(),((C.downgrade==1)&(C.contr12==0)).sum()))
# regression: incremental reaction, firm FE, cluster firm
d=C.copy(); d['dg_linked']=d.downgrade*d.contr12; d['dg_unlinked']=d.downgrade*(1-d.contr12)
fid=pd.factorize(_grp(d.ISSUERID))[0]  # cluster by ultimate parent
def dm(a): m=pd.Series(a).groupby(fid).transform('mean'); return a-m.values
Y=dm(d.car6.values); X=np.column_stack([dm(d[c].values.astype(float)) for c in ['sevonset','dg_linked','dg_unlinked']])
inv=np.linalg.pinv(X.T@X); b=inv@(X.T@Y); r=Y-X@b; sc=X*r[:,None]
import numpy as np
import glob as _g
_ref=pd.read_parquet(_g.glob('mref/HIST_MASTER_ISSUER_REFERENCE*')[0],columns=['ISSUERID','PARENT_ULTIMATE_ISSUERID']).drop_duplicates('ISSUERID')
_PMAP=dict(zip(_ref.ISSUERID,_ref.PARENT_ULTIMATE_ISSUERID))
def _grp(s):
    s=pd.Series(list(s)); return s.map(_PMAP).fillna(s).values
S=np.zeros((fid.max()+1,3))
for j in range(3): S[:,j]=np.bincount(fid,weights=sc[:,j],minlength=fid.max()+1)
V=inv@(S.T@S)@inv; se=np.sqrt(np.diag(V))
for i,nm in enumerate(['sevonset','downgrade(controversy-linked)','downgrade(not linked)']):
    print("   %s: %+.4f (t %+.1f)"%(nm,b[i],b[i]/se[i]))


**`pipeline/00_figure1_architecture.py`** — Figure 1: the MSCI rating-architecture schematic.

In [ ]:
# ====================================================================================
# PIPELINE FILE: pipeline/00_figure1_architecture.py
# Figure 1: the MSCI rating-architecture schematic.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig,ax=plt.subplots(figsize=(10,3.5),dpi=150); ax.set_xlim(0,100); ax.set_ylim(0,40); ax.axis('off')
def box(x,y,w,h,txt,fc,ec='#333333',fs=9,bold=False):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.4,rounding_size=1.2",fc=fc,ec=ec,lw=1.2))
    ax.text(x+w/2,y+h/2,txt,ha='center',va='center',fontsize=fs,weight='bold' if bold else 'normal',wrap=True)
def arrow(x1,y1,x2,y2,txt=None):
    ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle='-|>',mutation_scale=14,lw=1.3,color='#444444'))
    if txt: ax.text((x1+x2)/2,(y1+y2)/2+2.2,txt,ha='center',va='bottom',fontsize=7.5,style='italic',color='#555555')
BL=(0.86,0.90,0.97); GR=(0.90,0.94,0.90); YE=(0.98,0.95,0.82); OR=(0.98,0.90,0.82)
# controversy input (top, feeding into key-issue scores)
box(1,27,20,10,"ESG controversies\n(Moderate / Severe /\nVery severe)",OR,fs=8.5)
# stage boxes along the middle
y=14; h=11
box(1,y,20,h,"Key-issue\nscores",GR)
box(27,y,19,h,"Weighted-average\nkey-issue score\n(WAKIS)",GR)
box(52,y,19,h,"Industry-Adjusted\nScore (IAS, 0\u201310)",BL)
box(78,y,21,h,"Letter rating\nCCC \u2026 AAA\n(6 thresholds)",BL)
arrow(21,19.5,27,19.5,"materiality\nweights")
arrow(46,19.5,52,19.5,"industry\nbenchmarking")
arrow(71,19.5,78,19.5,"6 fixed\nthresholds")
arrow(11,27,11,25.2)  # controversy -> key-issue scores
# annotations under boxes (what we observe/measure)
ax.text(11,12.4,"observed input",ha='center',fontsize=7,color='#1f6f3f',style='italic')
ax.text(36.5,12.4,"published (own-score input)",ha='center',fontsize=7,color='#1f6f3f',style='italic')
ax.text(61.5,12.4,"published outcome (\u0394IAS)",ha='center',fontsize=7,color='#234b8a',style='italic')
ax.text(88.5,12.4,"coarse categories",ha='center',fontsize=7,color='#234b8a',style='italic')
# brace note
ax.text(50,4.5,"Attenuation is measured at each stage: severity-graded response (\u00A76.2), own-score vs non-own-score share at the benchmarking stage (\u00A76.3), and coarseness at the thresholds (\u00A76.2).",
        ha='center',va='center',fontsize=7.6,color='#333333')
fig.tight_layout(); fig.savefig('fig_architecture.png',bbox_inches='tight',dpi=150); print("architecture figure saved")


**`pipeline/04_figure2_monthly.py`** — Figure 2: monthly band panel / bunching figure.

In [ ]:
# ====================================================================================
# PIPELINE FILE: pipeline/04_figure2_monthly.py
# Figure 2: monthly band panel / bunching figure.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

import duckdb, pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore')
con=duckdb.connect('a.db'); con.sql("PRAGMA memory_limit='2GB'"); con.sql("PRAGMA temp_directory='tmp'"); con.sql('PRAGMA threads=4'); con.sql('PRAGMA disable_progress_bar')
SS='jul/HIST_SUMMARY_SCORES_20260731.parquet'; CF='jul/HIST_CONTROVERSIES_20260731.parquet'
mi=lambda d:(d//10000)*12+((d//100)%100); DE=mi(20260731); H=24
# dated IAS changes (latest version per START_DATE)
ias=con.sql(f"""WITH l AS (SELECT ISSUERID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,ROW_NUMBER() OVER(PARTITION BY ISSUERID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{SS}' WHERE FIELD_ID='INDUSTRY_ADJUSTED_SCORE' AND START_DATE>=20160101) SELECT ISSUERID,START_DATE,v,LAG(v) OVER(PARTITION BY ISSUERID ORDER BY START_DATE) pv FROM l WHERE rn=1 AND v IS NOT NULL""").df()
ias['dias']=ias.v-ias.pv
# dated severe (+very severe) onsets
sev=con.sql(f"""WITH l AS (SELECT ISSUERID,FIELD_ID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,ROW_NUMBER() OVER(PARTITION BY ISSUERID,FIELD_ID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{CF}' WHERE FIELD_ID LIKE 'CONTR\\_%\\_N\\_SEVERE' ESCAPE '\\' AND START_DATE>=20160101), pd AS (SELECT ISSUERID,START_DATE,SUM(v) s FROM l WHERE rn=1 AND v IS NOT NULL GROUP BY 1,2) SELECT ISSUERID,START_DATE,s,LAG(s) OVER(PARTITION BY ISSUERID ORDER BY START_DATE) ps FROM pd""").df()
sev['dsev']=sev.s-sev.ps
ev=sev[sev.dsev>1e-9][['ISSUERID','START_DATE']].copy(); ev['ev']=mi(ev.START_DATE); yr=ev.START_DATE//10000
ev['era']=np.where((yr>=2018)&(yr<=2019),'early',np.where((yr>=2021)&(yr<=2025),'late','other')); ev=ev[ev.era!='other']
dg=ias[ias.dias<-1e-9][['ISSUERID','START_DATE']].copy(); dg['dg']=mi(dg.START_DATE)
ev=ev.sort_values('ev'); dg=dg.sort_values('dg')
ev=pd.merge_asof(ev,dg[['ISSUERID','dg']],by='ISSUERID',left_on='ev',right_on='dg',direction='forward')
ev['lag']=ev.dg-ev.ev; ev['cens']=DE-ev.ev
def monthly(sub):
    recs=[]
    for m in range(1,H+1):
        atrisk=((sub.lag.isna()&(sub.cens>=m))|(sub.lag>=m)).sum(); fail=(sub.lag==m).sum()
        recs.append((m,fail/atrisk if atrisk>0 else np.nan))
    d=pd.DataFrame(recs,columns=['month','hazard']); d['cum']=1-(1-d.hazard).cumprod(); return d
mE=monthly(ev[ev.era=='early']); mL=monthly(ev[ev.era=='late'])
tab=pd.DataFrame({'month':mE.month,'hazard_early':mE.hazard.round(4),'cum_early':mE.cum.round(4),'hazard_late':mL.hazard.round(4),'cum_late':mL.cum.round(4)})
tab.to_csv('tab_monthly.csv',index=False)
print('events: early',len(ev[ev.era=="early"]),'late',len(ev[ev.era=="late"]))
print('cum incidence @24mo: early %.2f  late %.2f'%(mE.cum.iloc[-1],mL.cum.iloc[-1]))
# figure
fig,ax=plt.subplots(1,2,figsize=(10,3.9),dpi=150); ce=(0.20,0.30,0.70); cl=(0.80,0.30,0.20)
ax[0].plot(tab.month,100*tab.cum_early,'-o',color=ce,lw=1.9,ms=3.5,label='2018–2019')
ax[0].plot(tab.month,100*tab.cum_late,'--s',color=cl,lw=1.9,ms=3.5,label='2021–2025')
ax[0].set_xlabel('Months since severe controversy'); ax[0].set_ylabel('Cumulative downgrade incidence (%)')
ax[0].set_title('(a) Cumulative probability of a rating downgrade',fontsize=10); ax[0].set_xlim(1,24); ax[0].set_xticks(range(0,25,3)); ax[0].grid(alpha=.3); ax[0].legend(fontsize=8,loc='lower right')
ax[1].plot(tab.month,100*tab.hazard_early,'-o',color=ce,lw=1.6,ms=3.5,label='2018–2019')
ax[1].plot(tab.month,100*tab.hazard_late,'--s',color=cl,lw=1.6,ms=3.5,label='2021–2025')
ax[1].set_xlabel('Months since severe controversy'); ax[1].set_ylabel('Monthly downgrade hazard (%)')
ax[1].set_title('(b) Per-month hazard of a downgrade',fontsize=10); ax[1].set_xlim(1,24); ax[1].set_xticks(range(0,25,3)); ax[1].grid(alpha=.3); ax[1].legend(fontsize=8,loc='upper right')
fig.tight_layout(); fig.savefig('fig_monthly.png',bbox_inches='tight',dpi=150)
print('figure saved')


### 4.4 Pricing of the rating signal

**`pipeline/07_table5_pricing.py`** — Subsidiary pricing check: do the own- vs non-own-score components of ΔIAS predict forward returns?

In [ ]:
# ====================================================================================
# PIPELINE FILE: pipeline/07_table5_pricing.py
# Subsidiary pricing check: do the own- vs non-own-score components of ΔIAS predict forward returns?
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

import duckdb, pandas as pd, numpy as np, statsmodels.api as sm, warnings
import glob as _g
_ref=pd.read_parquet(_g.glob('mref/HIST_MASTER_ISSUER_REFERENCE*')[0],columns=['ISSUERID','PARENT_ULTIMATE_ISSUERID']).drop_duplicates('ISSUERID')
_PMAP=dict(zip(_ref.ISSUERID,_ref.PARENT_ULTIMATE_ISSUERID))
def _grp(s):
    s=pd.Series(list(s)); return s.map(_PMAP).fillna(s).values
warnings.filterwarnings('ignore')
con=duckdb.connect(); con.sql('PRAGMA disable_progress_bar'); con.sql("PRAGMA memory_limit='3GB'"); con.sql("PRAGMA temp_directory='tmp'")
R='/mnt/user-data/uploads/monthly_total_returns__1_.csv'
IR='mref/HIST_MASTER_INSTRUMENT_REFERENCE_none_20260831.parquet'
# 1) ISSUERID -> ISIN, restricted to ISINs that appear in the returns file; if several, pick the ISIN with most return obs
con.sql(f"""CREATE OR REPLACE TABLE ret AS SELECT isin, CAST(date AS DATE) d, tri, ISSUER_CNTRY_DOMICILE ctry,
   (EXTRACT(year FROM CAST(date AS DATE))) yr, EXTRACT(month FROM CAST(date AS DATE)) mo FROM read_csv_auto('{R}') WHERE tri IS NOT NULL""")
con.sql("CREATE OR REPLACE TABLE isin_obs AS SELECT isin, COUNT(*) n, ANY_VALUE(ctry) ctry FROM ret GROUP BY isin")
con.sql(f"""CREATE OR REPLACE TABLE map AS
  WITH pairs AS (SELECT DISTINCT ISSUERID, ISIN FROM '{IR}' WHERE ISIN IS NOT NULL AND ISSUERID IS NOT NULL),
       j AS (SELECT p.ISSUERID, p.ISIN, o.n, o.ctry, ROW_NUMBER() OVER(PARTITION BY p.ISSUERID ORDER BY o.n DESC) rk FROM pairs p JOIN isin_obs o ON o.isin=p.ISIN)
  SELECT ISSUERID, ISIN, ctry FROM j WHERE rk=1""")
nmap=con.sql("SELECT COUNT(*) n FROM map").df().n[0]; print("issuers mapped to a returns ISIN:", nmap)
# 2) year-end (December) tri per isin, then forward 12m return  fwd[t] = tri(Dec t+1)/tri(Dec t) - 1
con.sql("CREATE OR REPLACE TABLE dec AS SELECT isin, yr, ARG_MAX(tri, d) tri_dec FROM ret WHERE mo=12 GROUP BY isin, yr")
con.sql("""CREATE OR REPLACE TABLE fwd AS SELECT a.isin, a.yr, (b.tri_dec/a.tri_dec - 1.0) fwd12
   FROM dec a JOIN dec b ON b.isin=a.isin AND b.yr=a.yr+1 WHERE a.tri_dec>0""")
# 3) firm-year rating-change panel
P=pd.read_parquet('panel_fy.parquet').sort_values(['ISSUERID','yr']); g=P.groupby('ISSUERID'); P['py']=g.yr.shift(1)
P['dias']=np.where(P.yr-P.py==1,P.ias-g.ias.shift(1),np.nan); P['dwas']=np.where(P.yr-P.py==1,P.was-g.was.shift(1),np.nan)
P=P.dropna(subset=['dias','dwas'])[['ISSUERID','yr','dias','dwas']]
con.register('P',P)
# 4) merge: firm-year rating change + its forward return (via ISIN map)
D=con.sql("""SELECT P.ISSUERID, P.yr, P.dias, P.dwas, f.fwd12, m.ctry
   FROM P JOIN map m ON m.ISSUERID=P.ISSUERID JOIN fwd f ON f.isin=m.ISIN AND f.yr=P.yr""").df()
D=D[D.yr.between(2018,2024)].copy()
# winsorize forward returns at 1/99
lo,hi=D.fwd12.quantile([.01,.99]); D['fwd12w']=D.fwd12.clip(lo,hi)
# own vs non-own components: b_hat from dias~dwas on this sample; nonown = residual orthogonal to dwas
bhat=np.polyfit(D.dwas,D.dias,1)[0]
D['own']=bhat*D.dwas                 # benchmarking-amplified own-score component
D['nonown']=D.dias-bhat*D.dwas       # non-own-score (benchmarking/committee/other) component, orthogonal to dwas
print("sample: firm-years with returns =",len(D),"| issuers =",D.ISSUERID.nunique(),"| years",sorted(D.yr.unique()),"| bhat=%.2f"%bhat)
print("mean fwd12 (winsor) = %.3f | sd = %.3f"%(D.fwd12w.mean(),D.fwd12w.std()))
def reg(y,xs):
    d=D.dropna(subset=[y]+xs).copy()
    d['yr']=d['yr'].astype(int)
    X=pd.concat([d[xs], pd.get_dummies(d['yr'],prefix='y',drop_first=True).astype(float), pd.get_dummies(d['ctry'],prefix='c',drop_first=True).astype(float)],axis=1)
    r=sm.OLS(d[y],sm.add_constant(X)).fit(cov_type='cluster',cov_kwds={'groups':pd.factorize(_grp(d['ISSUERID']))[0]})
    return {x:(round(r.params[x],4),round(r.tvalues[x],2)) for x in xs}, int(r.nobs)
print("\n=== PRICING VALIDATION: 12-month-forward return on components of the annual rating change (year+country FE, firm-clustered) ===")
print(" fwd12 ~ own(benchmarking-amplified WAKIS) + nonown(non-own-score):", reg('fwd12w',['own','nonown']))
print(" fwd12 ~ dias (total rating change):", reg('fwd12w',['dias']))
print(" fwd12 ~ dwas (own WAKIS change) + nonown:", reg('fwd12w',['dwas','nonown']))
# joint test that neither component is priced (Wald)
d=D.dropna(subset=['fwd12w','own','nonown']).copy(); d['yr']=d['yr'].astype(int)
X=pd.concat([d[['own','nonown']],pd.get_dummies(d['yr'],prefix='y',drop_first=True).astype(float),pd.get_dummies(d['ctry'],prefix='c',drop_first=True).astype(float)],axis=1)
r=sm.OLS(d['fwd12w'],sm.add_constant(X)).fit(cov_type='cluster',cov_kwds={'groups':pd.factorize(_grp(d['ISSUERID']))[0]})
w=r.f_test("own=0, nonown=0")
print(" joint Wald H0: own=nonown=0  -> F=%.2f  p=%.3f"%(float(w.fvalue),float(w.pvalue)))
D.to_parquet('pricing_panel.parquet')


**`pillar_forward_backward/score_predicts_returns_fundamentals.py`** — Forward-looking test: does the pillar score predict future severe controversies / returns? (P2/P4 panels).

In [ ]:
# ====================================================================================
# PIPELINE FILE: pillar_forward_backward/score_predicts_returns_fundamentals.py
# Forward-looking test: does the pillar score predict future severe controversies / returns? (P2/P4 panels).
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""
score_predicts_returns_fundamentals.py — does the ESG rating (IAS) predict the future?

  (A) FORWARD RETURNS   : year-end IAS[t] -> next-12-month market-adjusted return.
  (B) FORWARD FUNDAMENTALS: year-end IAS[t] -> next-year operating ROA (oibdp/at),
                            controlling for current ROA.

Both at the ultimate-parent level (returns and Compustat are keyed by parent_id, which is
the same IID identifier as the panel's PARENT_ULTIMATE_ISSUERID). Year fixed effects;
standard errors clustered by parent. Scores aggregated to the parent-year by mean.
Estimated on the paper's canonical sample window: base year t in 2018-2025 excluding 2020
(so forward outcomes fall in 2019-2025), matching Tables 2/5 and the rest of the draft.

Inputs (licensed, not shipped):
  w2/panel_fy_validtime.parquet            issuer-year IAS/WAKIS + PARENT_ULTIMATE_ISSUERID
  p2/kit/data/returns.parquet              parent-month market-adjusted return (ra), midx=year*12+month
  w2/wrds_extra/compustat_annual.parquet   parent-year fundamentals (oibdp, at)

Run:  python score_predicts_returns_fundamentals.py   (writes P4_score_predicts.csv)
"""
import os, numpy as np, pandas as pd, statsmodels.api as sm

W = os.environ.get("W", "w2"); KIT = os.environ.get("KIT", "p2/kit/data")
z = lambda s: (s - s.mean()) / s.std()
KEEP = [2018, 2019, 2021, 2022, 2023, 2024, 2025]   # paper window: 2018-2025 excl. 2020 (base year t)

def creg(d, y, xs, cl="pid"):
    d = d.dropna(subset=[y] + xs + [cl]).copy(); d["yr"] = d["yr"].astype(int); yrs = sorted(d["yr"].unique())
    X = np.column_stack([np.ones(len(d))] + [(d["yr"] == yy).astype(float) for yy in yrs[1:]] + [d[x].values for x in xs])
    r = sm.OLS(d[y].values, X).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d[cl])[0]})
    p = len(xs); return [(xs[i], round(r.params[-p + i], 4), round(r.tvalues[-p + i], 1)) for i in range(p)], len(d), d[cl].nunique()

# parent-year ESG rating (IAS), year-end, from the valid-time panel
P = pd.read_parquet(f"{W}/panel_fy_validtime.parquet")[["yr", "ias", "PARENT_ULTIMATE_ISSUERID"]].rename(columns={"PARENT_ULTIMATE_ISSUERID": "pid"})
pg = P.groupby(["pid", "yr"]).agg(ias=("ias", "mean")).reset_index()

rows = []
# ---------- (A) forward returns ----------
ret = pd.read_parquet(f"{KIT}/returns.parquet"); ret["pid"] = ret.parent_id.astype(str)
lo, hi = ret.ra.quantile([.005, .995]); ret["raw_"] = ret.ra.clip(lo, hi)   # winsorise monthly return at 0.5/99.5, matching the Appendix H return tests
ret = ret.sort_values(["pid", "midx"]); g = ret.groupby("pid")
ret["f12"] = sum(g["raw_"].shift(-k) for k in range(1, 13)); ret["f12"] = ret["f12"].where(g["midx"].shift(-12) == ret["midx"] + 12)
ret["b12"] = sum(g["raw_"].shift(k) for k in range(0, 12))
dec = ret[ret["midx"] % 12 == 0].copy(); dec["yr"] = dec["midx"] // 12 - 1
R = pg.merge(dec[["pid", "yr", "f12", "b12"]], on=["pid", "yr"], how="inner")
R = R[R["yr"].isin(KEEP)].copy(); R["zias"] = z(R["ias"])
res, N, ncl = creg(R, "f12", ["zias"])
rows.append(dict(test="Next-12m market-adjusted return ~ IAS[t]", ias_coef_per_SD=res[0][1], ias_t=res[0][2], control="none", N=N, parents=ncl))
res2, N2, ncl2 = creg(R.dropna(subset=["b12"]), "f12", ["zias", "b12"])
rows.append(dict(test="Next-12m return ~ IAS[t] (+ past-12m return)", ias_coef_per_SD=res2[0][1], ias_t=res2[0][2], control="past-12m return", N=N2, parents=ncl2))

# ---------- (B) forward fundamentals ----------
comp = pd.read_parquet(f"{W}/wrds_extra/compustat_annual.parquet"); comp["pid"] = comp.parent_id.astype(str)
comp["at"] = pd.to_numeric(comp["at"], errors="coerce"); comp["yr"] = pd.to_datetime(comp.datadate).dt.year
comp["roa"] = (pd.to_numeric(comp.oibdp, errors="coerce") / comp["at"]).where(comp["at"] > 0).clip(-1, 1)
cm = comp[comp["at"] > 0].groupby(["pid", "yr"]).agg(roa=("roa", "mean")).reset_index().sort_values(["pid", "yr"])
cm["roa_next"] = cm.groupby("pid")["roa"].shift(-1); cm["ny"] = cm.groupby("pid")["yr"].shift(-1)
cm = cm[cm["ny"] - cm["yr"] == 1]
Fd = pg.merge(cm, on=["pid", "yr"], how="inner")
Fd = Fd[Fd["yr"].isin(KEEP)].dropna(subset=["roa_next", "roa", "ias"]).copy(); Fd["zias"] = z(Fd["ias"])
res, N, ncl = creg(Fd, "roa_next", ["zias", "roa"])
rows.append(dict(test="Next-year operating ROA ~ IAS[t] (+ current ROA)", ias_coef_per_SD=res[0][1], ias_t=res[0][2],
                 control=f"current ROA {res[1][1]} (t={res[1][2]})", N=N, parents=ncl))

T = pd.DataFrame(rows)
print(T.to_string(index=False))
print(f"\nmean forward-12m return = {R['f12'].mean()*100:.1f}% ; mean ROA = {Fd['roa'].mean():.3f} (sd {Fd['roa'].std():.3f})")
T.to_csv("P4_score_predicts.csv", index=False)
print("wrote P4_score_predicts.csv")


### 4.5 External clock (public-report lag)

**`external_clock/a4_clock_sample.py`** — Funnel, representativeness, and the severity-reweighted lag (A4_* outputs; the reweighted ≤90/≤30/>180 shares).

In [ ]:
# ====================================================================================
# PIPELINE FILE: external_clock/a4_clock_sample.py
# Funnel, representativeness, and the severity-reweighted lag (A4_* outputs; the reweighted ≤90/≤30/>180 shares).
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""Analysis 4: hand-checked clock sample: funnel, representativeness of the 223, severity-reweighted lag,
random 30 cases for an independent second dating."""
import numpy as np, pandas as pd
from scipy import stats
W = "/home/claude/w2"; KIT = "/home/claude/p2/kit"
ons = pd.read_csv(f"{KIT}/data/onsets.csv", dtype={"parent_id": str}, parse_dates=["msci_date"])
par = pd.read_csv(f"{KIT}/data/parents.csv", dtype={"parent_id": str}, low_memory=False).drop_duplicates("parent_id")
o = ons.merge(par[["parent_id", "country", "sector"]], on="parent_id", how="left")
o = o[o.country.isin(["US", "GB"])]
fl = o.flag_backfill_20190701.astype(bool) | o.flag_floor_20170831.astype(bool)
U = o[~fl].copy()
C = pd.read_pickle(f"{W}/x3/contro3_Controversies.pkl"); H = pd.read_pickle(f"{W}/x3/hand3_m.pkl")
H["MSCI date"] = pd.to_datetime(H["MSCI date"]); H["Audited earliest date"] = pd.to_datetime(H["Audited earliest date"], errors="coerce")
ver = H["Audit status"].astype(str).str.startswith("Verified"); v5 = H["MSCI date"] == "2026-03-23"
V = H[ver & ~v5].copy(); V["lag"] = (V["MSCI date"] - V["Audited earliest date"]).dt.days
vs_add = H["Date source"] == "GDELT dated article (very-severe add-on)"
c_in = C.merge(H[["Onset ID"]], on="Onset ID", how="left", indicator=True)
funnel = [
    ("US and UK controversy onsets, excluding the 1 July 2019 backfill and 31 August 2017 floor dates", len(U), len(o) - len(U),
     f"{len(o):,} US/UK onsets before excluding {int(fl.sum()):,} backfill/floor onsets"),
    ("With at least one GDELT article matching the firm and the key issue", len(C), len(U) - len(C),
     "no matched article in the GDELT keyword search"),
    ("Selected for the audit file", len(H), len(C) - (len(H) - int(vs_add.sum())),
     f"{len(H) - int(vs_add.sum())} matched onsets retained after a first screening round "
     f"({int(H[~vs_add]['Onset ID'].isin(C[C['Within 0–90 d']]['Onset ID']).sum())} with an article 0–90 days before the MSCI date), "
     f"plus {int(vs_add.sum())} remaining very severe onsets"),
    ("Incident and earliest public report verified", int(ver.sum()), len(H) - int(ver.sum()),
     "; ".join(f"{k}: {v}" for k, v in H.loc[~ver, "Audit status"].value_counts().items())),
    ("Final sample", len(V), int(v5[ver].sum()), "dated 23 March 2026, MSCI's re-rating of all issuers"),
]
F = pd.DataFrame(funnel, columns=["stage", "N", "dropped", "reason for drop"])
print(F.to_string()); F.to_csv(f"{W}/out/A4_funnel.csv", index=False)

# ------------------------------------------------------------------ representativeness: 223 vs other onsets
U["verified"] = U.onset_id.isin(V["Onset ID"])
print("verified onsets found among the 14,207:", int(U.verified.sum()))
comp = pd.read_parquet(f"{W}/wrds_extra/compustat_annual.parquet"); comp["parent_id"] = comp.parent_id.astype(str)
comp["at"] = pd.to_numeric(comp["at"], errors="coerce"); comp["datadate"] = pd.to_datetime(comp.datadate).astype("datetime64[ns]")
comp = comp[comp["at"] > 0].sort_values("datadate")
x = U.copy(); x["_d"] = (x.msci_date - pd.DateOffset(months=3)).astype("datetime64[ns]")
x = pd.merge_asof(x.sort_values("_d"), comp[["parent_id", "datadate", "at"]], left_on="_d", right_on="datadate",
                  by="parent_id", direction="backward", tolerance=pd.Timedelta(days=550))
x["log_at"] = np.log(x["at"]); x["year"] = x.msci_date.dt.year
x["period"] = np.select([x.year <= 2019, x.year == 2020, x.year <= 2025], ["2017–2019", "2020", "2021–2025"], "2026")
rows = []


def share_rows(var, lab):
    tab = pd.crosstab(x[var], x.verified)
    chi = stats.chi2_contingency(tab)
    for k in tab.index:
        rows.append(dict(variable=lab, category=k, verified_pct=100 * tab.loc[k, True] / tab[True].sum(),
                         other_pct=100 * tab.loc[k, False] / tab[False].sum(), test=f"chi2 p = {chi[1]:.3f}"))


share_rows("severity", "Severity"); share_rows("period", "Year of MSCI date"); share_rows("country", "Country")
sec = x.sector.fillna("missing"); top = sec.value_counts().index[:10]
x["sector2"] = np.where(sec.isin(top), sec, "other"); share_rows("sector2", "Sector")
a, b = x[x.verified].log_at.dropna(), x[~x.verified].log_at.dropna()
tt = stats.ttest_ind(a, b, equal_var=False)
rows.append(dict(variable="Log total assets (Compustat)", category=f"mean (N = {len(a)} / {len(b):,})",
                 verified_pct=a.mean(), other_pct=b.mean(), test=f"t = {tt.statistic:.2f}"))
rows.append(dict(variable="Onsets", category="N", verified_pct=int(x.verified.sum()), other_pct=int((~x.verified).sum()), test=""))
R = pd.DataFrame(rows); R.to_csv(f"{W}/out/A4_representativeness.csv", index=False); print(R.round(2).to_string())

# ------------------------------------------------------------------ severity-reweighted lag
pop = U.severity.value_counts(normalize=True)
V["sevk"] = V.Severity.map({"moderate": "moderate", "severe": "severe", "very_severe": "very_severe", "very severe": "very_severe"}).fillna(V.Severity)
smp = V.sevk.value_counts(normalize=True)
V["w"] = V.sevk.map(pop / smp)


def wq(v, w, q):
    i = np.argsort(v); v, w = v[i], w[i]; c = np.cumsum(w) / w.sum(); return v[np.searchsorted(c, q)]


lv, wv = V.lag.values.astype(float), V.w.values
RW = pd.DataFrame([
    dict(weighting="Unweighted (verified 223)", median=np.median(lv), mean=lv.mean(), p25=np.percentile(lv, 25), p75=np.percentile(lv, 75),
         le90=100 * (lv <= 90).mean()),
    dict(weighting="Reweighted to the severity mix of the 14,207 onsets", median=wq(lv, wv, .5), mean=np.average(lv, weights=wv),
         p25=wq(lv, wv, .25), p75=wq(lv, wv, .75), le90=100 * np.average(lv <= 90, weights=wv))])
mix = pd.DataFrame({"share_all_onsets": pop, "share_verified": smp, "weight": pop / smp})
print(mix.round(3)); print(RW.round(1).to_string())
RW.to_csv(f"{W}/out/A4_reweighted_lag.csv", index=False); mix.to_csv(f"{W}/out/A4_severity_mix.csv")

# ------------------------------------------------------------------ random 30 for an independent second dating
rs = V.sample(30, random_state=20261031)
blind = rs[["ID", "Firm", "Country", "Issue", "Severity", "MSCI date", "Identified incident / note"]].copy()
blind.columns = ["ID", "Firm", "Country", "MSCI key issue", "Severity", "MSCI date", "Incident (as identified in audit 1)"]
blind["Second coder: earliest public date"] = ""; blind["Second coder: source link"] = ""; blind["Second coder: notes"] = ""
key = rs[["ID", "Audited earliest date", "Audit source link"]]
with pd.ExcelWriter(f"{W}/out/A4_second_coder_30.xlsx") as xw:
    blind.to_excel(xw, sheet_name="For second coder", index=False)
    key.to_excel(xw, sheet_name="First coder (keep hidden)", index=False)
print("checked-by field in the matched file:", C["Checked by"].value_counts(dropna=False).to_dict())


**`external_clock/09_external_clock.py`** — External-clock estimator (eq. 9) on the keyword clock.

In [ ]:
# ====================================================================================
# PIPELINE FILE: external_clock/09_external_clock.py
# External-clock estimator (eq. 9) on the keyword clock.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""
09_external_clock.py  --  External-incident-clock identification (paper Section 8, eq. 9)

PURPOSE
  Re-time the rating's incorporation of a controversy against a PROVIDER-INDEPENDENT
  first-awareness date, instead of against MSCI's own logging date. This breaks the
  "MSCI-on-MSCI" circularity: with an outside clock, the months between the incident
  becoming public and the rating moving is staleness relative to PUBLIC information.

WHAT THIS SCRIPT DOES
  (A) SCRAPE first-awareness dates for the MSCI issuer universe from public sources:
         - GDELT DOC 2.0 API        (global news; earliest article mentioning the firm + ESG-adverse terms)
         - Violation Tracker (Good Jobs First)   (US enforcement actions; official dates)
      RepRisk is the preferred commercial feed; if a licensed RepRisk extract is present
      it is used in preference to GDELT.  (RepRisk cannot be scraped and is not free.)
  (B) RESOLVE each external incident to an MSCI issuer (name + ticker/ISIN fuzzy match).
  (C) BUILD event time = calendar months from first-awareness date to the first
      subsequent MSCI rating update (letter change), right-censored at 24 months.
  (D) ESTIMATE the discrete-time complementary-log-log hazard (paper eq. 9):
         h_{im} = 1 - exp{ -exp( lambda_m + delta' X_i ) }
      X_i = incident severity, reach, novelty; distance to nearest letter threshold;
      industry and calendar-month effects.
  (E) PLACEBO: re-estimate on enforcement dates only (Violation Tracker). Enforcement
      dates do not co-move mechanically with media, so a lag that persists there is not
      a shared-source artefact.

NETWORK NOTE
  Scraping requires outbound access to api.gdeltproject.org and
  violationtracker.goodjobsfirst.org. Run this step in an environment where those
  hosts are reachable. In a locked sandbox, run with --synthetic to verify the
  estimator end-to-end on a clearly-labelled SYNTHETIC incident file (NOT real results).

USAGE
  python 09_external_clock.py --scrape   --issuers issuers.csv --out incidents.csv
  python 09_external_clock.py --estimate --incidents incidents.csv --panel panel_fy.parquet
  python 09_external_clock.py --synthetic        # offline demo of the estimator only
"""
import argparse, sys, time, json, math
import numpy as np, pandas as pd

# ----------------------------------------------------------------------------- SCRAPERS
def scrape_gdelt(name, start="2017-01-01", end="2026-07-01", adverse_terms=None, pause=0.3):
    """Earliest GDELT article mentioning `name` together with an ESG-adverse term.
    Returns an ISO date string or None. Requires network (api.gdeltproject.org)."""
    import requests, urllib.parse
    adverse_terms = adverse_terms or ["lawsuit","fine","investigation","spill","recall",
        "fraud","violation","strike","boycott","emission","pollution","settlement"]
    q = f'"{name}" ({" OR ".join(adverse_terms)})'
    url = ("https://api.gdeltproject.org/api/v2/doc/doc?query="
           + urllib.parse.quote(q)
           + f"&mode=artlist&maxrecords=250&sort=dateasc&format=json"
           + f"&startdatetime={start.replace('-','')}000000&enddatetime={end.replace('-','')}000000")
    try:
        r = requests.get(url, timeout=30); r.raise_for_status()
        arts = r.json().get("articles", [])
        if not arts: return None
        dates = [a.get("seendate","") for a in arts if a.get("seendate")]
        time.sleep(pause)
        return min(dates)[:8] if dates else None     # YYYYMMDD
    except Exception as e:
        print(f"  [gdelt] {name!r}: {e}", file=sys.stderr); return None

def scrape_violationtracker(name, pause=0.3):
    """Earliest US enforcement-action date for `name` from Violation Tracker.
    Returns ISO date or None. Requires network (violationtracker.goodjobsfirst.org)."""
    import requests, urllib.parse
    url = ("https://violationtracker.goodjobsfirst.org/?company="
           + urllib.parse.quote(name) + "&order=pen_year&sort=asc")
    try:
        r = requests.get(url, timeout=30, headers={"User-Agent":"research-scraper"})
        r.raise_for_status()
        import re
        yrs = re.findall(r'<td[^>]*>\s*(20[0-2]\d)\s*</td>', r.text)
        time.sleep(pause)
        return f"{min(yrs)}-01-01" if yrs else None
    except Exception as e:
        print(f"  [vtracker] {name!r}: {e}", file=sys.stderr); return None

def do_scrape(issuers_csv, out_csv):
    iss = pd.read_csv(issuers_csv)            # columns: issuerid, name [, ticker, isin]
    rows=[]
    for _,r in iss.iterrows():
        g = scrape_gdelt(r["name"]); v = scrape_violationtracker(r["name"])
        rows.append(dict(issuerid=r["issuerid"], name=r["name"],
                         gdelt_first=g, enforce_first=v))
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"wrote {out_csv} ({len(rows)} issuers)")

# ----------------------------------------------------------------------------- ESTIMATOR
def cloglog_irls(X, y, iters=50, tol=1e-8):
    """Minimal complementary-log-log GLM via IRLS. X includes intercept. Returns beta, se."""
    X=np.asarray(X,float); y=np.asarray(y,float); beta=np.zeros(X.shape[1])
    for _ in range(iters):
        eta=np.clip(X@beta,-30,30); expeta=np.exp(eta)
        mu=1-np.exp(-expeta); mu=np.clip(mu,1e-10,1-1e-10)
        dmu=expeta*np.exp(-expeta)                      # d mu / d eta
        W=(dmu**2)/(mu*(1-mu)); z=eta+(y-mu)/dmu
        XtW=X.T*W; A=XtW@X + 1e-8*np.eye(X.shape[1]); beta_new=np.linalg.solve(A, XtW@z)
        if np.max(np.abs(beta_new-beta))<tol: beta=beta_new; break
        beta=beta_new
    cov=np.linalg.inv((X.T*W)@X + 1e-8*np.eye(X.shape[1])); return beta, np.sqrt(np.diag(cov))

def build_spell_panel(df):
    """df: one row per (firm,incident) with months_to_update (int) and censored flag,
    plus severity, dist_threshold, industry, cal_month. Expand to discrete-time spells."""
    recs=[]
    for _,r in df.iterrows():
        T=int(r["months_to_update"])
        for m in range(1, T+1):
            event = 1 if (m==T and not r["censored"]) else 0
            recs.append((r.name, m, event, r["severity"], r["dist_threshold"],
                         r["industry"], r["cal_month"]))
    return pd.DataFrame(recs, columns=["id","m","event","severity","dist_threshold","industry","cal_month"])

def estimate(spell, late_col=None):
    sp=spell.copy()
    # baseline month indicators (grouped to keep columns modest), + covariates
    sp["mbin"]=pd.cut(sp["m"],[0,3,6,9,12,18,24],labels=False)
    Xd=pd.get_dummies(sp["mbin"],prefix="m",drop_first=True).astype(float)
    feats=[sp[["severity","dist_threshold"]].reset_index(drop=True), Xd.reset_index(drop=True)]
    X=pd.concat(feats,axis=1).astype(float); X.insert(0,"const",1.0)
    beta,se=cloglog_irls(X.values, sp["event"].values)
    out=pd.DataFrame({"term":X.columns,"coef":beta,"se":se,"z":beta/se})
    return out

def make_synthetic(n=4000, seed=7):
    """Clearly-labelled SYNTHETIC incidents to exercise the estimator offline.
    NOT REAL DATA; produces illustrative coefficients only."""
    rng=np.random.default_rng(seed)
    severity=rng.integers(1,4,n)                        # 1 mod,2 sev,3 vsev
    dist=rng.uniform(0,1,n)                             # distance to letter threshold
    # true hazard rises with severity, falls with distance; baseline ~ monthly
    base=-3.0+0.6*severity-1.2*dist
    T=np.full(n,24); cens=np.ones(n,bool)
    for i in range(n):
        for m in range(1,25):
            h=1-math.exp(-math.exp(base[i]+0.02*m))
            if rng.uniform()<h: T[i]=m; cens[i]=False; break
    return pd.DataFrame(dict(months_to_update=T,censored=cens,severity=severity,
        dist_threshold=dist,industry=rng.integers(0,11,n),cal_month=rng.integers(1,13,n)))

# ----------------------------------------------------------------------------- MAIN
if __name__=="__main__":
    ap=argparse.ArgumentParser()
    ap.add_argument("--scrape",action="store_true"); ap.add_argument("--estimate",action="store_true")
    ap.add_argument("--synthetic",action="store_true")
    ap.add_argument("--issuers"); ap.add_argument("--incidents"); ap.add_argument("--panel"); ap.add_argument("--out")
    a=ap.parse_args()
    if a.scrape:
        do_scrape(a.issuers, a.out or "incidents.csv")
    elif a.synthetic:
        print(">>> SYNTHETIC DEMO (illustrative, NOT real results) <<<")
        df=make_synthetic(); sp=build_spell_panel(df); res=estimate(sp)
        pd.set_option("display.float_format",lambda x:f"{x:7.3f}")
        print(res.to_string(index=False))
        sev=res.loc[res.term=="severity"]
        print(f"\nSeverity gradient in the update hazard: coef={sev.coef.values[0]:.3f} "
              f"(z={sev.z.values[0]:.1f}) -> more severe incidents are incorporated FASTER, as expected.")
        print("Interpretation template: exp(coef) is the hazard ratio per severity step; a positive,")
        print("significant severity coefficient on the EXTERNAL clock is the identification the paper needs.")
    elif a.estimate:
        inc=pd.read_csv(a.incidents); 
        print("Loaded", len(inc), "incidents; merge with", a.panel, "then build_spell_panel()/estimate().")
        print("See EXTERNAL_CLOCK_PLAN.md for the merge keys and covariate construction.")
    else:
        ap.print_help()


**`audit/lag_v3.py`** — THE MANUSCRIPT CLOCK: hand-verified lag on the 223 verified onsets + audited-date returns (Table 9).

In [ ]:
# ====================================================================================
# PIPELINE FILE: audit/lag_v3.py
# THE MANUSCRIPT CLOCK: hand-verified lag on the 223 verified onsets + audited-date returns (Table 9).
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""Re-run of the external-clock (logging lag) tests on the 27-Oct-2026 hand audit (727 rows; audited earliest dates)."""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

sys.path.insert(0, "/home/claude/p2/kit"); sys.path.insert(0, "/home/claude/p8")
import es_engine as E
import phase2_analysis as P

X = Path(sys.argv[1]); F = Path(sys.argv[2]); OUT = Path(sys.argv[3]); OUT.mkdir(parents=True, exist_ok=True)
LOG = []


def log(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); LOG.append(s)


H = pd.read_pickle(X / "hand3_m.pkl")
for c in ["MSCI date", "News date", "Audited earliest date"]:
    H[c] = pd.to_datetime(H[c], errors="coerce")
H["ver"] = H["Audit status"].astype(str).str.startswith("Verified")
H["screen_ok"] = H["Audit verdict"].isin(["Correct (headline screen)", "Correct (headline only)"])
H["v5"] = H["MSCI date"] == "2026-03-23"
H["year"] = H["MSCI date"].dt.year
H["period"] = np.select([H.year <= 2019, H.year == 2020, H.year <= 2025], ["2018-19", "2020", "2021-25"], "2026")
H["sev"] = H["Severity rank (1–3)"].astype(int)
H["severe"] = (H.sev == 2).astype(int); H["vsev"] = (H.sev == 3).astype(int)
H["uk"] = (H.Country == "GB").astype(int)
H["y2020"] = (H.period == "2020").astype(int); H["early"] = (H.period == "2018-19").astype(int); H["y2026"] = (H.period == "2026").astype(int)
H["pid"] = H["Parent ID"]
H["ext"] = np.where(H.ver, H["Audited earliest date"], np.where(H.screen_ok, H["News date"], pd.NaT))
H["ext"] = pd.to_datetime(H.ext)
H["lag"] = (H["MSCI date"] - H.ext).dt.days

S = {"A. Verified (audited earliest date), excl. v5.0 re-rating day": H[H.ver & ~H.v5],
     "B. Verified, excl. all 2026": H[H.ver & (H.year < 2026)],
     "C. Verified incl. v5.0 re-rating day": H[H.ver],
     "D. Verified + headline-screen rows (GDELT date), excl. v5.0": H[(H.ver | H.screen_ok) & ~H.v5]}
PRIM = "A. Verified (audited earliest date), excl. v5.0 re-rating day"

# ---------------------------------------------------------------- L1 descriptive
def st(d, lab):
    return dict(sample=lab, N=len(d), events=d["Event ID"].nunique(), firms=d.pid.nunique(), mean=d.lag.mean(),
                median=d.lag.median(), p25=d.lag.quantile(.25), p75=d.lag.quantile(.75),
                le30=(d.lag <= 30).mean(), le90=(d.lag <= 90).mean(), le180=(d.lag <= 180).mean(),
                gt365=(d.lag > 365).mean())
rows = []
for k, d in S.items():
    rows.append(st(d, k))
A = S[PRIM]
for g, d in A.groupby("Severity"):
    rows.append(st(d, f"   A: {g}"))
for g, d in A.groupby("period"):
    rows.append(st(d, f"   A: {g}"))
for g, d in A.groupby("Country"):
    rows.append(st(d, f"   A: {g}"))
for g, d in A.groupby("Audit verdict"):
    rows.append(st(d, f"   A: verdict = {g}"))
ev = A.sort_values("sev", ascending=False).drop_duplicates("Event ID")
rows.append(st(ev, "A, one row per incident (Event ID)"))
L1 = pd.DataFrame(rows); L1.to_csv(OUT / "L1v3_lag_descriptive.csv", index=False)
log("=== L1 audited logging lag ===\n" + L1.round(2).to_string(index=False))

# ---------------------------------------------------------------- L2 regressions
rr = []
def fit(f, d, lab, quant=False):
    if quant:
        r = smf.quantreg(f, d).fit(q=0.5)
    else:
        r = smf.ols(f, d).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.pid)[0]})
    for t in ["severe", "vsev", "y2020", "early", "y2026", "uk"]:
        if t in r.params:
            rr.append(dict(sample=lab.split(",")[0], model=lab, term=t, coef=r.params[t], t=r.tvalues[t], p=r.pvalues[t], N=int(r.nobs)))
base = "severe + vsev + y2020 + early + uk"
for k in [PRIM, "B. Verified, excl. all 2026", "D. Verified + headline-screen rows (GDELT date), excl. v5.0"]:
    d = S[k].copy(); d["loglag"] = np.log1p(d.lag.clip(lower=0))
    b = base + (" + y2026" if d.y2026.sum() > 0 else "")
    fit(f"lag ~ {b}", d, f"{k[:2]} days, OLS")
    fit(f"lag ~ {b} + C(Issue)", d, f"{k[:2]} days, OLS + issue FE")
    fit(f"loglag ~ {b} + C(Issue)", d, f"{k[:2]} log(1+lag) + issue FE")
    fit(f"lag ~ {b} + C(Issue)", d, f"{k[:2]} median regression + issue FE", quant=True)
L2 = pd.DataFrame(rr); L2.to_csv(OUT / "L2v3_lag_regressions.csv", index=False)
log("\n=== L2 regressions ===\n" + L2.round(3).to_string(index=False))

# selection: which rows were verified?
H["one"] = 1
H["veri"] = H.ver.astype(int)
r = smf.ols("veri ~ severe + vsev + y2020 + early + y2026 + uk", H).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(H.pid)[0]})
SEL = pd.DataFrame({"term": r.params.index, "coef": r.params.values, "t": r.tvalues.values})
SEL.to_csv(OUT / "L3v3_selection_into_verified.csv", index=False)
log("\n=== L3 probability that a row has a verified date (linear probability) ===\n" + SEL.round(3).to_string(index=False))
dist = pd.cut(A.lag, [-10000, -1, 30, 90, 180, 365, 100000], labels=["<0", "0-30", "31-90", "91-180", "181-365", ">365"]).value_counts().sort_index()
dist = pd.DataFrame({"lag_bin": dist.index.astype(str), "N": dist.values, "share": dist.values / dist.sum()})
dist.to_csv(OUT / "L4v3_lag_distribution.csv", index=False)
log("\n" + dist.round(3).to_string(index=False))

# ---------------------------------------------------------------- K external vs internal clock to rating change
R = P.Ratings("/home/claude/p2/kit/data/ratings.parquet")
par = pd.read_csv("/home/claude/p2/kit/data/parents.csv", dtype={"parent_id": str}, low_memory=False).drop_duplicates("parent_id")
k = A.rename(columns={"pid": "parent_id", "Onset ID": "onset_id", "MSCI date": "msci_date"}).copy()
k = k.merge(par[["parent_id", "sector"]], on="parent_id", how="left").reset_index(drop=True)
k["spell_id"] = k.onset_id + "_" + k.index.astype(str)
krows, hz = [], []
for clock, dcol in [("external (audited news date)", "ext"), ("internal (MSCI date)", "msci_date")]:
    b = k.join(R.asof(k, dcol)).dropna(subset=["band"])
    b["cal_month"] = b[dcol].dt.month
    for kind in ["any", "down"]:
        sp, one = P.build_spells(b, dcol, R, kind)
        med, p12, p24 = P.km_median(one)
        krows.append(dict(clock=clock, event="letter change" if kind == "any" else "downgrade", spells=len(one),
                          events=int(one.event.sum()), KM_median_months=med, P12=p12, P24=p24))
        res, rws = P.fit_hazard(sp, ["severe", "vsev", "dist_threshold"], [], "parent_id", f"{clock}|{kind}")
        hz += [dict(r_, clock=clock, event=kind) for r_ in rws if r_["term"] in ["severe", "vsev", "dist_threshold"]]
K = pd.DataFrame(krows); K.to_csv(OUT / "K1v3_KM_external_vs_internal.csv", index=False)
HZ = pd.DataFrame(hz); HZ.to_csv(OUT / "K2v3_hazard_external_vs_internal.csv", index=False)
log("\n=== K1 months from news (external) vs from MSCI logging (internal) to the rating change ===\n" + K.round(3).to_string(index=False))
log("\n=== K2 hazard (cloglog, month bins), severity effects ===\n" + HZ.round(3).to_string(index=False))

# ---------------------------------------------------------------- V returns at audited dates
D = E.Daily(pd.read_parquet(F / "daily/wrds_extra/crsp_daily.parquet"), pd.read_parquet(F / "main/wrds_extra/ff_daily.parquet"),
            pd.read_parquet(F / "main/wrds_extra/crsp_market_daily.parquet"))
d = A.drop_duplicates(["pid", "ext"]).reset_index(drop=True)
WIN = [(-30, -2), (-1, 1), (0, 5), (-5, 20)]
cn = D.car(d.pid.values, d.ext, WIN, est=(-310, -90))
cp = D.car(d.pid.values, d.ext - pd.Timedelta(days=364), WIN, est=(-310, -90))
cm = D.car(d.pid.values, d["MSCI date"], WIN, est=(-310, -90))
vr = []
for w in ["car_-30_-2", "car_-1_1", "car_0_5", "car_-5_20"]:
    for kind, c in [("audited news date", cn), ("placebo (news - 364 d)", cp), ("MSCI date", cm)]:
        vr.append(dict(window=w, date=kind, **E.mean_test(c[w], d.pid, d.ext.astype(str))))
    vr.append(dict(window=w, date="news minus placebo", **E.mean_test(cn[w] - cp[w], d.pid, d.ext.astype(str))))
V = pd.DataFrame(vr); V.to_csv(OUT / "V1v3_returns_at_audited_dates.csv", index=False)
log("\n=== V returns at audited news dates (US-listed; one row per firm-date) ===\n" + V[["window", "date", "N", "mean", "t"]].round(4).to_string(index=False))
(OUT / "LOG_v3.txt").write_text("\n".join(LOG), encoding="utf-8")


### 4.6 Consequences: ownership reallocation, liquidity, and downgrade drift

**`consequences/a1_ownership_linked.py`** — Ownership linkage and revision frequency (Table 4 panel).

In [ ]:
# ====================================================================================
# PIPELINE FILE: consequences/a1_ownership_linked.py
# Ownership linkage and revision frequency (Table 4 panel).
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""Analysis 1: ESG-mandated ownership after controversy-linked vs other downgrades; controversy-date and public-report-date checks."""
import sys
import duckdb
import numpy as np
import pandas as pd
import statsmodels.api as sm

sys.path.insert(0, "/home/claude/p2/kit")
import phase2_analysis as P

W = "/home/claude/w2"
KIT = "/home/claude/p2/kit"
OUT = f"{W}/out"; import os; os.makedirs(OUT, exist_ok=True)


def fe(s, p):
    return pd.get_dummies(s.astype(str), prefix=p, drop_first=True).astype(float)


def cl_ols(y, X, g):
    d = pd.concat([y.rename("y"), X, g.rename("g")], axis=1).dropna()
    Xc = sm.add_constant(d[X.columns], has_constant="add")
    Xc = Xc.loc[:, (Xc.std() > 0) | (Xc.columns == "const")]
    r = sm.OLS(d.y, Xc).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.g.astype(str))[0]})
    return r, d


# ------------------------------------------------------------------ ratings, onsets, controls
R = P.Ratings(f"{KIT}/data/ratings.parquet")
par = pd.read_csv(f"{KIT}/data/parents.csv", dtype={"parent_id": str}, low_memory=False).drop_duplicates("parent_id")
ons = pd.read_csv(f"{KIT}/data/onsets.csv", dtype={"parent_id": str}, parse_dates=["msci_date"])
ons = ons[~(ons.flag_backfill_20190701.astype(bool) | ons.flag_floor_20170831.astype(bool))]
sev = ons[ons.severity.isin(["severe", "very_severe"])].copy()
sev["msci_date"] = sev.msci_date.astype("datetime64[ns]")

comp = pd.read_parquet(f"{W}/wrds_extra/compustat_annual.parquet")
comp["datadate"] = pd.to_datetime(comp.datadate).astype("datetime64[ns]"); comp["parent_id"] = comp.parent_id.astype(str)
for c in ["at", "ceq", "lt", "ni", "csho", "prcc_f", "dltt", "dlc"]:
    comp[c] = pd.to_numeric(comp[c], errors="coerce")
comp["log_at"] = np.log(comp["at"].where(comp["at"] > 0))
comp["lev"] = ((comp.dltt.fillna(0) + comp.dlc.fillna(0)) / comp["at"]).where(comp["at"] > 0).clip(0, 2)
comp["roa"] = (comp.ni / comp["at"]).where(comp["at"] > 0).clip(-1, 1)
me = comp.csho * comp.prcc_f
comp["bm"] = np.log((comp.ceq / me).where((comp.ceq > 0) & (me > 0)))
comp = comp.sort_values("datadate"); CTRL = ["log_at", "lev", "roa", "bm"]


def with_ctrl(df, dcol):
    x = df.reset_index().rename(columns={"index": "_i"})
    x["_d"] = (pd.to_datetime(x[dcol]) - pd.DateOffset(months=3)).astype("datetime64[ns]")
    m = pd.merge_asof(x.sort_values("_d"), comp[["parent_id", "datadate"] + CTRL].dropna(subset=["log_at"]),
                      left_on="_d", right_on="datadate", by="parent_id", direction="backward", tolerance=pd.Timedelta(days=550))
    return m.set_index("_i").sort_index()[CTRL]


# ------------------------------------------------------------------ downgrade events and linkage
ch = R.changes.copy(); ch["cdate"] = ch.cdate.astype("datetime64[ns]")
dg = ch[ch.down].copy()
# definition 1: severe/very severe onset logged in the 12 months before the downgrade (inclusive of the downgrade day)
m = dg.merge(sev[["parent_id", "msci_date", "issue", "onset_id"]], on="parent_id", how="left")
m["in12"] = (m.msci_date <= m.cdate) & (m.msci_date > m.cdate - pd.DateOffset(months=12))
link1 = m[m.in12].groupby(["parent_id", "cdate"]).size().rename("n_link").reset_index()
dg = dg.merge(link1, on=["parent_id", "cdate"], how="left"); dg["linked1"] = dg.n_link.notna()

# definition 2: the linked controversy's key-issue score fell between the day before the onset and the downgrade
ISSUE_SCORE = {"TOXIC_EMISS": "TOXIC_EMISS_WSTE_SCORE", "BUS_ETHICS_ETHICSFRAUD": "CORP_BEHAV_ETHICS_SCORE",
               "BUS_ETHICS_ANTICOMP": "CORP_BEHAV_ETHICS_SCORE", "BUS_ETHICS_CORRUPTION": "CORP_BEHAV_ETHICS_SCORE"}
pairs = m[m.in12][["parent_id", "cdate", "msci_date", "issue"]].copy()
pairs["field"] = pairs.issue.map(lambda s: ISSUE_SCORE.get(s, f"{s}_SCORE"))
con = duckdb.connect(); con.sql("PRAGMA threads=6")
ids = pd.DataFrame({"ISSUERID": pairs.parent_id.unique()}); con.register("ids", ids)
flds = pd.DataFrame({"FIELD_ID": pairs.field.unique()}); con.register("flds", flds)
sc = con.sql(f"""WITH l AS (SELECT s.ISSUERID,s.FIELD_ID,s.START_DATE,TRY_CAST(s.VALUE AS DOUBLE) v,
    ROW_NUMBER() OVER(PARTITION BY s.ISSUERID,s.FIELD_ID,s.START_DATE ORDER BY s.VERSION_START_DATE DESC) rn
    FROM '{W}/jul/HIST_SUMMARY_SCORES_20260731.parquet' s JOIN ids USING(ISSUERID) JOIN flds USING(FIELD_ID))
    SELECT ISSUERID parent_id, FIELD_ID field, START_DATE, v FROM l WHERE rn=1 AND v IS NOT NULL""").df()
sc["date"] = pd.to_datetime(sc.START_DATE.astype(str), format="%Y%m%d").astype("datetime64[ns]")
sc = sc.sort_values("date")


def score_at(df, dcol):
    x = df.reset_index().sort_values(dcol)
    x[dcol] = x[dcol].astype("datetime64[ns]")
    r = pd.merge_asof(x, sc[["parent_id", "field", "date", "v"]], left_on=dcol, right_on="date",
                      by=["parent_id", "field"], direction="backward")
    return r.set_index("index").sort_index().v


pairs["pre"] = pairs.msci_date - pd.Timedelta(days=1)
pairs["s_pre"] = score_at(pairs, "pre").values
pairs["s_dg"] = score_at(pairs, "cdate").values
pairs["fell"] = pairs.s_dg < pairs.s_pre - 1e-9
cov2 = pairs.groupby(["parent_id", "cdate"]).agg(any_scored=("s_pre", lambda s: s.notna().any() ), fell=("fell", "any")).reset_index()
dg = dg.merge(cov2, on=["parent_id", "cdate"], how="left")
dg["linked2"] = dg.fell.fillna(False).astype(bool)
print("downgrades (all years):", len(dg), "| linked1:", int(dg.linked1.sum()), "| linked2 (score fell in the linked issue):", int(dg.linked2.sum()),
      "| linked1 with an issue score observed:", int(dg.any_scored.fillna(False).sum()))

# ------------------------------------------------------------------ quarterly ownership panel (US-listed, CRSP MF)
q = pd.read_csv(f"{W}/crsp_ownership/mandate_ownership_quarterly.csv", dtype={"parent_id": str})
lo_, hi_ = q.mandate_own.quantile([0.01, 0.99])
q["all"] = q.mandate_own.clip(lo_, hi_) * 100
q["idx"] = q.mandate_own_index.clip(0, hi_) * 100
q["oth"] = (q.mandate_own - q.mandate_own_index).clip(0, hi_) * 100
q["qp"] = pd.PeriodIndex(q.q, freq="Q"); q["qi"] = q.qp.dt.year * 4 + q.qp.dt.quarter
q = q.sort_values(["parent_id", "qi"]).reset_index(drop=True)
qtr = lambda d: d.dt.year * 4 + d.dt.quarter
dg["qi"] = qtr(dg.cdate)
up = ch[~ch.down].copy(); up["qi"] = qtr(up.cdate)
g = dg.groupby(["parent_id", "qi"]).agg(l1=("linked1", "max"), l2=("linked2", "max")).reset_index().assign(dgq=1.0)
q = q.merge(g, on=["parent_id", "qi"], how="left").merge(up[["parent_id", "qi"]].drop_duplicates().assign(upgrade=1.0), how="left")
q["downgrade"] = q.dgq.fillna(0.0); q["upgrade"] = q.upgrade.fillna(0.0)
q["dg_link1"] = q.downgrade * q.l1.fillna(False).astype(float); q["dg_not1"] = q.downgrade - q.dg_link1
q["dg_link2"] = q.downgrade * q.l2.fillna(False).astype(float); q["dg_not2"] = q.downgrade - q.dg_link2

# controversy logged, no rating change in quarters q..q+4
sev["qi"] = qtr(sev.msci_date)
cq = sev[["parent_id", "qi"]].drop_duplicates()
chq = ch.copy(); chq["qi"] = qtr(chq.cdate); chset = set(zip(chq.parent_id, chq.qi))
cq["nochange"] = [all((p, k + j) not in chset for j in range(0, 5)) for p, k in zip(cq.parent_id, cq.qi)]
q = q.merge(cq[cq.nochange][["parent_id", "qi"]].assign(contro_nochg=1.0), how="left")
q = q.merge(cq[["parent_id", "qi"]].assign(contro_any=1.0), how="left")
q[["contro_nochg", "contro_any"]] = q[["contro_nochg", "contro_any"]].fillna(0.0)

# public-report date (hand-checked clock, verified, excl. 2026-03-23), US firms
H = pd.read_pickle(f"{W}/x3/hand3_m.pkl")
for c in ["MSCI date", "Audited earliest date"]:
    H[c] = pd.to_datetime(H[c], errors="coerce")
H = H[H["Audit status"].astype(str).str.startswith("Verified") & (H["MSCI date"] != "2026-03-23")]
print("verified rows:", len(H), "US:", int((H.Country == "US").sum()))
Hq = H[H.Country == "US"][["Parent ID", "Audited earliest date", "MSCI date"]].rename(columns={"Parent ID": "parent_id"})
Hq["qi"] = qtr(Hq["Audited earliest date"]); Hq["qm"] = qtr(Hq["MSCI date"])
q = q.merge(Hq[["parent_id", "qi"]].drop_duplicates().assign(public=1.0), how="left")
q = q.merge(Hq[["parent_id", "qm"]].drop_duplicates().rename(columns={"qm": "qi"}).assign(msci_hand=1.0), how="left")
q[["public", "msci_hand"]] = q[["public", "msci_hand"]].fillna(0.0)
nochg = lambda p, k: all((p, k + j) not in chset for j in range(0, 5))
q["public_nochg"] = q.public * np.array([nochg(p, k) for p, k in zip(q.parent_id, q.qi)], dtype=float)

q["date"] = q.qp.dt.start_time
q = q.join(with_ctrl(q, "date"))
q = q.set_index(["parent_id", "qi"], drop=False)
rows = []


def run(dep, hz, xs, lab, sample):
    s = q[dep]
    fut = s.reindex(pd.MultiIndex.from_arrays([q.parent_id, q.qi + hz])).values
    prev = s.reindex(pd.MultiIndex.from_arrays([q.parent_id, q.qi - 1])).values
    y = pd.Series(fut - prev, index=q.index)
    keep = sample
    X = pd.concat([q.loc[keep, xs + CTRL], fe(q.loc[keep, "qi"], "t")], axis=1)
    r, d = cl_ols(y[keep], X, q.loc[keep, "parent_id"])
    out = dict(spec=lab, funds=dep, h=hz, N=int(r.nobs), clusters=int(d.g.nunique()), mean_level=float(q.loc[keep, dep].mean()))
    for x in xs:
        out[x] = r.params.get(x, np.nan); out[x + "_t"] = r.tvalues.get(x, np.nan)
        out[x + "_n"] = int(d[x].sum()) if x in d else 0
    if "dg_link1" in xs or "dg_link2" in xs:
        a, b = ("dg_link1", "dg_not1") if "dg_link1" in xs else ("dg_link2", "dg_not2")
        w = r.t_test(f"{a} - {b} = 0"); out["diff"] = float(np.ravel(w.effect)[0]); out["diff_t"] = float(np.ravel(w.tvalue)[0]); out["diff_p"] = float(np.ravel(w.pvalue)[0])
    rows.append(out); return out


S18 = (q.qi >= 2018 * 4 + 1)
for dep in ["all", "idx", "oth"]:
    for hz in (1, 2, 4):
        run(dep, hz, ["dg_link1", "dg_not1", "upgrade"], "Def. 1: severe/very severe logged in prior 12 months", S18)
        run(dep, hz, ["dg_link2", "dg_not2", "upgrade"], "Def. 2: plus that issue's score fell", S18)
        run(dep, hz, ["contro_nochg", "downgrade", "upgrade"], "MSCI logging date, no rating change in q..q+4", S18)
        run(dep, hz, ["public", "msci_hand", "downgrade", "upgrade"], "Hand-checked clock: public-report and MSCI dates (US)", S18)
        run(dep, hz, ["public_nochg", "downgrade", "upgrade"], "Hand-checked clock: public-report date, no rating change in q..q+4", S18)
# replication check of Table 10 (all years, no linkage)
r10 = run("all", 4, ["downgrade", "upgrade"], "CHECK: Table 10 all years", q.qi > 0)
print("Table 10 check q+4:", round(r10["downgrade"], 4), round(r10["downgrade_t"], 2))
T = pd.DataFrame(rows); T.to_csv(f"{OUT}/A1_ownership_linked.csv", index=False)
pd.set_option("display.width", 250)
print(T.round(4).to_string())


**`consequences/section8_final.py`** — Section 7 real effects: ESG fund flows, MSCI-index vs other ESG funds (the 0.017%/0.03%/0.21% ownership figures).

In [ ]:
# ====================================================================================
# PIPELINE FILE: consequences/section8_final.py
# Section 7 real effects: ESG fund flows, MSCI-index vs other ESG funds (the 0.017%/0.03%/0.21% ownership figures).
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""
section8_final.py -- final analysis for the research draft (US & UK), using
  * the hand-audit files (670 hand-audit rows; 31 incidents verified with an earliest public date),
  * CRSP daily returns + Fama-French factors (daily event studies),
  * Compustat annual (controls), CRSP Mutual Fund ESG-mandated ownership (eq. 7 / eq. 8),
  * the kit data (MSCI ratings, controversy onsets, monthly market-adjusted returns).

Blocks
  L  logging lag on the hand-audited subsample (with the truncation caveat: rows were kept only with news 0-90 d)
  V  return-based validation of the news dates: CAR at the news date vs a placebo date 364 days earlier
  E  daily event study on MSCI's own dates: controversy logging dates by severity; letter downgrades/upgrades,
     downgrades with vs without a prior MSCI-logged controversy; cross-sectional regression with controls
  O  ESG-mandated ownership: eq. (7) CAR6 x MandateOwn and eq. (8) change in MandateOwn after downgrades
  H  eq. (9) downgrade hazard after controversy logging, with Compustat controls
"""
import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

LOG = []


def log(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); LOG.append(s)


def cl_ols(y, X, g):
    d = pd.concat([y.rename("y"), X, g.rename("g")], axis=1).dropna()
    Xc = sm.add_constant(d[X.columns], has_constant="add")
    Xc = Xc.loc[:, (Xc.std() > 0) | (Xc.columns == "const")]
    return sm.OLS(d.y, Xc).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.g.astype(str))[0]}), d


def coefs(r, terms, model, **kw):
    return [dict(model=model, term=t, coef=r.params[t], se=r.bse[t], t=r.tvalues[t], p=r.pvalues[t], N=int(r.nobs), **kw)
            for t in terms if t in r.params.index]


def fe(s, prefix):
    return pd.get_dummies(s.astype(str), prefix=prefix, drop_first=True).astype(float)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--kit", required=True); ap.add_argument("--wrds", required=True); ap.add_argument("--daily", required=True)
    ap.add_argument("--hand", required=True); ap.add_argument("--contro", required=True); ap.add_argument("--out", required=True)
    a = ap.parse_args()
    kit, W, out = Path(a.kit), Path(a.wrds), Path(a.out); out.mkdir(parents=True, exist_ok=True)
    sys.path.insert(0, str(kit)); sys.path.insert(0, str(Path(__file__).resolve().parent))
    import phase2_analysis as P
    import es_engine as E
    T = {}

    # ------------------------------------------------------------------ data
    par = pd.read_csv(kit / "data/parents.csv", dtype={"parent_id": str}, low_memory=False).drop_duplicates("parent_id")
    par["country"] = par.country.replace({"GB": "UK"})
    R = P.Ratings(kit / "data/ratings.parquet")
    ons = pd.read_csv(kit / "data/onsets.csv", dtype={"parent_id": str}, parse_dates=["msci_date"])
    ons = ons[~(ons.flag_backfill_20190701.astype(bool) | ons.flag_floor_20170831.astype(bool))].copy()
    ons["msci_date"] = ons.msci_date.astype("datetime64[ns]")
    ons = ons.merge(par[["parent_id", "country", "sector"]], on="parent_id")
    ons = ons[ons.country.isin(["US", "UK"])]
    ons["sev"] = ons.severity.map(P.SEV_CODE)
    H = pd.read_pickle(a.hand); Cn = pd.read_pickle(a.contro)
    H = H.rename(columns={"Parent ID": "parent_id", "Onset ID": "onset_id", "MSCI date": "msci_date", "News date": "news_date"})
    H["lag"] = (H.msci_date - H.news_date).dt.days
    H["sev"] = H["Severity rank (1–3)"].astype(int)
    V = Cn[Cn["Correct incident?"] == "Y"].copy()
    V = V.rename(columns={"Parent ID": "parent_id", "Onset ID": "onset_id", "MSCI date": "msci_date"})
    V["news_date"] = pd.to_datetime(V["Earliest public date"]); V["msci_date"] = pd.to_datetime(V.msci_date)
    V["lag"] = (V.msci_date - V.news_date).dt.days
    V["sev"] = V.Severity.map(P.SEV_CODE)
    comp = pd.read_parquet(W / "compustat_annual.parquet")
    comp["datadate"] = pd.to_datetime(comp.datadate).astype("datetime64[ns]"); comp["parent_id"] = comp.parent_id.astype(str)
    for c in ["at", "ceq", "lt", "ni", "csho", "prcc_f", "dltt", "dlc"]:
        comp[c] = pd.to_numeric(comp[c], errors="coerce")
    comp["log_at"] = np.log(comp["at"].where(comp["at"] > 0))
    comp["lev"] = ((comp.dltt.fillna(0) + comp.dlc.fillna(0)) / comp["at"]).where(comp["at"] > 0).clip(0, 2)
    comp["roa"] = (comp.ni / comp["at"]).where(comp["at"] > 0).clip(-1, 1)
    me = comp.csho * comp.prcc_f
    comp["bm"] = np.log((comp.ceq / me).where((comp.ceq > 0) & (me > 0)))
    comp = comp.sort_values("datadate")
    CTRL = ["log_at", "lev", "roa", "bm"]

    def with_ctrl(df, dcol):
        """Latest Compustat fiscal year ending at least 3 months before the date."""
        x = df.reset_index().rename(columns={"index": "_i"})
        x["_d"] = (pd.to_datetime(x[dcol]) - pd.DateOffset(months=3)).astype("datetime64[ns]")
        m = pd.merge_asof(x.sort_values("_d"), comp[["parent_id", "datadate"] + CTRL].dropna(subset=["log_at"]),
                          left_on="_d", right_on="datadate", by="parent_id", direction="backward",
                          tolerance=pd.Timedelta(days=550))
        return m.set_index("_i").sort_index().drop(columns=["_d", "datadate"])

    mo = pd.read_csv(W.parent / "crsp_ownership/mandate_ownership.csv", dtype={"parent_id": str})
    lo_, hi_ = mo.mandate_own.quantile([0.01, 0.99])
    mo["mown"] = mo.mandate_own.clip(lo_, hi_) * 100                  # percent of market cap
    log(f"ESG-mandated ownership: {mo.parent_id.nunique():,} firms; median {mo.mown.median():.3f}% of market cap, "
        f"p90 {mo.mown.quantile(.9):.3f}%")

    D = E.Daily(pd.read_parquet(a.daily), pd.read_parquet(W / "ff_daily.parquet"), pd.read_parquet(W / "crsp_market_daily.parquet"))
    log(f"CRSP daily: {len(D.pids):,} firms with a primary US-listed security, {str(D.days[0])[:10]} to {str(D.days[-1])[:10]}")
    WIN = [(-1, 1), (0, 5), (-5, 5), (0, 20)]

    # ================================================================== L: logging lag
    log("\n=== L. Logging lag (MSCI date minus news date), hand-audited subsample ===")

    def lagstats(d, lab):
        return dict(sample=lab, N=len(d), firms=d.parent_id.nunique(), mean=d.lag.mean(), median=d.lag.median(),
                    p25=d.lag.quantile(.25), p75=d.lag.quantile(.75), share_le30=(d.lag <= 30).mean(),
                    share_negative=(d.lag < 0).mean())
    L = [lagstats(V, "Verified incidents (web audit, earliest public date)")]
    for s, g in V.groupby("Severity"):
        L.append(lagstats(g, f"  verified: {s}"))
    L.append(lagstats(H, "Hand-audit file (670; news 0-90 d by construction)"))
    for s, g in H.groupby("Severity"):
        L.append(lagstats(g, f"  670: {s}"))
    for s, g in H.groupby("Period"):
        L.append(lagstats(g, f"  670: {s}"))
    for s, g in H.groupby("Country"):
        L.append(lagstats(g, f"  670: {s}"))
    L.append(lagstats(H[H.Tier == "A"], "670, tier A only (clean onsets)"))
    L.append(lagstats(H[H["Date source"] != "GDELT dated article"], "670, date from a hand-checked/alternative article"))
    T["L1_logging_lag"] = pd.DataFrame(L)
    log(T["L1_logging_lag"].round(2).to_string(index=False))
    rows = []
    for lab, d in [("670 all", H), ("670 tier A", H[H.Tier == "A"]), ("670 excl. 2020", H[H.Period != "2020 (COVID)"])]:
        X = pd.DataFrame({"severe": (d.sev == 2).astype(float), "very_severe": (d.sev == 3).astype(float),
                          "y2020": (d.Period == "2020 (COVID)").astype(float), "early_2018_19": (d.Period == "Early 2018-19").astype(float),
                          "UK": (d.Country == "GB").astype(float)}, index=d.index)
        X = X.loc[:, X.std() > 0]
        r, _ = cl_ols(d.lag.astype(float), X, d.parent_id)
        rows += coefs(r, list(X.columns), f"lag ~ severity + period + UK [{lab}]")
    T["L2_lag_regression"] = pd.DataFrame(rows)
    log(T["L2_lag_regression"].round(3).to_string(index=False))

    # ================================================================== V: return validation of the news dates
    log("\n=== V. Do the news dates carry a stock-price reaction? (US-listed, daily 4-factor CARs) ===")
    vr = []
    for lab, d in [("670 hand-audit file", H), ("670, tier A", H[H.Tier == "A"]),
                   ("670, hand-checked/alternative article", H[H["Date source"] != "GDELT dated article"]),
                   ("Verified incidents", V)]:
        d = d.reset_index(drop=True)
        cn = D.car(d.parent_id.values, d.news_date, WIN)
        cp = D.car(d.parent_id.values, d.news_date - pd.Timedelta(days=364), WIN)
        cm = D.car(d.parent_id.values, d.msci_date, WIN)
        for w in ["car_-1_1", "car_0_5", "car_-5_5"]:
            for kind, c in [("news date", cn), ("placebo (news - 364 d)", cp), ("MSCI date", cm)]:
                vr.append(dict(sample=lab, window=w, date=kind, **E.mean_test(c[w], d.parent_id, d.news_date.astype(str))))
            diff = cn[w] - cp[w]
            vr.append(dict(sample=lab, window=w, date="news minus placebo (paired)", **E.mean_test(diff, d.parent_id, d.news_date.astype(str))))
    T["V1_news_date_validation"] = pd.DataFrame(vr)
    log(T["V1_news_date_validation"][["sample", "window", "date", "N", "mean", "t", "share_neg"]].round(4).to_string(index=False))

    # ================================================================== E: daily event study on MSCI dates
    log("\n=== E1. CAR around MSCI controversy logging dates (all primary US/UK onsets with daily returns) ===")
    o = ons[ons.parent_id.isin(D.pix) & (ons.msci_date >= "2013-02-01")].reset_index(drop=True)
    co = D.car(o.parent_id.values, o.msci_date, WIN)
    o = pd.concat([o, co], axis=1)
    e1 = []
    for s, g in [("all", o)] + list(o.groupby("severity")):
        for w in ["car_-1_1", "car_0_5", "car_-5_5", "car_0_20", "carm_-1_1", "carm_0_5"]:
            e1.append(dict(event="controversy logged by MSCI", severity=s, window=w,
                           **E.mean_test(g[w], g.parent_id, g.msci_date.astype(str))))
    T["E1_CAR_controversy_logging"] = pd.DataFrame(e1)
    log(T["E1_CAR_controversy_logging"][["severity", "window", "N", "mean", "t", "share_neg"]].round(4).to_string(index=False))

    log("\n=== E2. CAR around MSCI letter-rating changes ===")
    ch = R.changes.copy()
    ch = ch[ch.parent_id.isin(D.pix) & (ch.cdate >= "2013-02-01")].reset_index(drop=True)
    # prior MSCI-logged controversy within 365 days before the change (any severity / severe or worse)
    oo = ons[["parent_id", "msci_date", "sev"]].sort_values("msci_date")
    x = ch.reset_index().rename(columns={"index": "_i"}).sort_values("cdate")
    for lab, sub in [("prior_any", oo), ("prior_severe", oo[oo.sev >= 2])]:
        mm = pd.merge_asof(x[["_i", "parent_id", "cdate"]].sort_values("cdate"),
                           sub.rename(columns={"msci_date": "pd_"})[["parent_id", "pd_"]], left_on="cdate", right_on="pd_",
                           by="parent_id", direction="backward", tolerance=pd.Timedelta(days=365))
        ch[lab] = mm.set_index("_i").pd_.reindex(ch.index).notna().astype(float).values
    cc = D.car(ch.parent_id.values, ch.cdate, WIN)
    ch = pd.concat([ch, cc], axis=1)
    e2 = []
    groups = [("downgrade", ch[ch.down]), ("upgrade", ch[~ch.down]),
              ("downgrade, MSCI controversy logged in prior 12 m", ch[ch.down & (ch.prior_any == 1)]),
              ("downgrade, no controversy in prior 12 m", ch[ch.down & (ch.prior_any == 0)]),
              ("downgrade, severe/very severe controversy in prior 12 m", ch[ch.down & (ch.prior_severe == 1)])]
    for lab, g in groups:
        for w in ["car_-1_1", "car_0_5", "car_-5_5", "car_0_20", "carm_-1_1", "carm_0_5"]:
            e2.append(dict(event=lab, window=w, **E.mean_test(g[w], g.parent_id, g.cdate.astype(str))))
    T["E2_CAR_rating_changes"] = pd.DataFrame(e2)
    log(T["E2_CAR_rating_changes"][["event", "window", "N", "mean", "t", "share_neg"]].round(4).to_string(index=False))

    log("\n=== E3. Downgrade CARs: cross-section with controls and ESG-mandated ownership ===")
    dg = ch[ch.down].copy()
    dg = dg.join(with_ctrl(dg, "cdate")[CTRL])
    dg["midx"] = dg.cdate.dt.year * 12 + dg.cdate.dt.month
    dg = dg.merge(mo[["parent_id", "midx", "mown"]], on=["parent_id", "midx"], how="left")
    dg["year"] = dg.cdate.dt.year
    e3 = []
    for y in ["car_-1_1", "car_0_5", "car_0_20"]:
        for lab, xs in [("controls", ["prior_any"] + CTRL), ("controls + ESG ownership", ["prior_any", "mown"] + CTRL)]:
            X = pd.concat([dg[xs], fe(dg.year, "y")], axis=1)
            r, d = cl_ols(dg[y], X, dg.parent_id)
            e3 += coefs(r, xs, f"{y} ~ {lab} + year FE")
    T["E3_downgrade_CAR_regression"] = pd.DataFrame(e3)
    log(T["E3_downgrade_CAR_regression"].round(4).to_string(index=False))

    # ================================================================== O: ESG-mandated ownership
    log("\n=== O1. Eq.(7): 6-month CAR after downgrades x ESG-mandated ownership (monthly, US) ===")
    ret = pd.read_parquet(kit / "data/returns.parquet"); ret["parent_id"] = ret.parent_id.astype(str)
    ret["ra"] = P.winsor(ret.ra)
    ret = ret.sort_values(["parent_id", "midx"])
    g = ret.groupby("parent_id")
    ret["car_f6"] = sum(g.ra.shift(-k) for k in range(1, 7))
    ret["car_f6"] = ret.car_f6.where(g.midx.shift(-6) == ret.midx + 6)
    chm = R.changes.copy(); chm["midx"] = chm.cdate.dt.year * 12 + chm.cdate.dt.month
    Em = ret.merge(mo[["parent_id", "midx", "mown"]], on=["parent_id", "midx"], how="inner")
    Em = Em.merge(chm[chm.down][["parent_id", "midx"]].drop_duplicates().assign(downgrade=1.0), how="left")
    Em = Em.merge(chm[~chm.down][["parent_id", "midx"]].drop_duplicates().assign(upgrade=1.0), how="left")
    om_ = ons[["parent_id", "msci_date"]].assign(o_m=lambda d: d.msci_date.dt.year * 12 + d.msci_date.dt.month)
    pr = om_[["parent_id", "o_m"]].merge(Em[["parent_id", "midx"]].drop_duplicates(), on="parent_id")
    pr = pr[(pr.midx - pr.o_m).between(0, 12)][["parent_id", "midx"]].drop_duplicates().assign(known12=1.0)
    Em = Em.merge(pr, how="left").fillna({"downgrade": 0.0, "upgrade": 0.0, "known12": 0.0})
    Em["date"] = pd.to_datetime(dict(year=Em.midx // 12 - (Em.midx % 12 == 0), month=(Em.midx - 1) % 12 + 1, day=1))
    Em = Em.join(with_ctrl(Em, "date")[CTRL])
    Em["mown_z"] = (Em.mown - Em.mown.mean()) / Em.mown.std()
    Em["dg_x_mown"] = Em.downgrade * Em.mown_z
    Em["dg_x_known"] = Em.downgrade * Em.known12
    o1 = []
    for lab, xs in [("base", ["downgrade", "upgrade", "mown_z", "dg_x_mown", "known12", "dg_x_known"]),
                    ("with controls", ["downgrade", "upgrade", "mown_z", "dg_x_mown", "known12", "dg_x_known"] + CTRL)]:
        X = pd.concat([Em[xs], fe(Em.midx, "t")], axis=1)
        r, d = cl_ols(Em.car_f6, X, Em.parent_id)
        o1 += coefs(r, xs, f"CAR6 ~ eq.(7) [{lab}] + month FE")
    T["O1_eq7_CAR6_mandate_ownership"] = pd.DataFrame(o1)
    log(T["O1_eq7_CAR6_mandate_ownership"].round(4).to_string(index=False))

    log("\n=== O2. Eq.(8): change in ESG-mandated ownership after downgrades (quarterly, US) ===")
    q = pd.read_csv(W.parent / "crsp_ownership/mandate_ownership_quarterly.csv", dtype={"parent_id": str})
    q["mown"] = q.mandate_own.clip(lo_, hi_) * 100
    q["qp"] = pd.PeriodIndex(q.q, freq="Q"); q = q.sort_values(["parent_id", "qp"])
    q["qi"] = q.qp.dt.year * 4 + q.qp.dt.quarter
    chq = R.changes.copy(); chq["qi"] = chq.cdate.dt.year * 4 + chq.cdate.dt.quarter
    q = q.merge(chq[chq.down][["parent_id", "qi"]].drop_duplicates().assign(downgrade=1.0), how="left")
    q = q.merge(chq[~chq.down][["parent_id", "qi"]].drop_duplicates().assign(upgrade=1.0), how="left")
    q = q.fillna({"downgrade": 0.0, "upgrade": 0.0})
    qq = q.set_index(["parent_id", "qi"]).mown
    q["date"] = q.qp.dt.start_time
    q = q.join(with_ctrl(q, "date")[CTRL])
    o2 = []
    for h in (2, 4, 8):
        fut = qq.reindex(pd.MultiIndex.from_arrays([q.parent_id, q.qi + h])).values
        prev = qq.reindex(pd.MultiIndex.from_arrays([q.parent_id, q.qi - 1])).values
        q[f"d_mown_{h}"] = fut - prev
        for lab, xs in [("base", ["downgrade", "upgrade"]), ("with controls", ["downgrade", "upgrade"] + CTRL)]:
            X = pd.concat([q[xs], fe(q.qi, "q")], axis=1)
            r, d = cl_ols(q[f"d_mown_{h}"], X, q.parent_id)
            o2 += coefs(r, xs, f"change in ESG ownership (pp) from q-1 to q+{h} ~ [{lab}] + quarter FE",
                        mean_dep=float(d.y.mean()), mean_level=float(q.mown.mean()))
    T["O2_eq8_change_in_mandate_ownership"] = pd.DataFrame(o2)
    log(T["O2_eq8_change_in_mandate_ownership"].round(4).to_string(index=False))
    # event-time path of ownership around downgrades (relative to quarter -1)
    path = []
    ev = q[q.downgrade == 1][["parent_id", "qi"]]
    for k in range(-4, 9):
        v = qq.reindex(pd.MultiIndex.from_arrays([ev.parent_id, ev.qi + k])).values
        b = qq.reindex(pd.MultiIndex.from_arrays([ev.parent_id, ev.qi - 1])).values
        dlt = v - b
        path.append(dict(quarter_rel_to_downgrade=k, N=int(np.isfinite(dlt).sum()), mean_change_pp=np.nanmean(dlt),
                         median_change_pp=np.nanmedian(dlt)))
    T["O3_ownership_path_around_downgrades"] = pd.DataFrame(path)
    log(T["O3_ownership_path_around_downgrades"].round(4).to_string(index=False))

    # ================================================================== H: hazard with controls
    log("\n=== H. Eq.(9) hazard of a downgrade after MSCI logging, with Compustat controls ===")
    m = ons.reset_index(drop=True).copy()
    m["spell_id"] = m.onset_id
    m = m.join(R.asof(m, "msci_date")).dropna(subset=["band"])
    m = m.join(with_ctrl(m, "msci_date")[CTRL])
    m["cal_month"] = m.msci_date.dt.month
    m["severe"] = (m.sev == 2).astype(float); m["very_severe"] = (m.sev == 3).astype(float)
    m["us"] = m.country == "US"
    hz = []
    for kind in ["down", "any"]:
        sp, one = P.build_spells(m, "msci_date", R, kind)
        for s_lab, s_df in [("US+UK", sp), ("US", sp[sp.us])]:
            for c_lab, xc in [("no controls, same sample", ["severe", "very_severe", "dist_threshold"]),
                              ("Compustat controls", ["severe", "very_severe", "dist_threshold", "log_at", "lev", "roa"])]:
                s2 = s_df.dropna(subset=["log_at", "lev", "roa"])
                res, rows = P.fit_hazard(s2, xc, ["sector", "cal_month"], "parent_id", f"{kind}|{s_lab}|{c_lab}")
                hz += [dict(r, event=kind, sample=s_lab, spec=c_lab) for r in rows
                       if r["term"] in ["severe", "very_severe", "dist_threshold", "log_at", "lev", "roa"]]
    T["H1_hazard_with_controls"] = pd.DataFrame(hz)
    log(T["H1_hazard_with_controls"].round(3).to_string(index=False))

    for k, v in T.items():
        v.to_csv(out / f"{k}.csv", index=False)
    (out / "FINAL_LOG.txt").write_text("\n".join(LOG), encoding="utf-8")
    pd.to_pickle(T, out / "tables.pkl")
    log(f"\nDONE -> {out}")


if __name__ == "__main__":
    main()


**`consequences/section8_extra.py`** — Abnormal volume & liquidity around downgrades (Tables 11–12).

In [ ]:
# ====================================================================================
# PIPELINE FILE: consequences/section8_extra.py
# Abnormal volume & liquidity around downgrades (Tables 11–12).
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""
section8_extra.py -- why is there no price reaction at MSCI rating changes?

  I  Index-review channel. ESG index funds (MSCI KLD 400, ESG Leaders/Select, SRI ...) trade a downgraded stock at the
     next MSCI index review, not on the rating date. Reviews take effect at the close of the last business day of
     February, May, August and November (announced about two weeks earlier).
     I1  CAR around review days for firms downgraded / upgraded since the previous review vs all other firm-reviews
     I2  change in ownership by MSCI-ESG index funds vs other ESG funds after downgrades (who sells, and when)
  V  Trading volume and liquidity around downgrades and controversy logging (needs crsp_daily_liq.parquet from
     wrds_volume.py): abnormal turnover, Amihud illiquidity, quoted spread, days -1..+20 vs the firm's own -60..-11 days.
"""
import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

LOG = []


def log(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); LOG.append(s)


def cl_ols(y, X, g):
    d = pd.concat([y.rename("y"), X, g.rename("g")], axis=1).dropna()
    Xc = sm.add_constant(d[X.columns], has_constant="add")
    Xc = Xc.loc[:, (Xc.std() > 0) | (Xc.columns == "const")]
    return sm.OLS(d.y, Xc).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.g.astype(str))[0]}), d


def coefs(r, terms, model, **kw):
    return [dict(model=model, term=t, coef=r.params[t], se=r.bse[t], t=r.tvalues[t], p=r.pvalues[t], N=int(r.nobs), **kw)
            for t in terms if t in r.params.index]


def fe(s, p):
    return pd.get_dummies(s.astype(str), prefix=p, drop_first=True).astype(float)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--kit", required=True); ap.add_argument("--wrds", required=True); ap.add_argument("--daily", required=True)
    ap.add_argument("--liq", default=None, help="crsp_daily_liq.parquet from wrds_volume.py (optional)")
    ap.add_argument("--out", required=True)
    a = ap.parse_args()
    kit, W, out = Path(a.kit), Path(a.wrds), Path(a.out); out.mkdir(parents=True, exist_ok=True)
    sys.path.insert(0, str(kit)); sys.path.insert(0, str(Path(__file__).resolve().parent))
    import phase2_analysis as P
    import es_engine as E
    T = {}
    R = P.Ratings(kit / "data/ratings.parquet")
    ons = pd.read_csv(kit / "data/onsets.csv", dtype={"parent_id": str}, parse_dates=["msci_date"])
    ons = ons[~(ons.flag_backfill_20190701.astype(bool) | ons.flag_floor_20170831.astype(bool))]
    D = E.Daily(pd.read_parquet(a.daily), pd.read_parquet(W / "ff_daily.parquet"), pd.read_parquet(W / "crsp_market_daily.parquet"))
    days = pd.Series(pd.to_datetime(D.days))

    # ------------------------------------------------------------------ I1 review-day CARs
    log("=== I1. CAR around MSCI index review days: firms downgraded/upgraded since the previous review vs others ===")
    rev = days.groupby([days.dt.year, days.dt.month]).max()
    rev = rev[rev.dt.month.isin([2, 5, 8, 11]) & (rev >= "2013-05-01")].sort_values().reset_index(drop=True)
    cut = pd.Series([days[max(0, int(np.searchsorted(days.values, r.to_datetime64())) - 10)] for r in rev])  # ~announcement
    prev_cut = cut.shift(1)
    ch = R.changes[R.changes.parent_id.isin(D.pix)].copy()
    rows = []
    for i in range(1, len(rev)):
        win = ch[(ch.cdate > prev_cut[i]) & (ch.cdate <= cut[i])]
        dn = set(win[win.down].parent_id); up = set(win[~win.down].parent_id)
        lo = ons[(ons.msci_date > prev_cut[i]) & (ons.msci_date <= cut[i])]
        for p in D.pids:
            rows.append((p, rev[i], p in dn, p in up, p in set(lo.parent_id)))
    pr = pd.DataFrame(rows, columns=["parent_id", "review", "downgraded", "upgraded", "controversy"])
    # keep firms that had a rating at the review
    first = R.r.groupby("parent_id").date.min()
    pr = pr[pr.review > pr.parent_id.map(first)].reset_index(drop=True)
    log(f"   {len(rev) - 1} reviews; {len(pr):,} firm-reviews; downgraded {int(pr.downgraded.sum()):,}, upgraded {int(pr.upgraded.sum()):,}")
    WIN = [(-10, -1), (0, 0), (-10, 0), (1, 5), (-10, 5)]
    c = D.car(pr.parent_id.values, pr.review, WIN)
    pr = pd.concat([pr, c], axis=1)
    mo = pd.read_csv(W.parent / "crsp_ownership/mandate_ownership.csv", dtype={"parent_id": str})
    mo["held_by_esg_index_funds"] = (mo.mandate_own_index > 0).astype(float)
    pr["midx"] = pr.review.dt.year * 12 + pr.review.dt.month
    pr = pr.merge(mo[["parent_id", "midx", "held_by_esg_index_funds"]], on=["parent_id", "midx"], how="left")
    pr["held_by_esg_index_funds"] = pr.held_by_esg_index_funds.fillna(0)
    i1 = []
    for g_lab, sel in [("downgraded since last review", pr.downgraded), ("upgraded since last review", pr.upgraded),
                       ("downgraded & held by MSCI-ESG index funds", pr.downgraded & (pr.held_by_esg_index_funds == 1)),
                       ("no rating change", ~pr.downgraded & ~pr.upgraded)]:
        g = pr[sel]
        for w in ["car_-10_-1", "car_0_0", "car_-10_0", "car_1_5", "car_-10_5"]:
            i1.append(dict(group=g_lab, window=w, **E.mean_test(g[w], g.parent_id, g.review.astype(str))))
    T["I1_review_day_CAR_means"] = pd.DataFrame(i1)
    log(T["I1_review_day_CAR_means"][["group", "window", "N", "mean", "t"]].round(4).to_string(index=False))
    pr["dg_x_held"] = pr.downgraded.astype(float) * pr.held_by_esg_index_funds
    pr["up_x_held"] = pr.upgraded.astype(float) * pr.held_by_esg_index_funds
    reg = []
    xs = ["downgraded", "upgraded", "controversy", "held_by_esg_index_funds", "dg_x_held", "up_x_held"]
    for w in ["car_-10_0", "car_1_5", "car_-10_5"]:
        X = pd.concat([pr[xs].astype(float), fe(pr.review.dt.strftime("%Y%m"), "r")], axis=1)
        r, _ = cl_ols(pr[w], X, pr.parent_id)
        reg += coefs(r, xs, f"{w} ~ rating change since last review + review FE")
    T["I1b_review_day_CAR_regression"] = pd.DataFrame(reg)
    log(T["I1b_review_day_CAR_regression"].round(4).to_string(index=False))

    # ------------------------------------------------------------------ I2 who sells: index ESG funds vs other ESG funds
    log("\n=== I2. Ownership change after downgrades: MSCI-ESG index funds vs other ESG funds (quarterly, pp of market cap) ===")
    q = pd.read_csv(W.parent / "crsp_ownership/mandate_ownership_quarterly.csv", dtype={"parent_id": str})
    lo_, hi_ = q.mandate_own.quantile([0.01, 0.99])
    q["idx"] = q.mandate_own_index.clip(0, hi_) * 100
    q["oth"] = (q.mandate_own - q.mandate_own_index).clip(0, hi_) * 100
    q["qp"] = pd.PeriodIndex(q.q, freq="Q"); q["qi"] = q.qp.dt.year * 4 + q.qp.dt.quarter
    chq = R.changes.copy(); chq["qi"] = chq.cdate.dt.year * 4 + chq.cdate.dt.quarter
    q = q.merge(chq[chq.down][["parent_id", "qi"]].drop_duplicates().assign(downgrade=1.0), how="left")
    q = q.merge(chq[~chq.down][["parent_id", "qi"]].drop_duplicates().assign(upgrade=1.0), how="left").fillna({"downgrade": 0.0, "upgrade": 0.0})
    i2 = []
    for col, lab in [("idx", "MSCI-ESG index funds"), ("oth", "other ESG-named funds")]:
        s = q.set_index(["parent_id", "qi"])[col]
        base = s.reindex(pd.MultiIndex.from_arrays([q.parent_id, q.qi - 1])).values
        for h in (0, 1, 2, 4):
            q["dy"] = s.reindex(pd.MultiIndex.from_arrays([q.parent_id, q.qi + h])).values - base
            X = pd.concat([q[["downgrade", "upgrade"]], fe(q.qi, "q")], axis=1)
            r, d = cl_ols(q.dy, X, q.parent_id)
            i2 += coefs(r, ["downgrade", "upgrade"], f"{lab}: change q-1 to q+{h}", mean_level=float(q[col].mean()))
    T["I2_index_vs_other_ESG_funds"] = pd.DataFrame(i2)
    log(T["I2_index_vs_other_ESG_funds"].round(5).to_string(index=False))

    # ------------------------------------------------------------------ V volume / liquidity
    if a.liq and Path(a.liq).exists():
        log("\n=== V. Abnormal trading volume and liquidity around downgrades and controversy logging ===")
        L = pd.read_parquet(a.liq)
        L["date"] = pd.to_datetime(L.date).astype("datetime64[ns]")
        L = L[L.permno.isin(set(pd.read_parquet(a.daily, columns=["permno"]).permno))]
        for c_ in ["vol", "shrout", "prc", "bid", "ask", "ret"]:
            if c_ in L:
                L[c_] = pd.to_numeric(L[c_], errors="coerce")
        L["prc"] = L.prc.abs()
        L["turn"] = L.vol / (L.shrout * 1000.0)                       # shares traded / shares outstanding
        L["amihud"] = (L.ret.abs() / (L.vol * L.prc)).replace([np.inf, -np.inf], np.nan) * 1e6
        L["spread"] = ((L.ask - L.bid) / ((L.ask + L.bid) / 2)).where((L.ask > 0) & (L.bid > 0) & (L.ask >= L.bid))
        # one security per parent (as in the return tests)
        prim = pd.Series({p: None for p in D.pids})
        dd = pd.read_parquet(a.daily, columns=["parent_id", "permno", "mcap_k"])
        prim = dd.groupby(["parent_id", "permno"]).mcap_k.mean().reset_index().sort_values("mcap_k", ascending=False).drop_duplicates("parent_id")
        L = L.merge(prim[["parent_id", "permno"]], on="permno")
        dix = {d: i for i, d in enumerate(D.days)}
        M = {}
        for v in ["turn", "amihud", "spread"]:
            A = np.full(D.R.shape, np.nan)
            ok = L[v].notna() & L.date.isin(dix.keys())
            A[L.loc[ok, "parent_id"].map(D.pix).values, L.loc[ok, "date"].map(dix).values] = np.log(L.loc[ok, v].values + (1e-6 if v == "turn" else 1e-9))
            M[v] = A
        def abn(pids, dates, v, w=(0, 5)):
            t0 = D.day_index(dates); res = np.full(len(pids), np.nan)
            for k, (p, t) in enumerate(zip(pids, t0)):
                i = D.pix.get(p)
                if i is None or t - 60 < 0 or t + w[1] >= M[v].shape[1]:
                    continue
                base = M[v][i, t - 60:t - 10]; ev = M[v][i, t + w[0]:t + w[1] + 1]
                if np.isfinite(base).sum() >= 20 and np.isfinite(ev).sum() >= 1:
                    res[k] = np.nanmean(ev) - np.nanmean(base)
            return res
        vr = []
        chd = R.changes[R.changes.parent_id.isin(D.pix)]
        evs = [("downgrade", chd[chd.down].parent_id.values, chd[chd.down].cdate),
               ("upgrade", chd[~chd.down].parent_id.values, chd[~chd.down].cdate)]
        oo = ons[ons.parent_id.isin(D.pix)]
        for s in ["moderate", "severe", "very_severe"]:
            g = oo[oo.severity == s]; evs.append((f"controversy logged: {s}", g.parent_id.values, g.msci_date))
        evs.append(("index review: downgraded since last review", pr[pr.downgraded].parent_id.values, pr[pr.downgraded].review))
        evs.append(("index review: no rating change", pr[~pr.downgraded & ~pr.upgraded].parent_id.values, pr[~pr.downgraded & ~pr.upgraded].review))
        for lab, pids, dts in evs:
            for v in ["turn", "amihud", "spread"]:
                for w in [(-1, 1), (0, 5), (0, 20)]:
                    x = abn(pids, pd.to_datetime(dts), v, w)
                    vr.append(dict(event=lab, measure=f"log {v}", window=f"[{w[0]},{w[1]}]",
                                   **E.mean_test(x, pids, pd.to_datetime(dts).astype(str).values)))
        T["V1_abnormal_volume_liquidity"] = pd.DataFrame(vr)
        log(T["V1_abnormal_volume_liquidity"][["event", "measure", "window", "N", "mean", "t"]].round(4).to_string(index=False))
    else:
        log("\n(V skipped: run wrds_volume.py and pass --liq crsp_daily_liq.parquet)")

    for k, v in T.items():
        v.to_csv(out / f"{k}.csv", index=False)
    (out / "EXTRA_LOG.txt").write_text("\n".join(LOG), encoding="utf-8")
    pd.to_pickle(T, out / "tables_extra.pkl")


if __name__ == "__main__":
    main()


**`pillar_forward_backward/pillar_forward_ownership.py`** — Table 8: pillar-level (E/S/G) incorporation and ownership.

In [ ]:
# ====================================================================================
# PIPELINE FILE: pillar_forward_backward/pillar_forward_ownership.py
# Table 8: pillar-level (E/S/G) incorporation and ownership.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""
pillar_forward_ownership.py — three analyses added for the E/S/G pillar split and the
forward- vs backward-looking question:

  (1) PILLAR split of the severity-graded incorporation (Table 2 spec: binding measure,
      year FE, ultimate-parent clustering, 2018-2025 excl. 2020, first-coverage-robust).
  (2) FORWARD-LOOKING test: does a firm's ESG pillar SCORE at t predict a severe
      controversy in that pillar at t+1, controlling for the current controversy?
  (3) OWNERSHIP WEIGHT: ESG-mandated mutual-fund ownership as a share of shares outstanding.

Inputs (licensed, not shipped):
  evt/panelA_vt_july.parquet          per (issuer, issue, year) valid-time mod/sev/vsev counts
  w2/panel_fy_validtime.parquet       issuer-year panel with dias, py, group
  w2/jul/HIST_CONTROVERSIES_*.parquet, HIST_SUMMARY_SCORES_*.parquet
  w2/crsp_ownership/mandate_ownership_quarterly.csv

Run:  python pillar_forward_ownership.py       (writes P1_*.csv, P2_*.csv, P3_*.csv + an Excel)
"""
import os, numpy as np, pandas as pd, duckdb, statsmodels.api as sm

EVT = os.environ.get("EVT", "evt"); W = os.environ.get("W", "w2")
SS = f"{W}/jul/HIST_SUMMARY_SCORES_20260731.parquet"; CF = f"{W}/jul/HIST_CONTROVERSIES_20260731.parquet"
PILLAR = {**{k: "E" for k in ["BIODIV_LAND_USE", "E_WASTE", "FINANCING_ENV_IMP", "PACK_MAT_WASTE",
                              "RAW_MAT_SRC", "TOXIC_EMISS", "WATER_STRESS"]},
          **{k: "G" for k in ["BUS_ETHICS_ANTICOMP", "BUS_ETHICS_CORRUPTION", "BUS_ETHICS_ETHICSFRAUD"]}}
pillar_of = lambda s: PILLAR.get(s, "S")
con = duckdb.connect(); con.sql("PRAGMA threads=4")

def clustered(d, y, xs, cl="group"):
    d = d.dropna(subset=[y] + xs + [cl]).copy(); d["yr"] = d["yr"].astype(int); yrs = sorted(d["yr"].unique())
    X = np.column_stack([np.ones(len(d))] + [(d["yr"] == yy).astype(float) for yy in yrs[1:]] + [d[x].values for x in xs])
    Y = d[y].values; XtXi = np.linalg.inv(X.T @ X); b = XtXi @ (X.T @ Y); e = Y - X @ b
    S = pd.DataFrame(X * e[:, None]).groupby(d[cl].values).sum().values
    N, k = X.shape; G = S.shape[0]; c = (G / (G - 1)) * ((N - 1) / (N - k))
    se = np.sqrt(np.diag(c * (XtXi @ (S.T @ S) @ XtXi))); p = len(xs)
    return {x: (round(b[-p + i], 4), round(b[-p + i] / se[-p + i], 1)) for i, x in enumerate(xs)}, len(d), G

# ============== (1) PILLAR split of incorporation ==============
A = pd.read_parquet(f"{EVT}/panelA_vt_july.parquet"); A["pillar"] = A["issue"].map(pillar_of)
A["lvl"] = np.where(A["vsev"] > 0, "vs", np.where(A["sev"] > 0, "sev", np.where(A["mod"] > 0, "mod", "none")))
def bcount(level, pre):
    b = A[A["lvl"] == level].groupby(["ISSUERID", "yr", "pillar"]).size().unstack("pillar", fill_value=0).reset_index()
    b.columns = ["ISSUERID", "yr"] + [f"{pre}_{c}" for c in b.columns[2:]]; return b
bs, bv = bcount("sev", "bsev"), bcount("vs", "bvs")
P = pd.read_parquet(f"{W}/panel_fy_validtime.parquet")[["ISSUERID", "yr", "dias", "py", "group"]]
fs = con.sql(f"SELECT ISSUERID, MIN(START_DATE)//10000 fs FROM '{CF}' WHERE regexp_matches(FIELD_ID,'_(MODERATE|SEVERE|VERY_SEVERE)_CONTROVERSIES$') GROUP BY 1").df()
ft = con.sql(f"SELECT ISSUERID, MIN(START_DATE)//10000 ft FROM '{CF}' WHERE FIELD_ID='TOTAL_NUMBER_CONTROVERSIES' GROUP BY 1").df()
P = P.merge(fs, on="ISSUERID", how="left").merge(ft, on="ISSUERID", how="left"); P["first"] = (P["fs"] == P["yr"]) | (P["ft"] == P["yr"])
m = P.merge(bs, on=["ISSUERID", "yr"], how="left").merge(bv, on=["ISSUERID", "yr"], how="left")
cols = [c for c in m.columns if c.startswith(("bsev_", "bvs_"))]
for c in cols: m[c] = m[c].fillna(0)
lagp = m[["ISSUERID", "yr"] + cols].rename(columns={"yr": "py", **{c: c + "_p" for c in cols}})
m = m.merge(lagp, on=["ISSUERID", "py"], how="left")
for c in cols:
    m[c + "_p"] = m[c + "_p"].fillna(0); m["d_" + c] = np.where(m["yr"] - m["py"] == 1, m[c] - m[c + "_p"], np.nan)
F = m[m["yr"].isin([2018, 2019, 2021, 2022, 2023, 2024, 2025])].dropna(subset=["dias"]).copy()
rows = []
for lab, D in [("Full (2020 excl.)", F), ("Excl. first-covered", F[~F["first"]])]:
    sev, N, ncl = clustered(D, "dias", ["d_bsev_E", "d_bsev_S", "d_bsev_G"])
    vs, _, _ = clustered(D, "dias", ["d_bvs_E", "d_bvs_S", "d_bvs_G"])
    for p in ["E", "S", "G"]:
        rows.append(dict(sample=lab, pillar=p, severe_coef=sev[f"d_bsev_{p}"][0], severe_t=sev[f"d_bsev_{p}"][1],
                         vsev_coef=vs[f"d_bvs_{p}"][0], vsev_t=vs[f"d_bvs_{p}"][1], N=N, clusters=ncl))
P1 = pd.DataFrame(rows); P1.to_csv("P1_pillar_incorporation.csv", index=False)
# frequency table
freq = []
for lvl in ["mod", "sev", "vsev"]:
    g = A[A[lvl] > 0].groupby("pillar").size(); tot = g.sum()
    for p in ["E", "S", "G"]: freq.append(dict(level=lvl, pillar=p, n=int(g.get(p, 0)), share=round(100 * g.get(p, 0) / tot, 1)))
P1f = pd.DataFrame(freq); P1f.to_csv("P1_pillar_frequency.csv", index=False)
print("(1) pillar incorporation + frequency written")

# ============== (2) FORWARD-LOOKING score test ==============
def asof(field):
    return con.sql(f"""WITH l AS (SELECT ISSUERID,START_DATE,TRY_CAST(VALUE AS DOUBLE) v,
        ROW_NUMBER() OVER(PARTITION BY ISSUERID,START_DATE ORDER BY VERSION_START_DATE DESC) rn
        FROM '{SS}' WHERE FIELD_ID='{field}' AND START_DATE>=20150101),
      s AS (SELECT ISSUERID,START_DATE,v FROM l WHERE rn=1 AND v IS NOT NULL),
      g AS (SELECT DISTINCT ISSUERID, UNNEST(range(2017,2026)) y, UNNEST(range(2017,2026))*10000+1231 thr FROM s)
      SELECT g.ISSUERID,g.y yr,arg_max(x.v,x.START_DATE) score FROM g JOIN s x ON g.ISSUERID=x.ISSUERID AND x.START_DATE<=g.thr GROUP BY 1,2""").df()
A["sv"] = ((A["sev"] > 0) | (A["vsev"] > 0)).astype(int)
sevp = A.groupby(["ISSUERID", "yr", "pillar"])["sv"].max().unstack("pillar", fill_value=0).reset_index()
sevp.columns = ["ISSUERID", "yr"] + [f"sev_{c}" for c in sevp.columns[2:]]
rows = []
for p, nm in [("E", "ENVIRONMENTAL"), ("S", "SOCIAL"), ("G", "GOVERNANCE")]:
    d = asof(f"{nm}_PILLAR_SCORE").merge(sevp[["ISSUERID", "yr", f"sev_{p}"]], on=["ISSUERID", "yr"], how="left")
    d[f"sev_{p}"] = d[f"sev_{p}"].fillna(0); d = d.sort_values(["ISSUERID", "yr"])
    d["lead"] = d.groupby("ISSUERID")[f"sev_{p}"].shift(-1); d["ny"] = d.groupby("ISSUERID")["yr"].shift(-1)
    d = d[(d["ny"] - d["yr"]) == 1].dropna(subset=["lead", "score"]); d = d[d["yr"].between(2018, 2024)].copy()
    d["z"] = (d["score"] - d["score"].mean()) / d["score"].std(); yrs = sorted(d["yr"].unique())
    X = np.column_stack([np.ones(len(d)), d["z"].values, d[f"sev_{p}"].values] + [(d["yr"] == y).astype(float).values for y in yrs[1:]])
    r = sm.OLS(d["lead"].values, X).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d["ISSUERID"])[0]})
    rows.append(dict(pillar=p, score_coef_per_SD=round(r.params[1], 4), score_t=round(r.tvalues[1], 1),
                     current_controversy_coef=round(r.params[2], 4), current_t=round(r.tvalues[2], 1),
                     base_rate=round(d["lead"].mean(), 4), N=len(d)))
P2 = pd.DataFrame(rows); P2.to_csv("P2_forward_looking_scores.csv", index=False)
print("(2) forward-looking score test written")

# ============== (3) OWNERSHIP WEIGHT ==============
q = pd.read_csv(f"{W}/crsp_ownership/mandate_ownership_quarterly.csv", dtype={"parent_id": str})
rows = []
for col, lab in [("mandate_own", "All ESG-named funds"), ("mandate_own_index", "Index-tracking ESG funds")]:
    v = q[col].dropna()
    rows.append(dict(measure=lab, mean_pct=round(100 * v.mean(), 3), median_pct=round(100 * v.median(), 3),
                     p90_pct=round(100 * v.quantile(.9), 3), n_firm_quarters=len(v)))
P3 = pd.DataFrame(rows); P3.to_csv("P3_ownership_weight.csv", index=False)
print("(3) ownership weight written")

# ============== Excel workbook ==============
with pd.ExcelWriter("Pillar_forward_ownership_results.xlsx") as xw:
    P1f.to_excel(xw, sheet_name="Pillar frequency", index=False)
    P1.to_excel(xw, sheet_name="Pillar incorporation", index=False)
    P2.to_excel(xw, sheet_name="Forward-looking scores", index=False)
    P3.to_excel(xw, sheet_name="Ownership weight", index=False)
print("wrote Pillar_forward_ownership_results.xlsx")


**`consequences/downgrade_drift_eventstudy.py`** — Table H3: event-time CARs around downgrades; the drift and its momentum robustness.

In [ ]:
# ====================================================================================
# PIPELINE FILE: consequences/downgrade_drift_eventstudy.py
# Table H3: event-time CARs around downgrades; the drift and its momentum robustness.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""
downgrade_drift_eventstudy.py — reconciles the Appendix H post-downgrade drift with the main
result (no price reaction at the rating change), and shows the drift is NOT an MSCI
re-issue/backfill ("washing out") artifact.

Produces:
  (H3) the event-time CAR path around letter downgrades (months -6..+6) on a BALANCED window
       (events with the full window present), plus the before / event-month / after window CARs -> Table H3
  (robustness) the 6-month forward drift under exclusions of the MSCI batch re-issue / backfill /
               re-rating dates and by sub-period (shows the drift is not a data artifact)
  (control)    whether controlling for the firm's own past-6-month return absorbs the drift

Inputs (from the external-clock kit, not licensed here): ratings.parquet (parent_id, start, ias,
letter, band) and returns.parquet (parent_id, midx, r, ra), where midx = year*12 + month and
ra is the market-adjusted monthly return. Set KIT to that folder.
Run:  KIT=/path/to/kit python downgrade_drift_eventstudy.py
"""
import os, numpy as np, pandas as pd, statsmodels.api as sm

KIT = os.environ.get("KIT", "kit")
out = []
def log(s): out.append(str(s)); print(s)

def ct(y, groups):
    """mean with a firm-clustered t-statistic."""
    r = sm.OLS(np.asarray(y, float), np.ones((len(y), 1))).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(groups)[0]})
    return float(r.tvalues[0])

# ---- letter downgrades from the band series ----
rt = pd.read_parquet(f"{KIT}/ratings.parquet"); rt["parent_id"] = rt.parent_id.astype(str)
rt["d"] = pd.to_datetime(rt.start.astype(str), format="%Y%m%d"); rt = rt.sort_values(["parent_id", "d"])
rt["pband"] = rt.groupby("parent_id").band.shift(1)
dg = rt[rt.band < rt.pband].copy()
dg["midx"] = dg.d.dt.year * 12 + dg.d.dt.month
dg = dg.reset_index(drop=True); dg["ev"] = dg.index

# ---- monthly market-adjusted returns, winsorised 0.5/99.5 ----
ret = pd.read_parquet(f"{KIT}/returns.parquet"); ret["parent_id"] = ret.parent_id.astype(str)
lo, hi = ret.ra.quantile([.005, .995]); ret["raw_"] = ret.ra.clip(lo, hi)
ret = ret.sort_values(["parent_id", "midx"]); gg = ret.groupby("parent_id")
ret["f6"] = sum(gg.raw_.shift(-k) for k in range(1, 7)); ret["f6"] = ret.f6.where(gg.midx.shift(-6) == ret.midx + 6)
ret["b6"] = sum(gg.raw_.shift(k) for k in range(0, 6)); ret["b6"] = ret.b6.where(gg.midx.shift(5) == ret.midx - 5)

# ================= (H3) event-time path, BALANCED -6..+6 window =================
E = dg[["ev", "parent_id", "midx"]].merge(pd.DataFrame({"k": range(-6, 7)}), how="cross")
E["tgt"] = E.midx + E.k
E = E.merge(ret[["parent_id", "midx", "raw_"]].rename(columns={"midx": "tgt", "raw_": "ra"}),
            on=["parent_id", "tgt"], how="inner")
full = E.groupby("ev").k.count(); B = E[E.ev.isin(full[full == 13].index)]      # keep full windows only
log(f"=== Table H3: event-time CAR around letter downgrades (balanced window, N={B.ev.nunique()}) ===")
path, cum = [], 0.0
for k in range(-6, 7):
    d = B[B.k == k]; m = d.ra.mean() * 100; cum += m
    path.append(dict(event_month=k, mean_pct=round(m, 2), t=round(ct(d.ra, d.parent_id), 2),
                     cumCAR_pct=round(cum, 2), N=len(d)))
pathdf = pd.DataFrame(path); log(pathdf.to_string(index=False))

def win(a, b, lab):
    s = B[B.k.between(a, b)].groupby("ev").agg(ra=("ra", "sum"), pid=("parent_id", "first")).reset_index()
    return dict(window=lab, CAR_pct=round(s.ra.mean() * 100, 2), t=round(ct(s.ra, s.pid), 2), N=len(s))
wins = pd.DataFrame([win(-6, -1, "before (-6..-1)"), win(0, 0, "event month (0)"), win(1, 6, "after (+1..+6)")])
log("\nWindow CARs (Table H3 body):"); log(wins.to_string(index=False))
log("The event-month return is of the same order as its neighbours, and the decline before the downgrade")
log("matches the decline after it: no abnormal return at the change; the drift straddles the event.")

# ================= (robustness) the drift is not a washing-out artifact =================
F = dg.merge(ret[["parent_id", "midx", "f6", "b6"]], on=["parent_id", "midx"], how="left").dropna(subset=["f6"])
F["ds"] = F.d.dt.strftime("%Y-%m-%d")
batch = set(F.ds.value_counts()[F.ds.value_counts() > 150].index)   # MSCI mass re-issue / re-rating dates
def rrow(d, lab): return dict(sample=lab, mean_pct=round(d.f6.mean() * 100, 2), t=round(ct(d.f6, d.parent_id), 2), N=len(d))
rob = pd.DataFrame([
    rrow(F, "all downgrades (baseline)"),
    rrow(F[~F.ds.isin(batch)], "excl. mass batch dates (re-rating+backfill)"),
    rrow(F[~F.d.dt.year.isin([2019, 2020])], "excl. 2019-2020 (backfill+re-issue)"),
    rrow(F[F.d.dt.year >= 2021], "2021 onward only"),
    rrow(F[F.d.dt.year <= 2018], "2018 and earlier (pre-backfill)")])
log("\n=== 6-month forward drift: NOT an MSCI re-issue/backfill artifact ===")
log(rob.to_string(index=False))

# ================= (control) past return does not absorb it =================
log("\n=== Does the firm's own past-6-month return absorb the forward drift? ===")
base = ret.dropna(subset=["f6", "b6"]).copy()
key = set(zip(dg.parent_id, dg.midx))
base["dn"] = [1.0 if (p, m) in key else 0.0 for p, m in zip(base.parent_id, base.midx)]
for xs, lab in [(["dn"], "downgrade only"), (["dn", "b6"], "downgrade + past-6m return")]:
    r = sm.OLS(base.f6, sm.add_constant(base[xs])).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(base.parent_id)[0]})
    log(f"  {lab:<30} downgrade coef={r.params['dn']*100:6.2f}%  t={r.tvalues['dn']:6.2f}")

pathdf.to_csv("H3_event_time_path.csv", index=False)
wins.to_csv("H3_window_CARs.csv", index=False)
rob.to_csv("H3_drift_robustness.csv", index=False)
open("downgrade_drift_LOG.txt", "w").write("\n".join(out) + "\n")
log("\nwrote H3_event_time_path.csv, H3_window_CARs.csv, H3_drift_robustness.csv, downgrade_drift_LOG.txt")


**`downgrade_followup/H3_build_and_run.py`** — Giovanna's two follow-up tests: (a) downgrade-month vs flanking-month contrast; (b) four-factor & momentum-matched drift; backfill robustness.

In [ ]:
# ====================================================================================
# PIPELINE FILE: downgrade_followup/H3_build_and_run.py
# Giovanna's two follow-up tests: (a) downgrade-month vs flanking-month contrast; (b) four-factor & momentum-matched drift; backfill robustness.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""
H3_build_and_run.py — rebuild the downgrade event-study kit from the raw MSCI + return files and
run the two tests Giovanna asked for (comments 638/946) plus the backfill/re-rating robustness (640).

Unlike the earlier H3_followup_tests.py (which assumed a pre-built CRSP kit), this version builds the
kit from the files actually available:

  INPUTS
    HIST_SUMMARY_SCORES_20260731.parquet               MSCI scores: IVA_COMPANY_RATING (letter), INDUSTRY_ADJUSTED_SCORE
    HIST_MASTER_INSTRUMENT_REFERENCE_*.parquet         ISSUERID <-> ISIN (ASSET_TYPE='CMN')
    monthly_total_returns.csv                          global monthly total returns by ISIN (tr_return_pct, country)
    ff_monthly.parquet                                 Fama-French + Carhart factors (mktrf, smb, hml, umd, rf)

  BUILD
    ratings:  letter -> band (AAA=7..CCC=1); a downgrade is band < previous band, dated at START_DATE.
    returns:  ISIN monthly total return (/100) -> ISSUERID (median across the issuer's common shares);
              market-adjusted 'ra' = return minus the country-month mean (as in the Table H3 note);
              momentum-matched 'cmadj' = ra minus the mean of its country x month x prior-12m-return tercile;
              four-factor 'ff4' (US-listed only) = excess return minus a firm's fitted Carhart four-factor expectation.

  TESTS
    (a) month-0 return vs the mean of months -6..-1 and +1..+6, firm-clustered difference t (comment 946).
    (b) the balanced -6..+6 event-time path under market-adjusted, momentum-matched and four-factor returns (comment 638).
    robustness: 6-month forward drift excl. 2019 backfill / mass re-rating dates / by sub-period, and a past-6m control (640).

Run:  python3 H3_build_and_run.py      (set the paths below)
"""
import os, duckdb, pandas as pd, numpy as np, statsmodels.api as sm

SS  = os.environ.get("SS",  "HIST_SUMMARY_SCORES_20260731.parquet")
IR  = os.environ.get("IR",  "HIST_MASTER_INSTRUMENT_REFERENCE_none_20260831.parquet")
RET = os.environ.get("RET", "monthly_total_returns.csv")
FF  = os.environ.get("FF",  "ff_monthly.parquet")
con = duckdb.connect(); con.sql("PRAGMA threads=4")
BAND = {"AAA":7,"AA":6,"A":5,"BBB":4,"BB":3,"B":2,"CCC":1}
ct = lambda y,g: float(sm.OLS(np.asarray(y,float),np.ones((len(y),1))).fit(cov_type="cluster",cov_kwds={"groups":pd.factorize(g)[0]}).tvalues[0])
wins = lambda s:(lambda lo,hi:s.clip(lo,hi))(*s.quantile([.005,.995]))

# ---------- ratings / downgrades ----------
lett=con.sql(f"""WITH l AS (SELECT ISSUERID,START_DATE,VALUE,ROW_NUMBER() OVER(PARTITION BY ISSUERID,START_DATE ORDER BY VERSION_START_DATE DESC) rn
   FROM '{SS}' WHERE FIELD_ID='IVA_COMPANY_RATING' AND START_DATE>=20150101) SELECT ISSUERID,START_DATE,VALUE letter FROM l WHERE rn=1""").df()
lett["band"]=lett.letter.map(BAND); lett=lett.dropna(subset=["band"]).sort_values(["ISSUERID","START_DATE"])
lett["pband"]=lett.groupby("ISSUERID").band.shift(1)
dg=lett[lett.band<lett.pband].rename(columns={"ISSUERID":"parent_id","START_DATE":"start"}).copy()
dg["d"]=pd.to_datetime(dg.start.astype(int).astype(str),format="%Y%m%d"); dg["midx"]=dg.d.dt.year*12+dg.d.dt.month
dg=dg.reset_index(drop=True); dg["ev"]=dg.index
print("downgrades:",len(dg))

# ---------- returns ----------
link=con.sql(f"SELECT ISIN, ANY_VALUE(ISSUERID) issuerid FROM '{IR}' WHERE ISIN IS NOT NULL AND ASSET_TYPE='CMN' GROUP BY ISIN").df()
ret=con.sql(f"""WITH r AS (SELECT isin, ISSUER_CNTRY_DOMICILE ctry, CAST(date AS DATE) dt, TRY_CAST(tr_return_pct AS DOUBLE)/100.0 r FROM read_csv_auto('{RET}'))
  SELECT l.issuerid parent_id, ANY_VALUE(r.ctry) ctry, (EXTRACT(year FROM r.dt)*12+EXTRACT(month FROM r.dt))::INT midx, median(r.r) r
  FROM r JOIN link l ON r.isin=l.ISIN WHERE r.r IS NOT NULL GROUP BY 1,3""").df()
ret["ra"]=ret.r-ret.groupby(["ctry","midx"]).r.transform("mean")
ret=ret.sort_values(["parent_id","midx"]); g=ret.groupby("parent_id")
ret["mom"]=sum(g.r.shift(k) for k in range(1,13))
ret["mb"]=ret.groupby(["ctry","midx"]).mom.transform(lambda s: pd.qcut(s,3,labels=False,duplicates="drop") if s.notna().sum()>=9 else np.nan)
ret["cmadj"]=ret.ra-ret.groupby(ret.ctry.astype(str)+ret.midx.astype(str)+ret.mb.astype("Int64").astype(str)).ra.transform("mean")
ff=pd.read_parquet(FF); ff["midx"]=ff.year.astype(int)*12+ff.month.astype(int)
us=ret[ret.ctry=="US"].merge(ff[["midx","mktrf","smb","hml","umd","rf"]],on="midx",how="left"); us["exc"]=us.r-us.rf
FAC=["mktrf","smb","hml","umd"]
def resid(gp):
    gp=gp.dropna(subset=["exc"]+FAC)
    if len(gp)<24: return pd.Series(np.nan,index=gp.index)
    b=sm.OLS(gp.exc,sm.add_constant(gp[FAC])).fit().params; return gp.exc-(b["const"]+(gp[FAC]*b[FAC]).sum(axis=1))
us["ff4"]=us.groupby("parent_id",group_keys=False).apply(resid)
ret=ret.merge(us[["parent_id","midx","ff4"]],on=["parent_id","midx"],how="left")

def path_and_contrast(col,label,sub=None):
    r=(ret if sub is None else ret[sub]); rr=r[["parent_id","midx",col]].dropna().copy(); rr["x"]=wins(rr[col])
    E=dg[["ev","parent_id","midx"]].merge(pd.DataFrame({"k":range(-6,7)}),how="cross"); E["tgt"]=E.midx+E.k
    E=E.merge(rr[["parent_id","midx","x"]].rename(columns={"midx":"tgt"}),on=["parent_id","tgt"],how="inner").dropna(subset=["x"])
    B=E[E.ev.isin(E.groupby("ev").k.count().pipe(lambda s:s[s==13]).index)]
    print(f"\n### {label} (N={B.ev.nunique()})")
    for a,b,lab in [(-6,-1,"before"),(0,0,"month0"),(1,6,"after")]:
        s=B[B.k.between(a,b)].groupby("ev").agg(x=("x","sum"),pid=("parent_id","first")).reset_index()
        print(f"   {lab:7s} {s.x.mean()*100:+.2f}%  t={ct(s.x,s.pid):+.2f}")
    fl=B[B.k!=0].groupby("ev").agg(f=("x","mean"),pid=("parent_id","first")).reset_index()
    z=B[B.k==0][["ev","x"]].rename(columns={"x":"m0"}); c=fl.merge(z,on="ev"); c["diff"]=c.m0-c.f
    print(f"   TEST(a): month0={c.m0.mean()*100:+.2f}% flank={c.f.mean()*100:+.2f}% diff={c['diff'].mean()*100:+.2f}pp t={ct(c['diff'],c.pid):+.2f} N={len(c)}")

path_and_contrast("ra","(RA) market-adjusted, full")
path_and_contrast("cmadj","(b2) momentum-matched, full")
path_and_contrast("ff4","(b1) four-factor abnormal, US-listed", sub=(ret.ctry=="US"))

# robustness
for col,name in [("ra","market-adjusted"),("cmadj","momentum-matched")]:
    r=ret.copy(); r["w"]=wins(r[col]); g=r.sort_values(["parent_id","midx"]).groupby("parent_id")
    r["f6"]=sum(g.w.shift(-k) for k in range(1,7)); r["f6"]=r.f6.where(g.midx.shift(-6)==r.midx+6)
    r["b6"]=sum(g.w.shift(k) for k in range(0,6)); r["b6"]=r.b6.where(g.midx.shift(5)==r.midx-5)
    F=dg.merge(r[["parent_id","midx","f6","b6"]],on=["parent_id","midx"],how="left").dropna(subset=["f6"])
    F["ds"]=F.d.dt.strftime("%Y-%m-%d"); batch=set(F.ds.value_counts()[F.ds.value_counts()>150].index)
    print(f"\n=== 6m forward drift robustness [{name}] ===")
    for d_,l in [(F,"baseline"),(F[~F.ds.isin(batch)],"excl re-rating/backfill dates"),(F[F.d.dt.year!=2019],"excl 2019"),
                 (F[~F.d.dt.year.isin([2019,2020])],"excl 2019-2020"),(F[F.d.dt.year>=2021],"2021+"),(F[F.d.dt.year<=2018],"pre-2019")]:
        print(f"   {l:30s} {d_.f6.mean()*100:+.2f}%  t={ct(d_.f6,d_.parent_id):+.2f}  N={len(d_)}")


### 4.7 Rebuild checks & result assembly

**`robustness/a6_section_numbers.py`** — Rebuilds the section-number / summary statistics quoted in the text.

In [ ]:
# ====================================================================================
# PIPELINE FILE: robustness/a6_section_numbers.py
# Rebuilds the section-number / summary statistics quoted in the text.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

import duckdb, numpy as np, pandas as pd
W="/home/claude/w2"; CF=f"{W}/jul/HIST_CONTROVERSIES_20260731.parquet"
con=duckdb.connect()
# --- 2020 measurement check: severity records ending / starting on the same day in 2020
sevre="regexp_matches(FIELD_ID,'_(MODERATE|SEVERE|VERY_SEVERE)_CONTROVERSIES$')"
r=con.sql(f"""WITH l AS (SELECT ISSUERID,FIELD_ID,START_DATE,END_DATE,ROW_NUMBER() OVER(PARTITION BY ISSUERID,FIELD_ID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{CF}' WHERE {sevre})
 SELECT END_DATE d, COUNT(*) n, COUNT(DISTINCT ISSUERID) iss FROM l WHERE rn=1 AND END_DATE BETWEEN 20180101 AND 20251231 GROUP BY 1 ORDER BY n DESC LIMIT 6""").df(); print("most common END_DATEs:\n",r)
r2=con.sql(f"""WITH l AS (SELECT ISSUERID,FIELD_ID,START_DATE,ROW_NUMBER() OVER(PARTITION BY ISSUERID,FIELD_ID,START_DATE ORDER BY VERSION_START_DATE DESC) rn FROM '{CF}' WHERE {sevre})
 SELECT START_DATE d, COUNT(*) n, COUNT(DISTINCT ISSUERID) iss FROM l WHERE rn=1 AND START_DATE BETWEEN 20180101 AND 20251231 GROUP BY 1 ORDER BY n DESC LIMIT 6""").df(); print("most common START_DATEs:\n",r2)
P=pd.read_parquet(f"{W}/panel_fy_validtime.parquet")
def cl(d,y,xs,c="group",fe_firm=False):
    d=d.dropna(subset=[y]+xs).copy(); d["yr"]=d.yr.astype(int)
    Y=d[y].values.astype(float); X=d[xs].values.astype(float)
    # demean by year (and firm if requested) via iterative demeaning
    def dm(a,keys):
        a=a.copy()
        for _ in range(30 if len(keys)>1 else 1):
            for k in keys:
                a=a-pd.DataFrame(a).groupby(d[k].values).transform('mean').values
        return a
    keys=["yr"]+(["ISSUERID"] if fe_firm else [])
    Yd=dm(Y.reshape(-1,1),keys).ravel(); Xd=dm(X,keys)
    XtXi=np.linalg.inv(Xd.T@Xd); b=XtXi@(Xd.T@Yd); e=Yd-Xd@b
    S=pd.DataFrame(Xd*e[:,None]).groupby(d[c].values).sum().values; N,k=Xd.shape; G=S.shape[0]
    se=np.sqrt(np.diag((G/(G-1))*((N-1)/(N-k))*(XtXi@(S.T@S)@XtXi)))
    return {x:(round(b[i],4),round(b[i]/se[i],2)) for i,x in enumerate(xs)}, N, G
W8=[2018,2019,2021,2022,2023,2024,2025]
F=P[P.yr.isin(W8)].dropna(subset=["dias","d_c_sev"])
F20=P[P.yr.isin(W8+[2020])].dropna(subset=["dias","d_c_sev"])
print("N excl/incl 2020:",len(F),len(F20))
for c in ["c_mod","c_sev","c_vs"]:
    print(c,"incl2020",cl(F20,"dias",["d_"+c]))
print("firm FE severe:",cl(F,"dias",["d_c_sev"],fe_firm=True))
# pass-through by severity: dWAKIS response and dIAS response, slope dIAS on dWAKIS
sl=cl(F,"dias",["dwas"]); print("slope dIAS on dWAKIS:",sl)
for c in ["c_mod","c_sev","c_vs"]:
    print(c,"dIAS",cl(F,"dias",["d_"+c])[0],"dWAKIS",cl(F,"dwas",["d_"+c])[0])
# R2
d=F.dropna(subset=["dwas"]); 
def r2(y,x): return np.corrcoef(y,x)[0,1]**2
print("R2 pooled",round(r2(d.dias,d.dwas),3))
yd=d.dias-d.groupby("yr").dias.transform("mean"); xd=d.dwas-d.groupby("yr").dwas.transform("mean"); print("within-year",round(r2(yd,xd),3))
yf=d.dias-d.groupby("ISSUERID").dias.transform("mean"); xf=d.dwas-d.groupby("ISSUERID").dwas.transform("mean"); print("within-firm",round(r2(yf,xf),3))
# Table 1 (consistent valid-time severity counts; no total/residual)
d=F
rows=[]
for v,l in [("ias","IAS"),("was","WAKIS"),("c_mod","Moderate controversies"),("c_sev","Severe controversies"),("c_vs","Very severe controversies"),("dias","ΔIAS"),("dwas","ΔWAKIS"),("d_c_mod","ΔModerate"),("d_c_sev","ΔSevere"),("d_c_vs","ΔVery severe")]:
    s=d[v].dropna(); rows.append(dict(var=l,N=len(s),mean=s.mean(),sd=s.std(),p25=s.quantile(.25),med=s.median(),p75=s.quantile(.75)))
T1=pd.DataFrame(rows); print(T1.round(3).to_string()); T1.to_csv(f"{W}/out/T1_new.csv",index=False)
print("share firm-years with any severity change:",round(((d.d_c_mod!=0)|(d.d_c_sev!=0)|(d.d_c_vs!=0)).mean(),3),
      "severe change:",round((d.d_c_sev!=0).mean(),3), "issuers",d.ISSUERID.nunique(),"groups",d.group.nunique())


**`robustness/verify_data_facts.py`** — Sanity checks on the headline data facts (sample sizes, counts).

In [ ]:
# ====================================================================================
# PIPELINE FILE: robustness/verify_data_facts.py
# Sanity checks on the headline data facts (sample sizes, counts).
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""
verify_data_facts.py — reproduces the descriptive data claims in Section 4 of the manuscript:
  (a) the within-2020 re-issue of MSCI's key-issue severity records,
  (b) the resulting collapse in severity records in force between end-2019 and end-2020,
  (c) the 1 July 2019 coverage-start backfill,
  (d) the small IAS-vs-WAKIS gap behind the Table 1 N for WAKIS / dWAKIS.

Inputs (licensed, not shipped): the July-2026 MSCI extract
  HIST_CONTROVERSIES_20260731.parquet, HIST_SUMMARY_SCORES_20260731.parquet
Set JUL to the folder that holds them, then: python verify_data_facts.py
"""
import duckdb, sys, os

JUL = os.environ.get("JUL", ".")
CF = f"{JUL}/HIST_CONTROVERSIES_20260731.parquet"
SS = f"{JUL}/HIST_SUMMARY_SCORES_20260731.parquet"
con = duckdb.connect(); con.sql("PRAGMA threads=4")
SEV = r'_(MODERATE|SEVERE|VERY_SEVERE)_CONTROVERSIES$'
out = []

def line(s): out.append(s); print(s)

line("== (a) Within-2020 re-issue of key-issue severity records ==")
# Closures live on the effective axis (END_DATE, integer); openings on START_DATE.
closed = con.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT ISSUERID) iss FROM '{CF}' "
                 f"WHERE regexp_matches(FIELD_ID,'{SEV}') AND END_DATE=20200828").df()
opened = con.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT ISSUERID) iss FROM '{CF}' "
                 f"WHERE regexp_matches(FIELD_ID,'{SEV}') AND START_DATE=20201110").df()
line(f"  closed 2020-08-28: {int(closed.n[0]):,} records / {int(closed.iss[0]):,} issuers   (paper: 748,944 / 50,158)")
line(f"  opened 2020-11-10: {int(opened.n[0]):,} records / {int(opened.iss[0]):,} issuers   (paper: 134,326 / 14,299)")

line("\n== (b) Severity records in force at each year-end (valid-time) ==")
for y in (2018, 2019, 2020, 2021):
    ye = y*10000 + 1231
    r = con.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT ISSUERID) iss FROM '{CF}' "
                f"WHERE regexp_matches(FIELD_ID,'{SEV}') AND START_DATE<={ye} AND END_DATE>{ye}").df()
    line(f"  end-{y}: {int(r.n[0]):>9,} records / {int(r.iss[0]):>6,} issuers")
line("  (paper: ~910,000/62,800 at end-2019 -> ~326,000/15,300 at end-2020)")

line("\n== (c) Coverage-start backfill, effective-dated 1 July 2019 ==")
cov = con.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT ISSUERID) iss FROM '{CF}' "
              f"WHERE regexp_matches(FIELD_ID,'{SEV}') AND START_DATE=20190701").df()
line(f"  START_DATE=2019-07-01: {int(cov.n[0]):,} records / {int(cov.iss[0]):,} issuers   (paper: 920,424 / 62,716)")

line("\n== (d) IAS-vs-WAKIS gap behind Table 1's WAKIS / dWAKIS N ==")
def inforce(field, y):
    return (f"SELECT ISSUERID, arg_max(TRY_CAST(VALUE AS DOUBLE), VERSION_START_DATE) v FROM '{SS}' "
            f"WHERE FIELD_ID='{field}' AND START_DATE>=20150101 AND START_DATE<={y}*10000+1231 "
            f"AND (END_DATE IS NULL OR END_DATE>={y}*10000+1231) GROUP BY 1")
for y in (2022,):
    ias = con.sql(inforce("INDUSTRY_ADJUSTED_SCORE", y)).df().dropna()
    wak = con.sql(inforce("WEIGHTED_AVERAGE_SCORE", y)).df().dropna()
    m = ias.merge(wak, on="ISSUERID", how="left", suffixes=("_ias", "_wak"))
    line(f"  year {y}: IAS firms={len(ias):,}; with WAKIS={int(m.v_wak.notna().sum()):,}; "
         f"IAS-but-no-WAKIS={int(m.v_wak.isna().sum()):,}")
line("  => a few firm-years each year carry an IAS with no WAKIS in force at year-end, so in Table 1")
line("     WAKIS=401,872 and dWAKIS (needs WAKIS in t and t-1)=401,835 vs 401,885 for all other variables.")

open("verify_data_facts_LOG.txt", "w").write("\n".join(out) + "\n")
print("\nwrote verify_data_facts_LOG.txt")


**`robustness/build_results.py`** — Assembles the robustness result tables into one place.

In [ ]:
# ====================================================================================
# PIPELINE FILE: robustness/build_results.py
# Assembles the robustness result tables into one place.
# Set input paths to your Drive (Section 0). Run Section 3 before Section 4.
# ====================================================================================

#!/usr/bin/env python3
"""Results note for the five requested analyses (Word, Times New Roman 12, black)."""
import numpy as np, pandas as pd
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Pt, Cm, RGBColor

O = "/home/claude/w2/out"
doc = Document()
st = doc.styles["Normal"]; st.font.name = "Times New Roman"; st.font.size = Pt(12); st.font.color.rgb = RGBColor(0, 0, 0)
st.element.rPr.rFonts.set(qn("w:eastAsia"), "Times New Roman")
for s in doc.sections:
    s.left_margin = s.right_margin = Cm(2.2); s.top_margin = s.bottom_margin = Cm(2.0)


def para(t, b=False, i=False, sz=12, al=None, after=6):
    p = doc.add_paragraph(); r = p.add_run(t); r.bold = b; r.italic = i; r.font.size = Pt(sz); r.font.color.rgb = RGBColor(0, 0, 0)
    p.paragraph_format.space_after = Pt(after)
    if al == "c": p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    if al == "j": p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    return p


def head(t):
    p = para(t, b=True, sz=13, after=4); p.paragraph_format.space_before = Pt(12)


def border(cell, edges):
    tcPr = cell._tc.get_or_add_tcPr(); b = OxmlElement("w:tcBorders")
    for e in edges:
        el = OxmlElement(f"w:{e}"); el.set(qn("w:val"), "single"); el.set(qn("w:sz"), "8"); el.set(qn("w:color"), "000000"); b.append(el)
    tcPr.append(b)


def table(rows, widths, header_rows=1, sz=9.5, bold_rows=()):
    t = doc.add_table(rows=len(rows), cols=len(rows[0])); t.autofit = False
    for i, row in enumerate(rows):
        for j, v in enumerate(row):
            c = t.cell(i, j); c.width = Cm(widths[j]); c.text = ""
            p = c.paragraphs[0]; r = p.add_run(str(v)); r.font.size = Pt(sz); r.font.color.rgb = RGBColor(0, 0, 0)
            r.bold = (i < header_rows) or (i in bold_rows)
            p.paragraph_format.space_after = Pt(0)
            if j > 0: p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            edges = []
            if i == 0: edges.append("top")
            if i == header_rows - 1: edges.append("bottom")
            if i == len(rows) - 1: edges.append("bottom")
            if edges: border(c, edges)
    return t


def note(t):
    p = para(t, sz=10, after=4, al="j"); p.paragraph_format.space_before = Pt(4)


def f(b, t, d=3):
    return "" if pd.isna(b) else f"{b:+.{d}f} ({t:.2f})".replace("-", "−")


def n0(x):
    return f"{int(x):,}"


para("Additional analyses requested on 31 October 2026", b=True, sz=15, al="c")
para("Results only; the draft has not been edited. Each table is followed by a three-line note.", i=True, al="c", after=10)

# ======================================================================= 1
A1 = pd.read_csv(f"{O}/A1_ownership_linked.csv")
FL = {"all": "All ESG-named funds", "idx": "MSCI-ESG index funds", "oth": "Other ESG-named funds"}
head("1. Holdings response to controversy-linked versus other downgrades")
for key, a, bnm, title in [("Def. 1", "dg_link1", "dg_not1", "Panel A. Linked = severe or very severe controversy logged by MSCI in the 12 months before the downgrade"),
                           ("Def. 2", "dg_link2", "dg_not2", "Panel B. Linked = as in Panel A, and the linked key issue's score fell between the onset and the downgrade")]:
    para(title, i=True, sz=10.5, after=2)
    rows = [["Funds", "h", "Linked downgrade (b₁)", "Other downgrade (b₂)", "b₁ − b₂", "Upgrade", "Mean level", "N / clusters", "Linked events"]]
    for fd in ["all", "idx", "oth"]:
        for h in (1, 2, 4):
            r = A1[(A1.spec.str.startswith(key)) & (A1.funds == fd) & (A1.h == h)].iloc[0]
            rows.append([FL[fd] if h == 1 else "", f"q+{h}", f(r[a], r[a + "_t"], 4), f(r[bnm], r[bnm + "_t"], 4),
                         f(r["diff"], r["diff_t"], 4), f(r.upgrade, r.upgrade_t, 4), f"{r.mean_level:.3f}%",
                         f"{n0(r.N)} / {n0(r.clusters)}", n0(r[a + "_n"])])
    table(rows, [3.3, 0.9, 2.6, 2.6, 2.0, 2.2, 1.5, 2.3, 1.4])
para("Panel C. Ownership change around controversy dates with no rating change in quarters q to q+4", i=True, sz=10.5, after=2).paragraph_format.space_before = Pt(8)
rows = [["Funds", "h", "MSCI logging date (severe/very severe)", "Events", "Public-report date (hand-checked, US)", "Events", "N / clusters"]]
for fd in ["all", "idx", "oth"]:
    for h in (1, 2, 4):
        c = A1[(A1.spec.str.startswith("MSCI logging")) & (A1.funds == fd) & (A1.h == h)].iloc[0]
        p = A1[(A1.spec.str.contains("public-report date, no rating")) & (A1.funds == fd) & (A1.h == h)].iloc[0]
        rows.append([FL[fd] if h == 1 else "", f"q+{h}", f(c.contro_nochg, c.contro_nochg_t, 4), n0(c.contro_nochg_n),
                     f(p.public_nochg, p.public_nochg_t, 4), n0(p.public_nochg_n), f"{n0(c.N)} / {n0(c.clusters)}"])
table(rows, [3.3, 0.9, 3.6, 1.4, 3.6, 1.4, 2.6])
note("Table R1. Change in ESG-mandated ownership (percentage points of market capitalisation) from quarter q−1 to q+h, US-listed "
     "shares, quarters from 2018Q1; quarter fixed effects; Compustat controls (log assets, leverage, ROA, log book-to-market); "
     "t-statistics clustered by firm (ultimate parent, one CRSP security per parent). Downgrades and upgrades are letter-band changes as "
     "in Table 10, which this code reproduces for the full period (q+4: −0.0201, t = −4.48). Onsets dated 2019-07-01 or 2017-08-31 are "
     "excluded. Panel C regressions also include downgrade and upgrade indicators. Linked events: firm-quarters with a linked "
     "downgrade and an observed ownership change.")
para("What it shows. (1) Downgrades that follow a severe controversy reduce ESG ownership as much as other downgrades: b₁ is "
     "negative and significant for all ESG-named funds (−0.026 to −0.034) and for MSCI-index funds at every horizon, and b₁ = b₂ is "
     "never rejected (p > 0.4). (2) MSCI-index funds do not move when a controversy is logged without a rating change "
     "(−0.001, t = −0.9 at q+1; −0.004, t = −1.9 at q+4), nor at the public report (n = 45, not significant). (3) Other ESG-named "
     "funds do drift down over four quarters after a logged controversy without a rating change (−0.013, t = −1.9; all funds −0.018, "
     "t = −2.3), so part of the discretionary ESG money responds to the controversy itself, not only to the rating.",
     sz=11, al="j")

# ======================================================================= 2
A2 = pd.read_csv(f"{O}/A2_table2_backfill.csv"); ov = pd.read_csv(f"{O}/A2_overlap.csv", index_col=0).iloc[:, 0]
head("2. Table 2 without backfilled and floored onsets")
S = ["Table 2 (all)", "Excl. backfill/floor onsets", "Excl. both"]
rows = [["Severity level", "Table 2: t", "Table 2: t+1", "Excl. backfill/floor: t", "Excl. backfill/floor: t+1",
         "Excl. backfill/floor and first covered year: t", "… : t+1"]]
for lv in ["Other (residual)", "Moderate", "Severe", "Very severe"]:
    rr = [lv]
    for s in S:
        x = A2[(A2["sample"] == s) & (A2.level == lv)].iloc[0]
        rr += [f(x.b_t, x.t_t, 4), f(x.b_t1, x.t_t1, 4)]
    rows.append(rr)
nrow = ["N (firm-years)"]; grow = ["N (ultimate-parent clusters)"]
for s in S:
    x = A2[A2["sample"] == s].iloc[0]; nrow += [n0(x.N_t), n0(x.N_t1)]; grow += [n0(x.G_t), n0(x.G_t1)]
rows += [nrow, grow]
table(rows, [3.2, 2.1, 2.1, 2.3, 2.3, 2.6, 2.1])
note(f"Table R2. OLS of ΔIAS (t) and ΔIAS (t+1) on the change in the count at each severity level, one level per regression as in "
     f"Table 2; paper sample (401,885 firm-years, IAS records from 2015); year fixed effects; standard errors clustered by ultimate "
     f"parent; t-statistics in brackets. Backfill/floor firm-years: issuer-years with a positive key-issue severity record starting on "
     f"2019-07-01 or 2017-08-31 ({n0(ov['backfill_any'])} firm-years, all in 2019, of which {n0(ov['backfill_with_change'])} have a "
     f"severity change; no 2017-08-31 record falls in a sample change-year). Overlap with the first-covered-year flag "
     f"(14,389 firm-years): {n0(ov['both'])} firm-years are in both, {n0(ov['backfill_not_first'])} only in the backfill flag, "
     f"{n0(ov['first_not_backfill'])} only in the first-covered flag; union {n0(ov['union'])}.")
para("What it shows. Dropping the 4,746 backfill firm-years (1.2% of the sample) leaves the moderate (−0.010, t = −2.7), severe "
     "(−0.054, t = −3.3) and very severe (−0.324, t = −3.6) contemporaneous coefficients essentially unchanged. The residual moves "
     "from a one-year-later effect (−0.006, t = −3.3) to a same-year effect (−0.010, t = −3.0; t+1 −0.003, t = −1.6), so the residual's "
     "lag in Table 2 comes from the 2019 backfill. The backfill flag and the first-covered-year flag mostly identify the same firm-years "
     "(3,927 of 4,746), and dropping both gives the same picture.", sz=11, al="j")

# ======================================================================= 3
A3 = pd.read_csv(f"{O}/A3_threshold_lpm.csv"); te = pd.read_csv(f"{O}/A3_tests.csv")
head("3. Distance to the letter threshold")
bins = ["<0.3", "0.3-0.6", "0.6-1.0", "1.0-1.43"]
rows = [["Controversy at t", "Very near (< 0.3)", "Near (0.3–0.6)", "Mid (0.6–1.0)", "Far (1.0–1.43)"]]
for s in ["none", "moderate", "severe", "very severe"]:
    rr = [s.capitalize() + (" †" if s == "very severe" else "")]
    for bn in bins:
        x = A3[(A3.severity == s) & (A3.bin == bn)].iloc[0]
        coef = "ref." if (s == "none" and bn == "1.0-1.43") else f(x.coef_pp / 100 * 100, x.t, 1).replace("(", "(t = ").replace(")", ")")
        rr.append(f"{coef}\n{x.raw_pct:.1f}% [{n0(x.N)}]")
    rows.append(rr)
get = lambda k: te[te.test == k].iloc[0]
rows.append(["Severe − none", f(get('severe minus none, <0.3').pp, get('severe minus none, <0.3').t, 1), "", "",
             f(get('severe minus none, 1.0-1.43').pp, get('severe minus none, 1.0-1.43').t, 1)])
rows.append(["Moderate − none", f(get('moderate minus none, <0.3').pp, get('moderate minus none, <0.3').t, 1), "", "",
             f(get('moderate minus none, 1.0-1.43').pp, get('moderate minus none, 1.0-1.43').t, 1)])
table(rows, [3.4, 3.2, 3.2, 3.2, 3.2], bold_rows=())
note("Table R3. Linear probability model of a letter downgrade by the next year-end (percentage points), on 16 distance-bin × "
     "severity cells (reference: no new controversy, far from the threshold) with year fixed effects; standard errors clustered by "
     "ultimate parent (432,677 firm-years, 28,061 clusters; firm-years 2018–2025 excl. 2020 with IAS records from 2015; CCC "
     "firm-years, which cannot be downgraded, excluded). Each cell: coefficient (t), then the raw downgrade rate [N]. Distance is the "
     "IAS minus the lower edge of its letter band (band width 1.43). Severity is the highest new key-issue onset in year t. "
     "Bottom rows: severe (moderate) minus none within the very-near and far bins. † Very severe cells are thin (47–65 firm-years) "
     "and not interpreted; the very-near very severe cell has no downgrades.")
para("What it shows. A severe controversy raises the probability of a downgrade within a year by about 20 points when the firm is "
     "within 0.3 of the threshold (t = 2.3) and by about 3 points when it is far (t = 2.7); the near-minus-far difference is 17 points "
     "(t = 2.0). Moderate controversies show the same pattern more precisely (12 against 3 points; difference 8.5, t = 3.8). The "
     "letter-stage attenuation result therefore holds in the LPM with parent clustering.", sz=11, al="j")
doc.add_picture(f"{O}/A3_bunching.png", width=Cm(16.5))
note("Figure R3. Distribution of year-end IAS around each letter cut-off (k × 10/7), firm-years 2018–2025 excl. 2020, IAS records from "
     "2015; 99.5% of year-end IAS values lie on the one-decimal grid, so bars are 0.1 wide; grey = below the cut-off (lower letter), "
     "blue = above. Counts are in A3_bunching_counts.csv.")
para("What it shows. The distribution is not smooth at the cut-offs. Mass piles up just above the CCC/B and B/BB cut-offs "
     "(2.9: 9,288 against 3,814 at 2.8) and just below the BB/BBB, BBB/A, A/AA and AA/AAA cut-offs (7.1: 16,801 against 2,749 at 7.2; "
     "4.2: 12,037 against 4,583 at 4.3). This is consistent with discretion at the thresholds of the kind Berg, Cornaggia, Foroni and "
     "Tripoli (2026) study, but on a 0.1 grid it is descriptive; whether the pattern matches the direction they report should be "
     "checked against their paper before it is cited.", sz=11, al="j")

# ======================================================================= 4
head("4. The hand-checked clock sample")
F4 = pd.read_csv(f"{O}/A4_funnel.csv")
rows = [["Stage", "N", "Dropped", "Reason for the drop"]] + [[r["stage"], n0(r["N"]), n0(r["dropped"]), r["reason for drop"]] for _, r in F4.iterrows()]
table(rows, [5.4, 1.4, 1.4, 8.4], sz=9)
note("Table R4a. Funnel from all US and UK onsets to the 223 verified controversies. The first-round rule that selected 670 of "
     "the 2,399 matched onsets is not recorded in the audit files; 638 of the 670 have a matched article dated 0–90 days before the "
     "MSCI date. 'Screen only' rows have a headline judged correct but no verified earliest report.")
R4 = pd.read_csv(f"{O}/A4_representativeness.csv")
rows = [["Variable", "Category", "Verified 223 (%)", "Other onsets (%)", "Test"]]
last = None
for r in R4.itertuples():
    rows.append([r.variable if r.variable != last else "", r.category,
                 f"{r.verified_pct:.1f}" if r.variable != "Onsets" else n0(r.verified_pct),
                 f"{r.other_pct:.1f}" if r.variable != "Onsets" else n0(r.other_pct), r.test])
    last = r.variable
table(rows, [4.0, 4.2, 2.6, 2.6, 3.0], sz=9)
note("Table R4b. The 223 verified controversies against the other 13,984 US/UK onsets. Sector from MSCI's parent file (GICS sector; "
     "'missing' where not recorded); log total assets from Compustat, available for 177 verified and 7,916 other onsets. Chi-square "
     "tests for categories, Welch t-test for log assets.")
RW = pd.read_csv(f"{O}/A4_reweighted_lag.csv"); MX = pd.read_csv(f"{O}/A4_severity_mix.csv", index_col=0)
rows = [["Weighting", "Median", "Mean", "P25–P75", "≤ 90 days"]] + [[r.weighting, f"{r.median:.0f}", f"{r.mean:.0f}", f"{r.p25:.0f}–{r.p75:.0f}", f"{r.le90:.0f}%"] for r in RW.itertuples()]
table(rows, [8.0, 1.8, 1.8, 2.4, 2.2])
note(f"Table R4c. Logging lag in days. Weights equal each severity's share among the 14,207 onsets divided by its share among the "
     f"223: moderate {MX.loc['moderate','share_all_onsets']*100:.1f}% vs {MX.loc['moderate','share_verified']*100:.1f}% (weight "
     f"{MX.loc['moderate','weight']:.2f}), severe {MX.loc['severe','share_all_onsets']*100:.1f}% vs {MX.loc['severe','share_verified']*100:.1f}% "
     f"({MX.loc['severe','weight']:.2f}), very severe {MX.loc['very_severe','share_all_onsets']*100:.1f}% vs "
     f"{MX.loc['very_severe','share_verified']*100:.1f}% ({MX.loc['very_severe','weight']:.2f}).")
para("What it shows. The verified sample is not representative: it over-weights very severe cases (25% against 1.5%), 2021–2025, "
     "Energy and Financials, and much larger firms (log assets 12.3 against 10.9). Reweighting to the population severity mix leaves "
     "the median lag at 80 days and lowers the mean from 108 to 104 days, so the 80-day headline does not come from the severity "
     "over-weighting; the size and sector differences cannot be reweighted away and should be stated as a scope limit.", sz=11, al="j")
para("Audit protocol (draft appendix paragraph, to be checked against how the audit was actually done).", b=True, sz=11)
para("For each controversy, the auditor identified the firm-specific incident that MSCI most plausibly logged under the stated key "
     "issue, searching the period up to about one year before the MSCI date (and up to seven days after it). Sources were searched in "
     "order of priority: primary documents (regulator and enforcement releases such as DOJ, SEC, FTC, CFPB, OCC, EPA, OSHA, FCA, CMA, "
     "SFO and ICO; court filings; company filings and RNS announcements), then the first news report, using GDELT-matched articles, "
     "Google News and NGO trackers as leads. The earliest public report is the earliest dated, publicly accessible document that names "
     "the firm and describes the incident; later coverage, opinion pieces and generic sector stories do not qualify. Each verified "
     "row records the date, a source link and a confidence grade (high, medium or low); rows whose incident could not be identified, "
     "or whose earliest date could not be confirmed, are classed as unresolved or screen-only and excluded from the lag sample.",
     sz=11, al="j")
para("Two points before this paragraph goes into the draft. First, the audit file's 'Checked by' field records 'Claude research "
     "audit' for all 727 rows (and 'Claude (web audit)' or 'Claude (headline screen)' in the matched file), while the draft calls the "
     "clock hand-audited; the paper should state who dated the cases and how AI-assisted searches were checked. Second, I cannot "
     "act as the independent second coder. A random draw of 30 of the 223 (seed 20261031) is in A4_second_coder_30.xlsx, with the "
     "first coder's dates on a separate sheet to keep hidden; agreement within 7 days can be computed once a second person fills it in.",
     sz=11, al="j")

# ======================================================================= 5
A5 = pd.read_csv(f"{O}/A5_implied.csv")
head("5. Methodology-implied effect (robustness)")
rows = [["Sample", "Slope on implied ΔWAKIS: observed ΔWAKIS", "Slope: observed ΔIAS", "Mean implied ΔWAKIS", "N / clusters"]]
lab = {0: "All firm-years (implied = 0 if no change)", 1: "Firm-years with a weighted key-issue severity change",
       2: "  with a net implied decline (new or escalated cases)", 3: "  with a single changed key issue"}
for i, r in A5.iterrows():
    rows.append([lab[i], f(r.slope_dWAKIS, r.t_dWAKIS, 3), f(r.slope_dIAS, r.t_dIAS, 3), f"{r.mean_implied:+.3f}".replace("-", "−"), f"{n0(r.N)} / {n0(r.clusters)}"])
table(rows, [5.8, 3.3, 2.8, 2.4, 2.4])
note("Table R5. OLS of the observed change in WAKIS (and IAS) on the implied change Σₖ wₖ·Δdeductionₖ; year fixed effects; standard "
     "errors clustered by ultimate parent; paper sample without first covered years. Deductions are the midpoints of MSCI's "
     "non-structural/structural ranges for the highest severity in the key issue (moderate 1.05, severe 2.1, very severe 4.0 points), "
     "as summarised from the controversy-deduction exhibit of MSCI's ESG Ratings Methodology (edition to be confirmed); weights are the "
     "key-issue weights in force at year-end; business-ethics issues, which enter through governance Key Metrics, are excluded.")
para("What it shows. Where the implied score falls because of new or escalated cases, about 30% of the implied deduction appears "
     "in WAKIS (slope 0.30, t = 3.9) and almost none survives to the IAS (0.13, t = 0.7). The approximation assumes: the deduction "
     "lands one-for-one on the key-issue score (no floor at zero, no censoring of the management score), the midpoint is the right "
     "deduction (structural status is unobserved), the highest-severity case binds, nothing else in the issue changes in the same "
     "year, and no Company-Specific Key Issue is added. Each assumption biases the slope towards zero, so read 0.30 as a lower bound.",
     sz=11, al="j")
doc.save(f"{O}/Additional_Analyses_Results.docx"); print("saved")


## Appendix — run order & outputs

**Order:** Section 0 → 1 (once, to build the cached files) → 3 (build `panel_fy_validtime.parquet`) → 4 (any exhibit).

The authoritative variable dictionary and the exact expected values for every table are in `docs/CONSTRUCTION_DOCUMENTATION.md`; the full coefficient/t-stat/N log is in `log/REPRODUCTION_LOG.txt`. Standard errors are clustered by ultimate parent throughout; letter-band width = 10/7 ≈ 1.4286 IAS points; 2020 is reported separately.